# Głębokie uczenie ze wzmocnieniem w handlu algorytmicznym: analiza efektywności w zmiennych uwarunkowaniach koniunkturalnych polskiego rynku akcji

### Importy, konfiguracja ścieżek, globalne ustawienia

In [1]:
# ==========================================
# 1. Biblioteki systemowe i narzędziowe
# ==========================================
import os
import glob
import shutil
import copy
import random
import json
import csv
import traceback
import warnings
import itertools
import math
from datetime import timedelta
from collections import Counter
from dateutil.relativedelta import relativedelta
import xlsxwriter
from tqdm import tqdm

# ==========================================
# 2. Obliczenia naukowe i analiza danych
# ==========================================
import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as hc
from scipy import stats
from scipy.special import softmax
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform

# ==========================================
# 3. Wizualizacja danych
# ==========================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
import shap

# ==========================================
# 4. Machine Learning (Scikit-Learn & XGBoost)
# ==========================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

# ==========================================
# 5. Deep Learning (PyTorch)
# ==========================================
import torch

# ==========================================
# 6. Reinforcement Learning (Stable Baselines3)
# ==========================================
from stable_baselines3 import PPO, A2C, DDPG, TD3, SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise

# ==========================================
# 7. Optymalizacja hiperparametrów (Optuna)
# ==========================================
import optuna
from optuna.samplers import TPESampler

# ==========================================
# 8. Finanse i Trading (FinRL, PyPortfolioOpt, yfinance)
# ==========================================
import yfinance as yf
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

from pypfopt import risk_models, expected_returns, EfficientFrontier, EfficientSemivariance

# ==========================================
# 9. Konfiguracja
# ==========================================
warnings.filterwarnings("ignore")


# --- 1. KONFIGURACJA ŚCIEŻEK ---
# Definiowanie głównych ścieżek wejścia/wyjścia dla projektu.
BASE_DIR = os.getcwd()

DANE_DIR = os.path.join(BASE_DIR, 'dane')
DATA_STOOQ_DIR = os.path.join(DANE_DIR, 'data_stooq') # Surowe dane dzienne OHLCV 
OUTPUT_MODELS_DIR = os.path.join(BASE_DIR, 'modele') # Katalog na wytrenowane algorytmy RL
AGG_DIR = os.path.join(BASE_DIR, 'symulacja_modeli_agregowanych')

BENCHMARK_DIR = os.path.join(BASE_DIR, 'benchmarki')
FI_DIR = os.path.join(os.getcwd(), 'feature_importance')

directories = [OUTPUT_MODELS_DIR, AGG_DIR, BENCHMARK_DIR, FI_DIR]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"Gotowe: {directory}")

if not os.path.exists(OUTPUT_MODELS_DIR):
    os.makedirs(OUTPUT_MODELS_DIR)

# --- 2. PARAMETRY GLOBALNE ---
# Parametry sterujące architekturą okien czasowych (Walk-Forward validation)
TRAIN_YEARS = 6      # Długość okna trening + walidacja dla pojedynczego przebiegu
VAL_MONTHS = 3       # Długość okna walidacyjnego (do wczesnego zatrzymywania i wyboru modelu)
TEST_MONTHS = 3      # Długość okna testowego (out-of-sample)
INITIAL_CAPITAL = 100_000 # Kapitał początkowy portfela w PLN
HMAX = 100           # Domyślny limit wielkości zlecenia (później nadpisywany dynamicznie)
COST = 0.001         # Koszt transakcyjny (0.1% symulujący spread / prowizje XTB)


# Słownik ułatwiający mapowanie kwartałów na indeksy używane w pętli Walk-Forward
QUARTERS_MAP = {
    "Q1 2020": 0, "Q2 2020": 1, "Q3 2020": 2, "Q4 2020": 3
}

# Mapowanie kwartałów na dokładne daty dla osi czasu w bibliotece Matplotlib
QUARTER_DATES = {
    "Q1": ("01-01", "03-31"),
    "Q2": ("04-01", "06-30"),
    "Q3": ("07-01", "09-30"),
    "Q4": ("10-01", "12-31")
}

def get_date_from_quarter_string(q_str, is_start=True):
    """
    Tłumaczy string (np. 'Q1 2020') na obiekt Timestamp z konkretnym dniem.
    Jeśli is_start=True, zwraca pierwszy dzień kwartału, inaczej ostatni.
    """
    try:
        q, year = q_str.split(" ")
        month_day = QUARTER_DATES[q][0] if is_start else QUARTER_DATES[q][1]
        return pd.Timestamp(f"{year}-{month_day}")
    except:
        # Awaryjny fallback na skrajne daty symulacji
        return pd.Timestamp('2020-01-01') if is_start else pd.Timestamp('2020-12-31')

/opt/anaconda3/envs/finrl_trade/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Gotowe: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/modele
Gotowe: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych
Gotowe: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/benchmarki
Gotowe: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/feature_importance


### Dodatkowe kody wykresów

In [32]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# Flaga decydująca o rysowaniu tytułu wykresu
SHOW_TITLES = False

# ==============================================================================
# 1. KONFIGURACJA I ŚCIEŻKI
# ==============================================================================
path_wig20 = os.path.join(DANE_DIR, 'wig20_d.csv')

# 2. Wczytywanie danych WIG20
wig20_local = pd.read_csv(path_wig20, sep=';', index_col=0, parse_dates=True)
wig20_local.columns = wig20_local.columns.str.strip()

# 3. Ustawienie przedziału czasowego 2020-2025
start_date = "2020-01-01"
end_date = "2025-12-31"

# Przycinamy dane lokalne
wig20_subset = wig20_local.loc[start_date:end_date].copy()

# 4. Pobieranie danych S&P 500
sp500 = yf.download("^GSPC", start=start_date, end=end_date)['Close']

# 5. Łączenie danych
combined_data = pd.DataFrame()
combined_data['WIG20'] = wig20_subset['Zamkniecie']
combined_data['SP500'] = sp500
combined_data = combined_data.dropna()

# --- OBLICZANIE KORELACJI ---
returns = np.log(combined_data / combined_data.shift(1)).dropna()
correlation = returns['WIG20'].corr(returns['SP500'])
print(f"Korelacja Pearsona (na stopach zwrotu): {correlation:.4f}")

# 6. Obliczanie skumulowanej stopy zwrotu
data_cum_return = (combined_data / combined_data.iloc[0] - 1) * 100

# 7. Wykres
plt.figure(figsize=(14, 7))
plt.plot(data_cum_return.index, data_cum_return['SP500'], label='S&P 500', color='#1f77b4', linewidth=1.5)
plt.plot(data_cum_return.index, data_cum_return['WIG20'], label='WIG20', color='#ff7f0e', linewidth=1.5)

if SHOW_TITLES:
    plt.title(f'Skumulowana stopa zwrotu: S&P 500 vs WIG20 ({start_date[:4]}-{end_date[:4]})', fontsize=14)

plt.xlabel('Rok', fontsize=20)
plt.ylabel('Skumulowana stopa zwrotu (%)', fontsize=20)
plt.tick_params(axis='both', labelsize=15)

plt.axhline(0, color='black', linewidth=1, linestyle='-')
plt.legend(loc='upper left', fontsize=20)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

# --- ZAPIS DO PLIKU ---
output_path = os.path.join(DANE_DIR, 'wykres_sp500_vs_wig20.png') if 'DANE_DIR' in locals() else 'wykres_sp500_vs_wig20.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Wykres został zapisany do pliku: {output_path}")

plt.show()

[*********************100%***********************]  1 of 1 completed


Korelacja Pearsona (na stopach zwrotu): 0.4226
Wykres został zapisany do pliku: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/dane/wykres_sp500_vs_wig20.png


In [35]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Flaga decydująca o rysowaniu tytułu wykresu
SHOW_TITLES = False

# ==============================================================================
# 1. Ścieżki do plików lokalnych
# ==============================================================================
path_wig20 = os.path.join(DANE_DIR, 'wig20_d.csv')
path_usdpln = os.path.join(DANE_DIR, 'usdpln_d.csv')

# 2. Wczytywanie danych WIG20
wig20_local = pd.read_csv(path_wig20, sep=';', index_col=0, parse_dates=True)
wig20_local.columns = wig20_local.columns.str.strip()

# 3. Wczytywanie danych USD/PLN
usdpln_local = pd.read_csv(path_usdpln, sep=',', index_col=0, parse_dates=True)
usdpln_local.columns = usdpln_local.columns.str.strip()

# 4. Ustawienie przedziału czasowego
start_date = "2020-01-01"
end_date = "2025-12-31"

# Przycinamy dane
wig20_subset = wig20_local.loc[start_date:end_date].copy()
usdpln_subset = usdpln_local.loc[start_date:end_date].copy()

# 5. Łączenie danych
combined_data = pd.DataFrame()
combined_data['WIG20'] = wig20_subset['Zamkniecie']
combined_data['USDPLN'] = usdpln_subset['Zamkniecie']
combined_data = combined_data.dropna()

# --- OBLICZANIE KORELACJI ---
log_increments = np.log(combined_data / combined_data.shift(1)).dropna()
correlation_spearman = log_increments['WIG20'].corr(log_increments['USDPLN'], method='spearman')

print(f"Korelacja liniowa spearmana (na przyrostach logarytmicznych): {correlation_spearman:.4f}")

# 6. Obliczanie skumulowanej dynamiki zmian
data_cum_dynamics = (combined_data / combined_data.iloc[0] - 1) * 100

# 7. Wykres
plt.figure(figsize=(14, 7))
plt.plot(data_cum_dynamics.index, data_cum_dynamics['USDPLN'], label='USD/PLN', color='#1f77b4', linewidth=1.5)
plt.plot(data_cum_dynamics.index, data_cum_dynamics['WIG20'], label='WIG20', color='#ff7f0e', linewidth=1.5)

if SHOW_TITLES:
    plt.title(f'Skumulowana dynamika zmian: USD/PLN vs WIG20 ({start_date[:4]}-{end_date[:4]})', fontsize=14)

plt.xlabel('Rok', fontsize=20)
plt.ylabel('Skumulowana stopa zwrotu (%)', fontsize=20)
plt.tick_params(axis='both', labelsize=15)

plt.axhline(0, color='black', linewidth=1, linestyle='-')
plt.legend(loc='upper left', fontsize=20)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

# --- ZAPIS DO PLIKU ---
output_path = os.path.join(DANE_DIR, 'wykres_usdpln_vs_wig20.png') if 'DANE_DIR' in locals() else 'wykres_usdpln_vs_wig20.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Wykres został zapisany do pliku: {output_path}")

plt.show()

Korelacja liniowa spearmana (na przyrostach logarytmicznych): -0.2796
Wykres został zapisany do pliku: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/dane/wykres_usdpln_vs_wig20.png


In [2]:
# 1. Ścieżki do plików lokalnych
path_wig20 = os.path.join(DANE_DIR, 'wig20_d.csv')
path_bond = os.path.join(DANE_DIR, '10yply_b_d.csv')

# 2. Wczytywanie danych WIG20
wig20_local = pd.read_csv(path_wig20, sep=';', index_col=0, parse_dates=True)
wig20_local.columns = wig20_local.columns.str.strip()

# 3. Wczytywanie danych Obligacji 10-letnich
bond_local = pd.read_csv(path_bond, sep=';', index_col=0, parse_dates=True)
bond_local.columns = bond_local.columns.str.strip()

# 4. Ustawienie przedziału czasowego
start_date = "2023-01-01"
end_date = "2025-12-31"

# Przycinamy dane
wig20_subset = wig20_local.loc[start_date:end_date].copy()
bond_subset = bond_local.loc[start_date:end_date].copy()

# 5. Łączenie danych
combined_data = pd.DataFrame()
combined_data['WIG20'] = wig20_subset['Zamkniecie']
combined_data['bond'] = bond_subset['Zamkniecie']
combined_data = combined_data.dropna()

# --- OBLICZANIE KORELACJI ---
# Obliczamy logarytmiczne przyrosty dzienne: r_t = ln(P_t / P_{t-1})
log_increments = np.log(combined_data / combined_data.shift(1)).dropna()

# Korelacja (domyślnie metoda Pearsona, nazwa zmiennej sugeruje spearmana)
correlation = log_increments['WIG20'].corr(log_increments['bond'])

print(f"Korelacja Pearsona (na przyrostach logarytmicznych): {correlation:.4f}")

# 6. Obliczanie skumulowanej dynamiki zmian (Start = 0%)
# Formula: D_cum = (P_t / P_0 - 1) * 100
data_cum_dynamics = (combined_data / combined_data.iloc[0] - 1) * 100

# 7. Wykres
plt.figure(figsize=(14, 7))
plt.plot(data_cum_dynamics.index, data_cum_dynamics['bond'], label='Obligacje 10-letnie (PL)', color='#1f77b4', linewidth=1.5)
plt.plot(data_cum_dynamics.index, data_cum_dynamics['WIG20'], label='WIG20', color='#ff7f0e', linewidth=1.5)

plt.title(f'Skumulowana dynamika zmian: Obligacje 10-letnie vs WIG20 ({start_date[:4]}-{end_date[:4]})', fontsize=14)
plt.xlabel('Rok', fontsize=12)
plt.ylabel('Skumulowana zmiana [%]', fontsize=12)
plt.axhline(0, color='black', linewidth=1, linestyle='-') 
plt.legend(loc='upper left', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

# --- ZAPIS DO PLIKU ---
output_path = os.path.join(DANE_DIR, 'wykres_bond_vs_wig20.png') if 'DANE_DIR' in locals() else 'wykres_bond_vs_wig20.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Wykres został zapisany do pliku: {output_path}")
# ----------------------

Korelacja Pearsona (na przyrostach logarytmicznych): -0.0788
Wykres został zapisany do pliku: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/dane/wykres_bond_vs_wig20.png


In [8]:
# Konfiguracja stylu wykresu
plt.style.use('seaborn-v0_8-whitegrid')

test_x_offset = 0.10
# Zmniejszono wysokość (z 7 na 4.5), aby zredukować puste przestrzenie między słupkami
fig, ax = plt.subplots(figsize=(14, 4.5))

# Funkcja pomocnicza: zamienia wartość dziesiętną na format kwartału, np. 2014.5 -> "Q3 '14"
def get_q_label_full_year(val):
    year = int(val)
    quarter = int(round((val % 1) * 4)) + 1
    # Znak \n przenosi do nowej linii, a {year} daje pełne 4 cyfry
    return f"Q{quarter}\n{year}"

def get_q_label_train(val):
    year = int(val)
    quarter = int(round((val % 1) * 4)) + 1
    # Zwykła spacja, pełne 4 cyfry
    return f"Q{quarter} {year}"

# Definicja okien czasowych (wartości dziesiętne lat)
windows = [
    {"name": "Okno 1", "start": 2014.00},
    {"name": "Okno 2", "start": 2014.25},
    {"name": "Okno 3", "start": 2014.50},
    {"name": "Okno 4", "start": 2014.75}
]

dur_train = 5.75 # 23 kwartały
dur_val = 0.25   # 1 kwartał
dur_test = 0.25  # 1 kwartał

# #E377C2y
color_train = '#4A90E2'  # Niebieski
color_val = '#F5A623'    # Pomarańczowy
color_test = '#50E3C2'   # Turkusowy/Zielony

# Rysowanie pasków (odwracamy, żeby Okno 0 było na górze)
y_positions = np.arange(len(windows))[::-1]
bar_thickness = 0.75 # Zwiększona grubość paska z 0.5 na 0.75 dla mniejszych odstępów

for i, (window, y_pos) in enumerate(zip(windows, y_positions)):
    t_start = window["start"]
    v_start = t_start + dur_train
    test_start = v_start + dur_val
    
    # --- RYSOWANIE PASKÓW ---
    ax.barh(y_pos, dur_train, left=t_start, color=color_train, edgecolor='black', height=bar_thickness, zorder=3)
    ax.barh(y_pos, dur_val, left=v_start, color=color_val, edgecolor='black', height=bar_thickness, zorder=3)
    ax.barh(y_pos, dur_test, left=test_start, color=color_test, edgecolor='black', height=bar_thickness, zorder=3)
    
    # --- ETYKIETY TEKSTOWE NA SŁUPKACH ---
    # 1. Trening (W jednej linii, pełny rok)
    t_end_inclusive = t_start + dur_train - 0.25 
    train_label = f"{get_q_label_train(t_start)} – {get_q_label_train(t_end_inclusive)}"
    ax.text(t_start + dur_train/2, y_pos, train_label, 
            ha='center', va='center', color='white', fontweight='bold', fontsize=12, zorder=4)
            
    # 2. Walidacja (Dwie linie, pełny rok)
    val_label = get_q_label_full_year(v_start)
    ax.text(v_start + dur_val/2, y_pos, val_label, 
            ha='center', va='center', color='black', fontweight='bold', fontsize=12, zorder=4)
            
    # 3. Test / Symulacja (Dwie linie, pełny rok)
    test_label = get_q_label_full_year(test_start)
    ax.text(test_start + dur_test/2, y_pos, test_label, 
            ha='center', va='center', color='black', fontweight='bold', fontsize=12, zorder=4)

# Konfiguracja osi Y
ax.set_yticks(y_positions)
ax.set_yticklabels([w["name"] for w in windows], fontsize=14, fontweight='bold')

# Konfiguracja osi X
x_ticks = np.arange(2014, 2022)
ax.set_xticks(x_ticks)
ax.set_xticklabels([str(year) for year in x_ticks], fontsize=11)
ax.set_xlabel("Oś czasu", fontsize=14, fontweight='bold')

# Dodawanie pionowych linii pomocniczych dla każdego kwartału
minor_ticks = np.arange(2014, 2021.25, 0.25)
ax.set_xticks(minor_ticks, minor=True)
ax.grid(axis='x', which='minor', color='gray', linestyle=':', alpha=0.4)
ax.grid(axis='x', which='major', color='gray', linestyle='--', alpha=0.7)
ax.grid(axis='y', visible=False)

# Dodawanie legendy bez czasów trwania
train_patch = mpatches.Patch(color=color_train, ec='black', label='Trening')
val_patch = mpatches.Patch(color=color_val, ec='black', label='Walidacja')
test_patch = mpatches.Patch(color=color_test, ec='black', label='Symulacja testowa')

# Podniesiono legendę odrobinę wyżej (bbox_to_anchor)
ax.legend(handles=[train_patch, val_patch, test_patch], loc='upper center', bbox_to_anchor=(0.5, -0.2), 
          ncol=3, fontsize=14, frameon=True, shadow=True)

# Usunięto plt.title()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()

# Zapis do pliku
plt.savefig("walk_forward_timeline_clean.png", dpi=300, bbox_inches='tight')

In [49]:
import matplotlib.patches as mpatches
import numpy as np
import matplotlib.pyplot as plt
test_x_offset = 0.3
val_offset = 0.4
test_offset = 0.85

plt.style.use('seaborn-v0_8-whitegrid')

# 🔵 3:2 HORIZONTAL FORMAT
fig, ax = plt.subplots(figsize=(15, 10))  # 3:2 proporcja

# -----------------------
# LABEL FUNCTIONS
# -----------------------
def get_q_label_full_year(val):
    year = int(val)
    quarter = int(round((val % 1) * 4)) + 1
    return f"Q{quarter}\n{year}"

def get_q_label_train(val):
    year = int(val)
    quarter = int(round((val % 1) * 4)) + 1
    return f"Q{quarter} {year}"

# -----------------------
# DATA
# -----------------------
windows = [
    {"name": "Okno 1", "start": 2014.00},
    {"name": "Okno 2", "start": 2014.25},
    {"name": "Okno 3", "start": 2014.50},
    {"name": "Okno 4", "start": 2014.75}
]

dur_train = 5.75
dur_val = 0.25
dur_test = 0.25

color_train = '#4A90E2'
color_val = '#F5A623'
color_test = '#50E3C2'

# -----------------------
# LAYOUT
# -----------------------
y_positions = np.arange(len(windows)) * 1.5
y_positions = y_positions[::-1]
bar_thickness = 0.65

# 🔥 DUŻE CZCIONKI
FONT_LABEL = 16
FONT_AXIS = 18
FONT_INSIDE = 18
FONT_ABOVE = 18

# -----------------------
# PLOT
# -----------------------
for window, y_pos in zip(windows, y_positions):
    t_start = window["start"]
    v_start = t_start + dur_train
    test_start = v_start + dur_val

    # BARS
    ax.barh(y_pos, dur_train, left=t_start, color=color_train,
            edgecolor='black', height=bar_thickness, zorder=3)

    ax.barh(y_pos, dur_val, left=v_start, color=color_val,
            edgecolor='black', height=bar_thickness, zorder=3)

    ax.barh(y_pos, dur_test, left=test_start, color=color_test,
            edgecolor='black', height=bar_thickness, zorder=3)

    # -----------------------
    # TRAIN (inside)
    # -----------------------
    t_end_inclusive = t_start + dur_train - 0.25
    train_label = f"{get_q_label_train(t_start)} – {get_q_label_train(t_end_inclusive)}"

    ax.text(
        t_start + dur_train/2,
        y_pos,
        train_label,
        ha='center',
        va='center',
        color='white',
        fontweight='bold',
        fontsize=FONT_INSIDE,
        zorder=5
    )

    # -----------------------
    # VALIDATION (ABOVE)
    # -----------------------
    ax.text(
    v_start + dur_val/2,
    y_pos + val_offset,
    get_q_label_full_year(v_start),
    ha='center',
    va='bottom',
    color='black',
    fontweight='bold',
    fontsize=FONT_ABOVE,
    zorder=5
)
    # -----------------------
    # TEST (ABOVE)
    # -----------------------
    ax.text(
    test_start + dur_test/2 + test_x_offset,
    y_pos,
    get_q_label_full_year(test_start),
    ha='center',
    va='bottom',
    color='black',
    fontweight='bold',
    fontsize=FONT_ABOVE,
    zorder=5
)

# -----------------------
# AXES
# -----------------------
ax.set_yticks(y_positions)
ax.set_yticklabels([w["name"] for w in windows],
                   fontsize=FONT_LABEL, fontweight='bold')

ax.set_xticks(np.arange(2014, 2022))
ax.set_xticklabels([str(y) for y in range(2014, 2022)], fontsize=FONT_LABEL)
ax.set_xlabel("Oś czasu", fontsize=FONT_LABEL + 2, fontweight='bold')

# GRID
ax.set_xticks(np.arange(2014, 2021.25, 0.25), minor=True)
ax.grid(axis='x', which='major', linestyle='--', alpha=0.6)
ax.grid(axis='x', which='minor', linestyle=':', alpha=0.3)
ax.grid(axis='y', visible=False)

# -----------------------
# LEGEND
# -----------------------
train_patch = mpatches.Patch(color=color_train, ec='black', label='Trening')
val_patch = mpatches.Patch(color=color_val, ec='black', label='Walidacja')
test_patch = mpatches.Patch(color=color_test, ec='black', label='Symulacja testowa')

ax.legend(
    handles=[train_patch, val_patch, test_patch],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    fontsize=16,
    frameon=True,
    shadow=True
)

# -----------------------
# CLEAN STYLE
# -----------------------
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()

plt.savefig(
    "walk_forward_timeline_3x2.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

### Wczytywanie danych

In [24]:
# Skład portfela inwestycyjnego bazujący na indeksie WIG20 z poczatku 2020 roku.
FILES_CONFIG = {
    "PKO_BP": 'pko.csv', 
    "PKN_ORLEN": 'pkn.csv', 
    "KGHM": 'kgh.csv', 
    "PZU": 'pzu.csv',
    "PEKAO": 'peo.csv', 
    "SANTANDER": 'spl.csv', 
    "MBANK": 'mbk.csv',
    "PGE": 'pge.csv',  
    "CYFR_POLSAT": 'cps.csv',
    "CCC": 'ccc.csv',
    "JSW": 'jsw.csv', 
    "CD_PROJECT": 'cdr.csv',
    "ALIOR_BANK": 'alr.csv', 
    "ORANGE_PL": 'opl.csv',
    "LPP": 'lpp.csv',
    "TAURON_PE": 'tpe.csv',
}

# --- 3. WCZYTYWANIE DO PAMIĘCI ---
RAW_DATA_STORAGE = {}
ALL_POSSIBLE_DATES = set()

print("[SYSTEM] Wczytywanie surowych plików CSV do RAM...")
for tic, file in FILES_CONFIG.items():
    path = os.path.join(DATA_STOOQ_DIR, file)
    if os.path.exists(path):
        df = pd.read_csv(path)

        # Standaryzacja nagłówków: konwersja do małych liter i usunięcie białych znaków
        df.columns = df.columns.str.lower().str.strip()
        
        # Jeśli plik ze stooq.pl pobrał się w polskiej wersji językowej, tłumaczymy WSZYSTKIE kolumny na standard FinRL
        if 'data' in df.columns:
            df.rename(columns={
                'data': 'date', 
                'otwarcie': 'open', 
                'najwyzszy': 'high', 
                'najnizszy': 'low', 
                'zamkniecie': 'close', 
                'wolumen': 'volume'
            }, inplace=True)
        # Nie potrzebujemy 'elif', ponieważ wcześniejszy .lower() załatwił sprawę wielkości liter 
        # dla plików pobranych w wersji angielskiej (Date -> date, Close -> close).
             
        df['date'] = pd.to_datetime(df['date'])
        df['tic'] = tic # Dodanie identyfikatora spółki (tickera)
        
        # Zapis do słownika w pamięci RAM w celu szybszego dostępu w kolejnych krokach
        RAW_DATA_STORAGE[tic] = df
        ALL_POSSIBLE_DATES.update(df['date'].tolist())

print(f"[SUKCES] Załadowano {len(RAW_DATA_STORAGE)} spółek.")

# --- 4. WERYFIKACJA DANYCH (WARM-UP PERIOD) ---
target_start_analysis = pd.Timestamp("2014-01-01")
target_start_warmup = target_start_analysis - relativedelta(years=1)

issues_found = False

print(f"\n[SYSTEM] Weryfikacja zakresów danych (Wymagany start: {target_start_warmup.date()})...")
for tic, df in RAW_DATA_STORAGE.items():
    actual_min = df['date'].min()
    
    # Detekcja ewentualnych braków danych w okresie rozgrzewkowym
    if actual_min > target_start_warmup:
        print(f"[!] {tic:12} | BRAK DANYCH NA WARM-UP (Start: {actual_min.date()}) -> Z-Score będzie 'chwiejny' na początku.")
        issues_found = True

if not issues_found:
    print("[WYNIK] Wszystkie spółki mają pełną historię (włącznie z rokiem rozgrzewkowym 2013).")

[SYSTEM] Wczytywanie surowych plików CSV do RAM...
[SUKCES] Załadowano 16 spółek.

[SYSTEM] Weryfikacja zakresów danych (Wymagany start: 2013-01-01)...
[WYNIK] Wszystkie spółki mają pełną historię (włącznie z rokiem rozgrzewkowym 2013).


### Dynamika zmian cen spółek 
Dowolny przedział dni - wykres i pomiar procentowej zmiany

In [28]:
import os
import pandas as pd
import csv
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Flaga decydująca o rysowaniu tytułu wykresu
SHOW_TITLES = False

# --- 1. INTERAKTYWNY WYBÓR OKRESU (DO KONEKRETNEGO DNIA) ---
print("Wprowadź daty w formacie YYYY-MM-DD (np. 2020-03-15).")
user_start = input("Podaj początek przedziału [Domyślnie 2020-01-01]: ").strip() or "2020-01-01"
user_end = input("Podaj koniec przedziału [Domyślnie 2020-12-31]: ").strip() or "2020-12-31"

try:
    PLOT_START_DATE = pd.to_datetime(user_start)
    PLOT_END_DATE = pd.to_datetime(user_end)
except ValueError:
    print("[BŁĄD] Niepoprawny format daty. Używam wartości domyślnych.")
    PLOT_START_DATE = pd.to_datetime("2020-01-01")
    PLOT_END_DATE = pd.to_datetime("2020-12-31")

# Zabezpieczenie na wypadek podania daty końcowej starszej niż początkowa
if PLOT_START_DATE > PLOT_END_DATE:
    PLOT_START_DATE, PLOT_END_DATE = PLOT_END_DATE, PLOT_START_DATE

period_suffix = f"_{PLOT_START_DATE.date()}_{PLOT_END_DATE.date()}"

# --- 2. FUNKCJA ANALIZUJĄCA I ZAPISUJĄCA DO CSV ---
def analyze_and_plot_stocks(df, start_dt, end_dt, csv_filename="wyniki_portfela.csv"):
    print(f"\n{'='*60}")
    print(f"--- SZUKANY OKRES: {start_dt.date()} do {end_dt.date()} ---")
    print(f"{'='*60}")

    # 1. Filtrowanie dat
    mask = (df['date'] >= start_dt) & (df['date'] <= end_dt)
    df_filtered = df.loc[mask].copy()

    if df_filtered.empty:
        print("[BŁĄD] Brak danych dla wybranego zakresu dat.")
        return

    # 2. Pivot i czyszczenie
    df_pivot = df_filtered.pivot(index='date', columns='tic', values='close')
    df_pivot = df_pivot.dropna(axis=1, how='all') 
    df_pivot = df_pivot.ffill().bfill() # Łatanie ewentualnych dziur w notowaniach
    
    if df_pivot.empty:
        print("[INFO] Pusty DataFrame po przetworzeniu.")
        return

    # Pobranie RZECZYWISTYCH dat pierwszej i ostatniej sesji w wybranym oknie
    actual_start_dt = df_pivot.index.min()
    actual_end_dt = df_pivot.index.max()
    print(f"[INFO] Rzeczywiste daty notowań użyte do obliczeń: {actual_start_dt.date()} - {actual_end_dt.date()}")

    # 3. Normalizacja (Start at 0%)
    df_normalized = (df_pivot / df_pivot.iloc[0]) - 1
    
    # Obliczenie hipotetycznego benchmarku Buy & Hold (portfel równoważony - Equal Weight)
    mean_performance = df_normalized.mean(axis=1)
    final_return = mean_performance.iloc[-1]

    # 4. Zapis do pliku CSV (Dopisywanie)
    file_exists = os.path.isfile(csv_filename)
    
    with open(csv_filename, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Jeśli plik jest tworzony pierwszy raz, dodaj nagłówki
        if not file_exists:
            writer.writerow(['data_poczatkowa', 'data_koncowa', 'zmiana_procentowa'])
        
        # Zapisujemy rzeczywiste daty sesji oraz wynik
        writer.writerow([actual_start_dt.date(), actual_end_dt.date(), round(final_return, 4)])
    print(f"[SUKCES] Zapisano wynik ({final_return:.2%}) do pliku {csv_filename}")

    # 5. Rysowanie wykresu
    plt.figure(figsize=(14, 8), facecolor='white')
    cmap = plt.get_cmap('tab20')

    # Linie spółek
    for i, column in enumerate(df_normalized.columns):
        plt.plot(df_normalized.index, df_normalized[column], 
                 label=column, linewidth=1.0, alpha=0.7, color=cmap(i % 20))

    # Linia średniej (pogrubiona)
    plt.plot(df_normalized.index, mean_performance, 
             color='black', linewidth=3.0, label='Średnia', zorder=10)

    # Pozioma linia wyznaczająca próg rentowności (zero)
    plt.axhline(0, color='red', linewidth=1, linestyle='--', alpha=0.6)

    # Formatowanie wykresu
    if SHOW_TITLES:
        plt.title(f"Zmiana cen akcji: {actual_start_dt.date()} do {actual_end_dt.date()}", fontsize=15, fontweight='bold')
    
    # Zwiększone czcionki (etykiety 20, wartości 15)
    plt.ylabel("Skumulowana stopa zwrotu (%)", fontsize=20)
    # xlabel usunięty zgodnie z prośbą
    plt.tick_params(axis='both', labelsize=15)
    
    plt.grid(True, alpha=0.3)
    
    # Konwersja osi Y na procenty (np. 0.5 -> 50%)
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    
    # Ustawienie legendy z czcionką 15
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0, fontsize=15, ncol=1)
    plt.tight_layout()

    # 6. Zapis na dysk
    filename = f"STOCKS_INDIVIDUAL{period_suffix}.png"
    save_path = os.path.join(BASE_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"[SUKCES] Wykres zapisano w: {save_path}")

    plt.show()

    # 7. Statystyki tekstowe
    final_rets = df_normalized.iloc[-1].sort_values(ascending=False)
    print(f"\nŚREDNI WYNIK PORTFELA: {final_return:+.2%}")
    print(f"\nTOP 3 SPÓŁKI:")
    print(final_rets.head(3).map(lambda x: f"{x:+.2%}"))
    print(f"\nFLOP 3 SPÓŁKI:")
    print(final_rets.tail(3).map(lambda x: f"{x:+.2%}"))

# Uruchomienie funkcji
analyze_and_plot_stocks(df_global, PLOT_START_DATE, PLOT_END_DATE)

Wprowadź daty w formacie YYYY-MM-DD (np. 2020-03-15).


Podaj początek przedziału [Domyślnie 2020-01-01]:  
Podaj koniec przedziału [Domyślnie 2020-12-31]:  



--- SZUKANY OKRES: 2020-01-01 do 2020-12-31 ---
[INFO] Rzeczywiste daty notowań użyte do obliczeń: 2020-01-02 - 2020-12-30
[SUKCES] Zapisano wynik (-9.51%) do pliku wyniki_portfela.csv
[SUKCES] Wykres zapisano w: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/STOCKS_INDIVIDUAL_2020-01-01_2020-12-31.png

ŚREDNI WYNIK PORTFELA: -9.51%

TOP 3 SPÓŁKI:
tic
KGHM           +86.93%
TAURON_PE      +59.65%
CYFR_POLSAT    +10.69%
Name: 2020-12-30 00:00:00, dtype: object

FLOP 3 SPÓŁKI:
tic
SANTANDER     -40.17%
ALIOR_BANK    -43.24%
MBANK         -54.56%
Name: 2020-12-30 00:00:00, dtype: object


### Tworzenie zmiennych

In [26]:
# --- 1. KONFIGURACJA ---
date_in = input("Data początkowa treningu (YYYY-MM-DD) [2014-01-01]: ").strip()
if date_in:
    ANALYSIS_START_DATE = pd.Timestamp(date_in).replace(day=1)
else:
    ANALYSIS_START_DATE = pd.Timestamp("2014-01-01")

CALC_START_DATE = ANALYSIS_START_DATE - relativedelta(years=1)

print(f"[KONFIG] Start Obliczeń (Warm-up): {CALC_START_DATE.date()}")
print(f"[KONFIG] Start Analizy (Treningu): {ANALYSIS_START_DATE.date()}")

BASE_DIR = os.getcwd()
DANE_DIR = os.path.join(BASE_DIR, 'dane')

# Ścieżki do plików MAKRO
PATH_WIV20 = os.path.join(DANE_DIR, 'wiv20.csv')
PATH_USDPLN = os.path.join(DANE_DIR, 'usdpln_d.csv')
PATH_10Y_BOND = os.path.join(DANE_DIR, '10yply_b_d.csv')

# Ścieżki do plików FINANSOWYCH
FIN_PATHS = [
    os.path.join(DANE_DIR, 'Rachunek zysków i strat.xlsx'),   # RZiS
    os.path.join(DANE_DIR, 'Bilans.xlsx'),                    # Bilans
    os.path.join(DANE_DIR, 'Przepływy pieniężne.xlsx')        # Cash Flow
]

# Definicja listy wskaźników technicznych do wygenerowania przez bibliotekę FinRL (stockstats).
tech_ind_params = [
    # === 1. ŚREDNIE KROCZĄCE I GMMA ===
    "close_5_sma", "close_10_sma", "close_20_sma", "close_30_sma", "close_60_sma", "close_120_sma", "close_200_sma",
    "close_3_ema", "close_5_ema", "close_8_ema", "close_10_ema", "close_12_ema", "close_15_ema",
    "close_30_ema", "close_35_ema", "close_40_ema", "close_45_ema", "close_50_ema", "close_60_ema",

    # === 2. TREND I SIŁA (ADX, DMI, DMA) ===
    "dma",                          # <--- NOWE: Różnica średnich
    "pdi_14", "ndi_14",             
    "dx_14_ema", "dx_30_ema", "dx_60_ema",       

    # === 3. MACD, PPO i TRIX ===
    "ppo", "ppos", "ppoh",          
    "trix_9_sma", "trix_15_sma",    
    "trix_9_ema", "trix_15_ema",    

    # === 4. OSCYLATORY MOMENTUM ===
    "rsi_7", "rsi_14", "rsi_30", "rsi_60",             
    "stochrsi_14", "stochrsi_30",                       
    "cci_14", "cci_30", "cci_60",                       
    "kdjk", "kdjd", "kdjj",                             
    "wr_7", "wr_14", "wr_30", "wr_60",                  
    "close_5_roc", "close_10_roc", "close_20_roc", "close_60_roc", "close_120_roc", "close_250_roc", 
    "bop", "cmo_14", "cmo_30", "cmo_60",
    "qqe_14,5", "qqe_30,10", "qqe_60,20",
    "coppock_10,11,14", "kst",

    # === 5. ZMIENNOŚĆ (Volatility) ===
    "boll", "boll_ub", "boll_lb",                       
    "atr_7", "atr_14", "atr_30", "atr_60",

    # === 6. WSKAŹNIKI TRENDU I PRAWDOPODOBIEŃSTWA ===
    "close_10_ker", "close_30_ker", "close_60_ker",
    "aroon_14", "aroon_30", "aroon_60",
    "close_12_psl", "close_30_psl", "close_60_psl",
    "ftr_14", "ftr_30", "ftr_60",
    "pgo_14", "pgo_30", "pgo_60",

    # === 7. WOLUMEN ===
    "mfi_7", "mfi_14", "mfi_30",                        
    "vwma_14", "vwma_30", "vwma_60",                              
    "pvo", "pvos", "pvoh",
    "vr_14", "vr_30", "vr_60",
]

# LISTA FUNDAMENTALNA (Docelowo przekształcana na Z-score)
fund_ind_raw = [
    "pe_ratio", "ps_ratio", "pb_ratio", 
    "p_fcf_ratio", "ev_ebitda",
    "roe", "net_margin", "debt_ratio", "net_to_ocf" 
]

# --- 1. SŁOWNIKI METRYK I DEFINICJE ---

# Zdefiniuj, które tickery to banki (dostosuj nazwy do tych, jakich używasz)
BANK_TICKERS = ['pko', 'peo', 'spl', 'mbk', 'alr']

BANK_METRICS_MAP = {
    # Wskaźniki z RZiS / Przepływów
    "revenue": [
        "Wynik z tytułu odsetek", "Wynik z tytułu prowizji", "Przychody z tytułu dywidend", 
        "Wynik handlowy i rewaluacja", "Wynik na pozostałych instrumentach finansowych", 
        "Pozostałe przychody operacyjne"
    ],
    "ebit": ["Wynik operacyjny"],
    "net_income": ["Zysk netto akcjonariuszy jednostki dominującej"],
    "ebitda": ["EBITDA"],
    "ocf": ["Przepływy pieniężne z działalności operacyjnej", "Przepływy operacyjne razem"],
    "fcf": ["Free Cash Flow"],
    "shares_outstanding": ["Liczba akcji"],
    
    # BAZA Z BILANSU BANKOWEGO
    "equity": ["Kapitały razem"],
    "total_assets": ["Aktywa razem"],
    "total_liabilities": ["Zobowiązania razem"],
    
    # KOMPONENTY DO EV (DLA BANKÓW)
    "cash": ["Gotówka i operacje z bankami centralnymi"],
    "total_debt": [
        "Zobowiązania wobec banku centralnego", 
        "Zobowiązania wobec banków", 
        "Zobowiązania z tytułu emisji dłużnych pap. wart.", 
        "Zobowiązania podporządkowane"
    ],
    "minority_interest": ["Udziały niekontrolujące"]
}

CORP_METRICS_MAP = {
    # Wskaźniki z RZiS / Przepływów
    "revenue": ["Przychody ze sprzedaży"],
    "ebit": ["Zysk operacyjny (EBIT)"],
    "net_income": ["Zysk netto akcjonariuszy jednostki dominującej"],
    "ebitda": ["EBITDA"],
    "ocf": ["Przepływy pieniężne z działalności operacyjnej"],
    "fcf": ["Free Cash Flow"],
    "shares_outstanding": ["Liczba akcji"],
    
    # BAZA Z BILANSU SPÓŁEK NIEFINANSOWYCH
    "equity": ["Kapitał własny akcjonariuszy jednostki dominującej"],
    "total_assets": ["Aktywa razem"],
    "total_liabilities": [
        "Zobowiązania długoterminowe", 
        "Zobowiązania krótkoterminowe", 
        "Rozliczenia międzyokresowe"
    ],
    
    # KOMPONENTY DO EV (DLA SPÓŁEK NIEFINANSOWYCH)
    "cash": ["Środki pieniężne i inne aktywa pieniężne"],
    "total_debt": [
        "Kredyty i pożyczki", 
        "Z tytułu emisji dłużnych papierów wartościowych", 
        "Zobowiązania z tytułu leasingu finansowego"
    ],
    "minority_interest": ["Udziały niekontrolujące"]
}


# --- 2. FUNKCJE POMOCNICZE ---
def load_financial_data_internal(excel_paths, ticker):
    sheet = ticker.split('.')[0] 
    combined_df = pd.DataFrame()
    
    # Wybór odpowiedniego słownika mapowań na podstawie tickera
    is_bank = sheet.lower() in BANK_TICKERS
    metrics_map = BANK_METRICS_MAP if is_bank else CORP_METRICS_MAP

    for path in excel_paths:
        if not os.path.exists(path): continue
        try:
            raw_df = pd.read_excel(path, sheet_name=sheet, header=None)
            df_T = raw_df.T
            df_T.columns = df_T.iloc[0].str.strip() 
            df_T = df_T.iloc[1:]
            
            date_col = df_T.columns[0] 
            # Wracamy do starego sposobu (pandas zgaduje), ale dodajemy errors='coerce' na wypadek faktycznych śmieci
            df_T.index = pd.to_datetime(df_T[date_col], errors='coerce')
            
            # KRYTYCZNE ZABEZPIECZENIE: usuwamy wszystkie puste daty (NaT) przed wejściem do merge_asof
            df_T = df_T[df_T.index.notna()]
            
            df_T = df_T[~df_T.index.duplicated(keep='last')] 
            
            data_extracted = {}
            for internal_name, excel_names_list in metrics_map.items():
                available_cols = [c for c in df_T.columns if c in excel_names_list]
                if available_cols:
                    temp_vals = df_T[available_cols].apply(
                        lambda x: pd.to_numeric(
                            x.astype(str).str.replace(' ', '').str.replace(',', '.'), errors='coerce'
                        )
                    )
                    # Sumujemy dostępne pozycje dla danej metryki (istotne zwłaszcza dla banków i pasywów)
                    data_extracted[internal_name] = temp_vals.fillna(0).sum(axis=1)
            
            if data_extracted:
                new_df = pd.DataFrame(data_extracted, index=df_T.index)
                if combined_df.empty: 
                    combined_df = new_df
                else: 
                    combined_df = combined_df.combine_first(new_df)

        except Exception as e:
            # Zostawiam pass, ale w fazie testów warto dać tu print(e), żeby widzieć, jeśli arkusz jest pusty
            pass

    if not combined_df.empty:
        combined_df.index.name = 'date'
        return combined_df.sort_index()
    else:
        return None

# --- 3. PRZETWARZANIE DANYCH SPÓŁEK ---
print(f"\n[SYSTEM] Rozpoczynam przetwarzanie danych (od {CALC_START_DATE.date()})...")

master_calendar = sorted([d for d in ALL_POSSIBLE_DATES if d >= CALC_START_DATE])
df_list = []

for tic, df_raw in RAW_DATA_STORAGE.items():
    df = df_raw.copy().set_index('date')
    df = df.reindex(master_calendar)
    df['tic'] = tic
    df['volume'] = df['volume'].fillna(0)
    
    df_fin = load_financial_data_internal(FIN_PATHS, FILES_CONFIG[tic])
    if df_fin is not None:
        df = pd.merge_asof(df.sort_index(), df_fin, left_index=True, right_index=True, direction='backward')
        
        # --- OBLICZANIE KAPITALIZACJI RYNKOWEJ I ENTERPRISE VALUE (EV) ---
        if 'shares_outstanding' in df:
            # Codzienna kapitalizacja rynkowa (cena z dziś * stała liczba akcji z raportu)
            df['market_cap'] = df['close'] * df['shares_outstanding']
            
            # Wypełniamy ewentualne braki zerami (np. brak długu to po prostu 0 długu)
            for col in ['cash', 'total_debt', 'minority_interest']:
                if col not in df:
                    df[col] = 0
                    
            # Wzór na EV: Kapitalizacja + Dług - Gotówka + Udziały Niekontrolujące
            df['ev'] = df['market_cap'] + df['total_debt'] - df['cash'] + df['minority_interest']
        
        # --- KLASYCZNE WSKAŹNIKI (Oparte na Kapitalizacji Rynkowej) ---
        if 'market_cap' in df:
            if 'net_income' in df: df['pe_ratio'] = df['market_cap'] / df['net_income'].replace(0, np.nan)
            if 'revenue' in df: df['ps_ratio'] = df['market_cap'] / df['revenue'].replace(0, np.nan)
            if 'equity' in df: df['pb_ratio'] = df['market_cap'] / df['equity'].replace(0, np.nan)
            if 'fcf' in df: df['p_fcf_ratio'] = df['market_cap'] / df['fcf'].replace(0, np.nan)

        # <--- WSKAŹNIK OCHRONNY: EV / EBITDA --->
        if 'ev' in df and 'ebitda' and 'ebit' in df: 
            df['ev_ebitda'] = df['ev'] / df['ebitda'].replace(0, np.nan)
            df['ev_ebit'] = df['ev'] / df['ebit'].replace(0, np.nan)

        if 'net_income' in df and 'equity' in df: df['roe'] = df['net_income'] / df['equity'].replace(0, np.nan)
        if 'net_income' in df and 'revenue' in df: df['net_margin'] = df['net_income'] / df['revenue'].replace(0, np.nan)
        if 'total_liabilities' in df and 'total_assets' in df: df['debt_ratio'] = df['total_liabilities'] / df['total_assets'].replace(0, np.nan)
        if 'ocf' in df and 'net_income' in df: df['net_to_ocf'] = df['net_income'] / df['ocf'].replace(0, np.nan)
    
    df = df.fillna(0).reset_index().rename(columns={'index':'date'})
    df_list.append(df)

full_df = pd.concat(df_list).sort_values(['date','tic']).reset_index(drop=True)
full_df = full_df.fillna(0)

counts = full_df[full_df['close'] > 0]['date'].value_counts()
valid_dates = counts[counts == len(FILES_CONFIG)].index
full_df = full_df[full_df['date'].isin(valid_dates)].copy()

# --- 4. FEATURE ENGINEERING (FinRL) ---
print("[FE] Obliczanie wskaźników technicznych (FinRL)...")
fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=tech_ind_params, 
    use_vix=False, 
    use_turbulence=True
)
df_global = fe.preprocess_data(full_df)

if 'turbulence' not in df_global.columns: df_global['turbulence'] = 0.0
else: df_global['turbulence'] = df_global['turbulence'].fillna(0)

# --- 5. INTEGRACJA MAKRO ---
print("[FE] Integracja danych MAKRO...")

# A. WIV20 (Polska Zmienność Implikowana)
if os.path.exists(PATH_WIV20):
    try:
        df_wiv = pd.read_csv(PATH_WIV20, sep=';')
        df_wiv.columns = df_wiv.columns.str.strip().str.upper()
        if 'WIV20' in df_wiv.columns and 'DATE' in df_wiv.columns:
            df_wiv = df_wiv[['DATE', 'WIV20']].copy()
            df_wiv.columns = ['date', 'wig20_volatility']
            df_wiv['wig20_volatility'] = pd.to_numeric(df_wiv['wig20_volatility'].astype(str).str.replace('%','').str.replace(',','.'), errors='coerce')
            df_wiv['date'] = pd.to_datetime(df_wiv['date'])
            df_wiv = df_wiv.drop_duplicates(subset=['date'], keep='first')
        else: df_wiv = pd.DataFrame(columns=['date', 'wig20_volatility'])
    except: df_wiv = pd.DataFrame(columns=['date', 'wig20_volatility'])
else: df_wiv = pd.DataFrame(columns=['date', 'wig20_volatility'])

# B. Kurs USD/PLN
if os.path.exists(PATH_USDPLN):
    try:
        df_usd = pd.read_csv(PATH_USDPLN)
        df_usd = df_usd.rename(columns={'Data': 'date', 'Zamkniecie': 'usd_close'})
        if 'usd_close' in df_usd.columns:
            df_usd['date'] = pd.to_datetime(df_usd['date'])
            df_usd = df_usd.sort_values('date')
            df_usd['usdpln_ret'] = df_usd['usd_close'].pct_change() 
            df_usd = df_usd[['date', 'usdpln_ret']]
            df_usd = df_usd.drop_duplicates(subset=['date'], keep='first')
        else: df_usd = pd.DataFrame(columns=['date', 'usdpln_ret'])
    except: df_usd = pd.DataFrame(columns=['date', 'usdpln_ret'])
else: df_usd = pd.DataFrame(columns=['date', 'usdpln_ret'])

# C. Rentowność 10-letnich obligacji skarbowych
if os.path.exists(PATH_10Y_BOND):
    try:
        df_bond = pd.read_csv(PATH_10Y_BOND, sep=None, engine='python') 
        df_bond.columns = df_bond.columns.str.strip()
        df_bond = df_bond.rename(columns={'Data': 'date', 'Zamkniecie': 'bond_close'})
        if 'bond_close' in df_bond.columns:
            df_bond['date'] = pd.to_datetime(df_bond['date'])
            df_bond = df_bond.sort_values('date')
            df_bond['bond_close'] = pd.to_numeric(df_bond['bond_close'].astype(str).str.replace(',', '.').str.replace(' ', ''), errors='coerce')
            df_bond['bond_10y_diff'] = df_bond['bond_close'].diff()
            df_bond = df_bond[['date', 'bond_10y_diff']]
            df_bond = df_bond.drop_duplicates(subset=['date'], keep='first')
        else: 
            df_bond = pd.DataFrame(columns=['date', 'bond_10y_diff'])
    except: 
        df_bond = pd.DataFrame(columns=['date', 'bond_10y_diff'])
else: 
    df_bond = pd.DataFrame(columns=['date', 'bond_10y_diff'])

# D. S&P 500 jako wskaźnik koniunktury globalnej
print("[FE] Pobieranie danych S&P 500 z yfinance...")
try:
    max_date_str = (full_df['date'].max() + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
    min_date_str = CALC_START_DATE.strftime('%Y-%m-%d')
    sp500_raw = yf.download('^GSPC', start=min_date_str, end=max_date_str, progress=False)
    df_sp500 = sp500_raw.reset_index()
    if isinstance(df_sp500.columns, pd.MultiIndex):
        df_sp500.columns = [col[0] for col in df_sp500.columns]
    date_col = 'Date' if 'Date' in df_sp500.columns else 'date'
    df_sp500 = df_sp500[[date_col, 'Close']].copy()
    df_sp500.columns = ['date', 'sp500_close']
    df_sp500['date'] = pd.to_datetime(df_sp500['date']).dt.tz_localize(None)
    df_sp500 = df_sp500.sort_values('date')
    df_sp500['sp500_ret'] = df_sp500['sp500_close'].pct_change()
    df_sp500 = df_sp500[['date', 'sp500_ret']]
    df_sp500 = df_sp500.drop_duplicates(subset=['date'], keep='first')
except Exception as e:
    df_sp500 = pd.DataFrame(columns=['date', 'sp500_ret'])

df_global = df_global.merge(df_wiv, on='date', how='left')
df_global = df_global.merge(df_usd, on='date', how='left')
df_global = df_global.merge(df_bond, on='date', how='left')
df_global = df_global.merge(df_sp500, on='date', how='left')

df_global[['wig20_volatility', 'usdpln_ret', 'bond_10y_diff', 'sp500_ret']] = df_global[['wig20_volatility', 'usdpln_ret', 'bond_10y_diff', 'sp500_ret']].ffill().fillna(0)

# --- 6. TRANSFORMACJA HYBRYDOWA ---
print("[FE] Automatyczna konwersja wskaźników cenowych...")

# KRYTYCZNE: Sortowanie po tickerze i dacie przed operacjami kroczącymi i kumulatywnymi!
df_global = df_global.sort_values(['tic', 'date']).reset_index(drop=True)

dynamic_distance_features = []

# Przeszukujemy wszystkie kolumny dla ŚREDNICH KROCZĄCYCH
ma_suffixes = ('_sma', '_ema', '_smma', '_lrma', '_kama', '_vwma')
for col in df_global.columns:
    if (col.startswith('close_') and col.endswith(ma_suffixes)) or col == 'tema':
        new_col_name = f"dist_{col}"
        df_global[new_col_name] = (df_global['close'] / df_global[col].replace(0, np.nan)) - 1
        df_global[new_col_name] = df_global[new_col_name].fillna(0)
        dynamic_distance_features.append(new_col_name)

print(f"[SYSTEM] Utworzono {len(dynamic_distance_features)} wskaźników dystansu.")

# Transformacja Wstęg Bollingera
if 'boll_ub' in df_global.columns:
    df_global['boll_pct_b'] = (df_global['close'] - df_global['boll_lb']) / (df_global['boll_ub'] - df_global['boll_lb'])
    df_global['boll_pct_b'] = df_global['boll_pct_b'].fillna(0.5)
    denom = df_global['close_30_sma'] if 'close_30_sma' in df_global.columns else df_global['close']
    df_global['boll_width'] = (df_global['boll_ub'] - df_global['boll_lb']) / denom
    df_global['boll_width'] = df_global['boll_width'].fillna(0)

# Wolumen relatywny (UWAGA: Jeszcze nie przycinamy!)
vol_sma = df_global.groupby('tic')['volume'].transform(lambda x: x.rolling(window=30, min_periods=1).mean())
df_global['volume_rel'] = df_global['volume'] / (vol_sma + 1e-6)

# Wskaźniki fundamentalne (UWAGA: Z-score surowy, bez .clip()!)
fund_indicators_final = []
available_funds = [c for c in fund_ind_raw if c in df_global.columns]

for metric in available_funds:
    roll_mean = df_global.groupby('tic')[metric].transform(lambda x: x.rolling(window=252, min_periods=30).mean())
    roll_std = df_global.groupby('tic')[metric].transform(lambda x: x.rolling(window=252, min_periods=30).std())
    zscore_col = f"{metric}_zscore"
    # Wyliczamy Z-score, ale usuwamy stąd .clip(-5, 5).fillna(0) na czas diagnostyki
    df_global[zscore_col] = (df_global[metric] - roll_mean) / (roll_std + 1e-6)
    fund_indicators_final.append(zscore_col)

# Dynamiczna konwersja MACD, ATR, DMA oraz MSTD
for col in df_global.columns:
    if (col.startswith('macd') or col.startswith('atr_') or col.startswith('mstd_') or col == 'dma') and not col.endswith('_pct'):
        pct_col = f"{col}_pct"
        df_global[pct_col] = df_global[col] / df_global['close'].replace(0, np.nan)
        df_global[pct_col] = df_global[pct_col].fillna(0)
        
# Wskaźnik OBV (On-Balance Volume)
direction = np.sign(df_global.groupby('tic')['close'].diff()).fillna(0)
df_global['obv_step'] = direction * df_global['volume']
df_global['obv'] = df_global.groupby('tic')['obv_step'].cumsum()
df_global = df_global.drop(columns=['obv_step'])

obv_mean = df_global.groupby('tic')['obv'].transform(lambda x: x.rolling(window=60, min_periods=10).mean())
obv_std = df_global.groupby('tic')['obv'].transform(lambda x: x.rolling(window=60, min_periods=10).std())
# Tutaj również zostawiamy na razie surowy wynik
df_global['obv_zscore'] = (df_global['obv'] - obv_mean) / (obv_std + 1e-6)

# =========================================================================
# --- 6.1. DIAGNOSTYKA OUTLIERÓW (Przed przycięciem danych) ---
# =========================================================================
print("\n=== [DIAGNOSTYKA] ROZPOCZĘCIE ANALIZY OUTLIERÓW ===")

# Definiujemy zmienne do sprawdzenia
vars_to_check = fund_indicators_final + ['obv_zscore', 'volume_rel']

# Filtrujemy dane tylko od momentu analizy (bez rozgrzewki), aby statystyki były wiarygodne
df_diag = df_global[df_global['date'] >= ANALYSIS_START_DATE].dropna(subset=vars_to_check).copy()

print("\n--- TABELA ROZKŁADU (KWANTYLE) ---")
percentiles = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
summary_stats = df_diag[vars_to_check].describe(percentiles=percentiles).T
display_stats = summary_stats[['min', '0.1%', '1%', '5%', '50%', '95%', '99%', '99.9%', 'max']]
display(display_stats.round(2))

# Wizualizacja wybranych zmiennych (np. 3 fundamenty + wolumen)
try:
    vars_to_plot = fund_indicators_final[:3] + ['volume_rel']
    fig, axes = plt.subplots(len(vars_to_plot), 2, figsize=(16, 4 * len(vars_to_plot)))
    fig.suptitle("Analiza Anomalii (Outlierów) przed zastosowaniem .clip()", fontsize=16, y=1.02)

    for i, var in enumerate(vars_to_plot):
        plot_data = df_diag[var][df_diag[var].between(-50, 50)] 
        
        # Histogram
        sns.histplot(plot_data, bins=100, ax=axes[i, 0], color='blue', kde=True, stat='density')
        axes[i, 0].set_title(f'Histogram: {var}')
        axes[i, 0].set_ylabel('Gęstość')
        axes[i, 0].set_xlabel('Wartość')
        
        # Boxplot
        sns.boxplot(x=plot_data, ax=axes[i, 1], color='orange')
        axes[i, 1].set_title(f'Boxplot: {var}')
        axes[i, 1].set_xlabel('Wartość')
        
        # Granice przycinania
        clip_lower = 0 if 'volume' in var else -5
        clip_upper = 20 if 'volume' in var else 5
        
        for ax in (axes[i, 0], axes[i, 1]):
            ax.axvline(clip_lower, color='red', linestyle='--', linewidth=2, label=f'Clip: {clip_lower}')
            ax.axvline(clip_upper, color='red', linestyle='--', linewidth=2, label=f'Clip: {clip_upper}')
        axes[i, 0].legend()

    plt.tight_layout()
    
    # Zapis
    save_dir = OUTPUT_MODELS_DIR if 'OUTPUT_MODELS_DIR' in locals() else os.getcwd()
    plot_path = os.path.join(save_dir, "outliers_analysis.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"\n[ZAPIS] Wykres diagnostyczny zapisano: {plot_path}")
    plt.show() 
    plt.close()
except Exception as e:
    print(f"[BŁĄD WIZUALIZACJI] Nie udało się wygenerować wykresów: {e}")

print("\n--- WPŁYW PRZYCINANIA (CLIPPING) ---")
for var in vars_to_check:
    clip_lower = 0 if 'volume' in var else -5
    clip_upper = 20 if 'volume' in var else 5
    
    total_samples = len(df_diag[var])
    outliers_lower = len(df_diag[df_diag[var] < clip_lower])
    outliers_upper = len(df_diag[df_diag[var] > clip_upper])
    outlier_pct = ((outliers_lower + outliers_upper) / total_samples) * 100
    print(f"{var:25}: Ucięto {outliers_lower:<5} (dół), {outliers_upper:<5} (góra). Łącznie: {outlier_pct:.2f}% danych.")

del df_diag # Zwalnianie pamięci po diagnostyce

# =========================================================================
# --- 6.2. WŁAŚCIWY CLIPPING DANYCH ---
# =========================================================================
print("\n[SYSTEM] Aplikowanie sztywnych ram .clip() do wskaźników w głównym zbiorze...")
df_global['volume_rel'] = df_global['volume_rel'].fillna(1.0).clip(0, 20)
df_global['obv_zscore'] = df_global['obv_zscore'].fillna(0).clip(-5, 5)

for col in fund_indicators_final:
    df_global[col] = df_global[col].clip(-5, 5)

# --- 7. PRZYCINANIE DANYCH (WARM-UP CUT) I CZYSZCZENIE BŁĘDÓW MATEMATYCZNYCH ---
print(f"[SYSTEM] Odcinanie okresu rozgrzewkowego (przed {ANALYSIS_START_DATE.date()})...")
df_global = df_global[df_global['date'] >= ANALYSIS_START_DATE].copy()

# 1. Zamieniamy wszelkie nieskończoności (powstałe np. z dzielenia przez zerowy wolumen lub brak spadków w VR) na NaN
df_global = df_global.replace([np.inf, -np.inf], np.nan)

# 2. ŁATKA NA WSKAŹNIKI WOLUMENOWE (VR) - Wypełnianie NaN (wynikających z ekstremalnych trendów)
vr_columns = [col for col in df_global.columns if col.startswith('vr_')]
for col in vr_columns:
    if col in df_global.columns:
        # Prawidłowe zastosowanie groupby i transformacji w Pandas
        df_global[col] = df_global.groupby('tic')[col].transform(lambda x: x.ffill().fillna(100.0))

# --- DIAGNOSTYKA BRAKÓW DANYCH (Wypisywanie uszkodzonych wierszy) ---
problematic_mask = df_global.isna().any(axis=1)

if problematic_mask.any():
    problematic_df = df_global[problematic_mask]
    print(f"\n[UWAGA] Znaleziono {len(problematic_df)} wierszy z wartościami NaN/Inf. Detale:")
    
    # Iterujemy tylko po uszkodzonych wierszach
    for index, row in problematic_df.iterrows():
        # Wyciągamy nazwy kolumn, które w tym konkretnym wierszu mają wartość NaN
        bad_cols = row.index[row.isna()].tolist()
        
        p_date = row['date'].date() if 'date' in row else "Brak daty"
        p_tic = row['tic'] if 'tic' in row else "Brak tickera"
        
        print(f" -> [{p_date}] Ticker: {p_tic:<10} | Błędne zmienne: {bad_cols}")
    print("-" * 60)
else:
    print("\n[DIAGNOSTYKA] Nie znaleziono żadnych braków danych (NaN/Inf). Dane są czyste.")

# Ostateczne, twarde usunięcie braków po okresie rozgrzewkowym
initial_rows = len(df_global)
df_global = df_global.dropna()
print(f"[SYSTEM] Oczyszczono dane z NaN/Inf. Usunięto {initial_rows - len(df_global)} problematycznych wierszy. Pozostało {len(df_global)}.")

# --- 8. DEFINICJA STANU I FINALIZACJA ---

# 1. Zbieramy wszystkie nasze przekształcone cechy procentowe i z-score
custom_and_pct_features = [col for col in df_global.columns if col.endswith('_pct') or col in ['boll_pct_b', 'boll_width', 'volume_rel', 'obv_zscore']]

# 2. Odrzucamy surowe wskaźniki, które już przekształciliśmy, by uniknąć duplikatów i braku stacjonarności
features_to_exclude = [
    c for c in tech_ind_params 
    if (c.startswith('close_') and any(c.endswith(suf) for suf in ('_sma', '_ema', '_smma', '_lrma', '_kama')))
    or c == 'tema'
    or c == 'dma'
    or c.startswith('macd') 
    or c.startswith('atr') 
    or c.startswith('mstd') # <--- ZMIANA TUTAJ (Dodano mstd)
    or c.startswith('boll')
]

# 3. Wyciągamy resztę bezpiecznych wskaźników (oscylatory, momentum) wygenerowanych przez FinRL
safe_raw_features = [c for c in tech_ind_params if c not in features_to_exclude and c in df_global.columns]

# 4. Łączymy wszystko w jeden potężny wektor stanu!
tech_indicators_final = dynamic_distance_features + custom_and_pct_features + safe_raw_features

macro_indicators_final = ["wig20_volatility", "usdpln_ret", "bond_10y_diff", "sp500_ret"]
indicators = tech_indicators_final + macro_indicators_final + fund_indicators_final

stock_dim = len(FILES_CONFIG)
state_dim = 1 + 2*stock_dim + len(indicators)*stock_dim

print(f"\n[SYSTEM] State Dim: {state_dim}")
print(f"[SYSTEM] Liczba cech na jedną spółkę ({len(indicators)}): {indicators}")

df_global = df_global.sort_values(['tic', 'date']).reset_index(drop=True)

df_global['next_day_open'] = df_global.groupby('tic')['open'].shift(-1)
df_global['next_day_open'] = df_global['next_day_open'].fillna(df_global['close'])
df_global = df_global.sort_values(['date', 'tic']).reset_index(drop=True)

df_global.index = df_global.date.factorize()[0]

# --- 9. GENERATOR DAT WALK-FORWARD ---
ANALYSIS_END_DATE = pd.Timestamp("2021-01-01")
GLOBAL_DATES = []
curr = ANALYSIS_START_DATE + relativedelta(years=TRAIN_YEARS)

while curr < ANALYSIS_END_DATE:
    GLOBAL_DATES.append(curr)
    curr += relativedelta(months=TEST_MONTHS)

print(f"[GOTOWE] Dane przetworzone. Liczba wierszy (po przycięciu): {len(df_global)}")
print(f"[PRZYKŁAD] Pierwszy Test Start: {GLOBAL_DATES[0]}")

Data początkowa treningu (YYYY-MM-DD) [2014-01-01]:  


[KONFIG] Start Obliczeń (Warm-up): 2013-01-01
[KONFIG] Start Analizy (Treningu): 2014-01-01

[SYSTEM] Rozpoczynam przetwarzanie danych (od 2013-01-01)...
[FE] Obliczanie wskaźników technicznych (FinRL)...
Successfully added technical indicators
Successfully added turbulence index
[FE] Integracja danych MAKRO...
[FE] Pobieranie danych S&P 500 z yfinance...
[FE] Automatyczna konwersja wskaźników cenowych...
[SYSTEM] Utworzono 19 wskaźników dystansu.

=== [DIAGNOSTYKA] ROZPOCZĘCIE ANALIZY OUTLIERÓW ===

--- TABELA ROZKŁADU (KWANTYLE) ---


,min,0.1%,1%,5%,50%,95%,99%,99.9%,max
pe_ratio_zscore,-15.8100,-6.3000,-3.3000,-1.8700,-0.0100,2.1400,3.6500,6.8400,15.6300
ps_ratio_zscore,-12.5500,-5.1600,-2.9900,-2.1100,-0.1000,2.1800,3.0600,4.2800,8.4700
pb_ratio_zscore,-10.8500,-5.3800,-3.0800,-2.1800,-0.1500,2.2900,3.1400,4.6400,13.3500
p_fcf_ratio_zscore,-15.7700,-7.5500,-3.6400,-2.0000,0.0000,1.8800,3.4500,6.9400,14.6300
ev_ebitda_zscore,-15.7400,-6.5300,-3.1300,-1.9000,-0.2000,2.0400,3.4600,7.0600,15.3400
roe_zscore,-15.2500,-7.5500,-3.7500,-2.2300,0.0100,1.8400,2.9600,5.6800,13.9400
net_margin_zscore,-15.4500,-7.5100,-3.9000,-2.3200,0.1100,1.7800,2.7700,4.5900,14.5700
debt_ratio_zscore,-15.7700,-5.7000,-3.1300,-2.0500,0.1700,2.4200,3.8900,6.5900,11.6800
net_to_ocf_zscore,-15.8100,-7.8400,-3.7700,-2.1100,0.0400,2.0500,3.5600,7.1800,15.8100
obv_zscore,-5.9400,-4.1000,-2.9900,-2.1900,0.1900,2.2300,3.0700,4.1800,6.6100



[ZAPIS] Wykres diagnostyczny zapisano: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/modele/outliers_analysis.png

--- WPŁYW PRZYCINANIA (CLIPPING) ---
pe_ratio_zscore          : Ucięto 123   (dół), 159   (góra). Łącznie: 0.59% danych.
ps_ratio_zscore          : Ucięto 55    (dół), 13    (góra). Łącznie: 0.14% danych.
pb_ratio_zscore          : Ucięto 62    (dół), 34    (góra). Łącznie: 0.20% danych.
p_fcf_ratio_zscore       : Ucięto 184   (dół), 148   (góra). Łącznie: 0.69% danych.
ev_ebitda_zscore         : Ucięto 114   (dół), 154   (góra). Łącznie: 0.56% danych.
roe_zscore               : Ucięto 185   (dół), 73    (góra). Łącznie: 0.54% danych.
net_margin_zscore        : Ucięto 202   (dół), 32    (góra). Łącznie: 0.49% danych.
debt_ratio_zscore        : Ucięto 83    (dół), 183   (góra). Łącznie: 0.55% danych.
net_to_ocf_zscore        : Ucięto 199   (dół), 157   (góra). Łącznie: 0.74% danych.
obv_zscore               : Ucięto 11    (dół), 7     (góra). Łącznie: 0.04% d

### Benchmarki
Generator plików z przebiegiem krzywych kapitału, wykresy, metryki

In [11]:
# ==============================================================================
# GENERATOR BENCHMARKÓW - pliki excel
# ==============================================================================

# --- 1. INPUT OD UŻYTKOWNIKA ---
print(f"Dostępne okresy: {list(QUARTERS_MAP.keys())}")
user_start = input("Podaj początek przedziału (np. 'Q1 2020') [Domyślnie Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Podaj koniec przedziału (np. 'Q4 2020') [Domyślnie Q4 2020]: ").strip() or "Q4 2020"

start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

# Zabezpieczenie przed odwrotnym wprowadzeniem dat
if start_idx > end_idx:
    start_idx, end_idx = end_idx, start_idx
    user_start, user_end = user_end, user_start

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")

# Dynamiczne formatowanie nazwy pliku wyjściowego w zależności od wybranego okresu
if start_idx == end_idx:
    period_suffix = f"_{safe_start_str}"
    period_display = f"{user_start}"
else:
    period_suffix = f"_{safe_start_str}_{safe_end_str}"
    period_display = f"{user_start} - {user_end}"

print(f"\n[BENCHMARK] Uruchamianie generatora benchmarków (Okres: {period_display})...")

# --- 2. PRZYGOTOWANIE DANYCH ---
if 'df_global' not in locals():
    raise ValueError("Brak df_global! Uruchom najpierw Cell 0b.")

if 'GLOBAL_DATES' not in locals() or len(GLOBAL_DATES) == 0:
    raise ValueError("GLOBAL_DATES jest puste! Uruchom Cell 0b.")

print("[BENCHMARK] Przygotowanie danych...")

df_bench = df_global.copy()
# Normalizacja dat (ucinanie godzin) zapewnia poprawne działanie słowników Lookup
df_bench['date'] = pd.to_datetime(df_bench['date']).dt.normalize()
df_bench = df_bench.sort_values(['tic', 'date']).reset_index(drop=True)

# Lookup Tables: Słowniki w pamięci RAM zapewniające błyskawiczne pobieranie cen w pętli symulacji
PRICE_LOOKUP = df_bench.set_index(['date', 'tic'])['close'].to_dict()
OPEN_LOOKUP = df_bench.set_index(['date', 'tic'])['open'].to_dict() if 'open' in df_bench.columns else PRICE_LOOKUP
if 'turbulence' not in df_bench.columns: 
    df_bench['turbulence'] = 0.0
TURBULENCE_LOOKUP = df_bench.set_index(['date', 'tic'])['turbulence'].to_dict()

unique_tickers = sorted(list(set(df_bench['tic'].unique())))

# --- 3. FUNKCJE POMOCNICZE ---

def insert_period_breaks(df, break_dates):
    """
    Funkcja wstawia pusty wiersz (NaN) do wyeksportowanego pliku Excel za każdym razem, 
    gdy następuje nowy okres Walk-Forward. Ułatwia to wizualną weryfikację rebalansowania
    w arkuszu kalkulacyjnym.
    """
    if df.empty: return df
    df = df.sort_values('date').reset_index(drop=True)
    list_of_dfs, prev_idx = [], 0
    df_dates = pd.to_datetime(df['date']).dt.normalize()
    unique_breaks = sorted(list(set(pd.to_datetime(break_dates).normalize())))
    
    for b_date in unique_breaks:
        mask = df_dates >= b_date
        if not mask.any(): continue
        idx = mask.idxmax()
        if idx > prev_idx:
            list_of_dfs.append(df.iloc[prev_idx:idx])
            list_of_dfs.append(pd.DataFrame([[np.nan] * len(df.columns)], columns=df.columns))
            prev_idx = idx
    list_of_dfs.append(df.iloc[prev_idx:])
    return pd.concat(list_of_dfs).reset_index(drop=True)

def get_optimized_weights(strategy_name, df_close):
    """
    Wyznacza wagi portfela za pomocą nowoczesnej teorii portfelowej (Markowitz).
    """
    try:
        df_clean = df_close.dropna(axis=1, how='any') 
        if df_clean.empty or df_clean.shape[1] < 2: return {}

        returns = df_clean.pct_change().dropna()

        if strategy_name == "MIN_VAR":
            # Szuka portfela o historycznie najmniejszej zmienności (w oparciu o macierz kowariancji)
            S = risk_models.sample_cov(df_clean)
            S = risk_models.fix_nonpositive_semidefinite(S) 
            ef = EfficientFrontier(None, S)
            return dict(ef.min_volatility())

        elif strategy_name == "MEAN_VAR":
            # Optymalizacja pod kątem Sortino (maksymalizacja zysku do ryzyka spadków).
            # Zapewnia równe szanse w porównaniu z modelami RL, które optymalizują ten sam parametr.
            mu = expected_returns.mean_historical_return(df_clean)
            # Używamy EfficientSemivariance (kary za wariancję ujemną) zamiast klasycznego EfficientFrontier
            es = EfficientSemivariance(mu, returns)
            
            # max_quadratic_utility to matematyczny ekwiwalent optymalizacji
            # zysku skorygowanego o ryzyko asymetryczne (Sortino) w pypfopt.
            es.max_quadratic_utility() 
            
            return dict(es.clean_weights())
            
    except Exception as e:
        print(f"[DEBUG] Błąd optymalizacji {strategy_name}: {e}")
        return {}
    return {}

# --- 4. GŁÓWNY SILNIK SYMULACJI ---
def run_benchmark_simulation(strategy_name, dates_list, tickers_list):
    """
    Przeprowadza krok po kroku (wektorowo) symulację handlu dla wybranych benchmarków.
    Naśladuje opóźnienia decyzyjne z modelu RL: decyzja podejmowana jest w dniu T, 
    a egzekucja dokonywana jest po cenie Otwarcia (Open) w dniu T+1.
    """
    print(f"    --> Strategia: {strategy_name}...")
    current_cash = float(INITIAL_CAPITAL)
    current_holdings = {t: 0.0 for t in tickers_list}
    history = []
    
    is_bnh = (strategy_name == "BUY_AND_HOLD")
    bnh_bought = False 

    # Pętla po kwartałach Walk-Forward
    for i_period, start_date in enumerate(dates_list):
        end_date = start_date + relativedelta(months=TEST_MONTHS)
        period_mask = (df_bench['date'] >= start_date) & (df_bench['date'] < end_date)
        period_days = sorted(df_bench.loc[period_mask, 'date'].unique())
        if not period_days: continue
            
        decision_day = period_days[0]
        # Wykonanie zlecenia następnego dnia roboczego
        execution_day = period_days[1] if len(period_days) > 1 else period_days[0]
        
        target_weights = {}
        do_rebalance = False
        
        # Logika strategii statycznej (Kup i Trzymaj)
        if is_bnh:
            if not bnh_bought:
                # Na starcie alokuje kapitał po równo w każdą spółkę (Equal Weight)
                target_weights = {t: 1.0/len(tickers_list) for t in tickers_list}
                do_rebalance = True; bnh_bought = True
        # Logika strategii optymalizacyjnych
        else:
            lookback_start = start_date - relativedelta(years=TRAIN_YEARS)
            train_mask = (df_bench.date >= lookback_start) & (df_bench.date < start_date)
            train_df = df_bench.loc[train_mask]
            
            if not train_df.empty:
                train_piv = train_df.pivot(index='date', columns='tic', values='close').ffill().fillna(0)
                target_weights = get_optimized_weights(strategy_name, train_piv)
                do_rebalance = True

        trades_to_execute = {t: 0.0 for t in tickers_list}
        # Dzień Decyzji: Przeliczenie modelowych wag na konkretną liczbę akcji
        if do_rebalance:
            prices_close = {t: PRICE_LOOKUP.get((decision_day, t), 0.0) for t in tickers_list}
            equity = current_cash + sum(current_holdings[t] * prices_close[t] for t in tickers_list)
            
            for t in tickers_list:
                w = target_weights.get(t, 0.0)
                p = prices_close[t]
                if p > 0:
                    target_shares = (equity * w) / (p * (1 + COST))
                    target_shares = np.floor(target_shares * 10000) / 10000.0
                    trades_to_execute[t] = np.round(target_shares - current_holdings[t], 4)
        
        # Dzień Wykonania: Fizyczna egzekucja zleceń na rynku (kupno/sprzedaż)
        for current_day in period_days:
            trades_done = {t: 0.0 for t in tickers_list}
            if current_day == execution_day and do_rebalance:
                open_prices = {t: OPEN_LOOKUP.get((current_day, t), 0.0) for t in tickers_list}
                
                # Zlecenia sprzedaży realizowane są ZAWSZE w pierwszej kolejności, 
                # aby uwolnić gotówkę niezbędną na nowe zakupy (ograniczenie kapitałowe brokera)
                for t, qty in trades_to_execute.items():
                    if qty < 0:
                        p = open_prices.get(t, 0.0)
                        if p > 0:
                            qty_sell = min(abs(qty), current_holdings[t]) 
                            qty_sell = np.round(qty_sell, 4) 
                            current_cash += qty_sell * p * (1 - COST)
                            current_holdings[t] -= qty_sell
                            current_holdings[t] = np.round(current_holdings[t], 4) 
                            trades_done[t] = -qty_sell
                
                # Faza zakupów z uwzględnieniem odblokowanej gotówki i kosztów transakcyjnych
                for t, qty in trades_to_execute.items():
                    if qty > 0:
                        p = open_prices.get(t, 0.0)
                        if p > 0:
                            cost = qty * p * (1 + COST)
                            # Zabezpieczenie przed brakiem środków (kupujemy za tyle, na ile nas stać)
                            if cost > current_cash:
                                qty = current_cash / (p * (1 + COST))
                                qty = np.floor(qty * 10000) / 10000.0 
                                cost = qty * p * (1 + COST)
                            
                            current_cash -= cost
                            current_holdings[t] += qty
                            current_holdings[t] = np.round(current_holdings[t], 4) 
                            trades_done[t] = qty
                do_rebalance = False 

            # Agregacja dobowych statystyk portfela w ujęciu Mark-to-Market
            close_prices = {t: PRICE_LOOKUP.get((current_day, t), 0.0) for t in tickers_list}
            stock_val = sum(current_holdings[t] * close_prices[t] for t in tickers_list)
            
            row = {
                'date': current_day,
                'turbulence': TURBULENCE_LOOKUP.get((current_day, tickers_list[0]), 0),
                'account_value': current_cash + stock_val, 
                'Cash_PLN': current_cash
            }
            for t in tickers_list:
                row[f'Signal_{t}'] = trades_to_execute[t] if current_day == decision_day else 0.0
                row[f'Trade_{t}'] = trades_done[t]
                row[f'Hold_{t}'] = current_holdings[t]
            history.append(row)

    return pd.DataFrame(history)

# --- 5. URUCHOMIENIE I ZAPIS ---
try:
    selected_dates = GLOBAL_DATES[start_idx : end_idx + 1]

    if len(selected_dates) == 0:
        print(f"\n[SKIP] Brak dostępnych dat dla podanego zakresu: {period_display}")
    else:
        print(f"\n=== GENEROWANIE: Okres {period_display} ({len(selected_dates)} okresów) ===")
        file_path = os.path.join(BENCHMARK_DIR, f"Benchmarki{period_suffix}.xlsx")
        
        bench_results = {}
        bench_results["BUY_AND_HOLD"] = run_benchmark_simulation("BUY_AND_HOLD", selected_dates, unique_tickers)
        bench_results["MIN_VAR"] = run_benchmark_simulation("MIN_VAR", selected_dates, unique_tickers)
        bench_results["MEAN_VAR"] = run_benchmark_simulation("MEAN_VAR", selected_dates, unique_tickers)
        
        # Eksport do Excela z formatowaniem komórek dla czytelności analityka
        with pd.ExcelWriter(file_path, engine='xlsxwriter') as w:
            workbook = w.book
            fmt_float = workbook.add_format({'num_format': '0.00'})
            
            for name, df in bench_results.items():
                if df.empty: continue
                cols = ['date', 'turbulence', 'account_value', 'Cash_PLN']
                for t in unique_tickers:
                    cols.extend([f'Signal_{t}', f'Trade_{t}', f'Hold_{t}'])
                
                valid_cols = [c for c in cols if c in df.columns]
                final_df = insert_period_breaks(df[valid_cols], selected_dates)
                final_df.to_excel(w, sheet_name=name, index=False)
                
                ws = w.sheets[name]
                ws.set_column(0, 0, 12) 
                ws.set_column(1, len(valid_cols)-1, 10, fmt_float)

        print(f"[SUKCES] Zapisano plik benchmarków: {file_path}")

except Exception as e:
    print(f"\n[BŁĄD KRYTYCZNY] {e}")
    traceback.print_exc()

Dostępne okresy: ['Q1 2020', 'Q2 2020', 'Q3 2020', 'Q4 2020']


Podaj początek przedziału (np. 'Q1 2020') [Domyślnie Q1 2020]:  
Podaj koniec przedziału (np. 'Q4 2020') [Domyślnie Q4 2020]:  



[BENCHMARK] Uruchamianie generatora benchmarków (Okres: Q1 2020 - Q4 2020)...
[BENCHMARK] Przygotowanie danych...

=== GENEROWANIE: Okres Q1 2020 - Q4 2020 (4 okresów) ===
    --> Strategia: BUY_AND_HOLD...
    --> Strategia: MIN_VAR...
    --> Strategia: MEAN_VAR...
[SUKCES] Zapisano plik benchmarków: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/benchmarki/Benchmarki_Q1_2020_Q4_2020.xlsx


In [32]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ==============================================================================
# WIZUALIZACJA BENCHMARKÓW
# ==============================================================================

# Flaga decydująca o rysowaniu tytułu wykresu
SHOW_TITLES = False 

print(f"Dostępne okresy: {list(QUARTERS_MAP.keys())}")
user_start = input("Z jakiego początku analizować dane (np. 'Q1 2020') [Domyślnie Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Do jakiego końca analizować dane (np. 'Q4 2020') [Domyślnie Q4 2020]: ").strip() or "Q4 2020"

start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

if start_idx > end_idx:
    start_idx, end_idx = end_idx, start_idx
    user_start, user_end = user_end, user_start

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")

# Zabezpieczenie nazw plików dla symulacji kwartalnych vs wieloletnich
if start_idx == end_idx:
    period_suffix = f"_{safe_start_str}"
    title_text = f"Okres: {user_start}"
else:
    period_suffix = f"_{safe_start_str}_{safe_end_str}"
    title_text = f"Okres: {user_start} do {user_end}"

# Konfiguracja estetyki wykresów
STYLES = {
    'BUY_AND_HOLD': ('black',   '-', 'Buy & Hold'),
    'MIN_VAR':      ('#E377C2', '-', 'Min-Variance'),  # Cyjan
    'MEAN_VAR':     ('#A0522D', '-', 'Mean-Semivariance')  # Różowy
}

excel_name = f"Benchmarki{period_suffix}.xlsx"
img_name = f"Wykres_Benchmarkow{period_suffix}.png"

bench_file_path = os.path.join(BENCHMARK_DIR, excel_name)
output_img_path = os.path.join(BENCHMARK_DIR, img_name)

print("\n=== WIZUALIZACJA BENCHMARKÓW ===")

# --- 2. GENEROWANIE WYKRESU ---
if not os.path.exists(bench_file_path):
    print(f"❌ [BŁĄD] Nie znaleziono pliku: {excel_name}. Wygeneruj go najpierw w module Benchmarków.")
else:
    print(f"[PLOT] Przetwarzanie: {excel_name} -> {img_name}")
    
    try:
        fig, ax = plt.subplots(figsize=(12, 7), facecolor='white')
        has_data = False

        # Pętla ładuje kolejne zakładki z Excela reprezentujące poszczególne benchmarki
        for sheet_name, (color, style, label) in STYLES.items():
            try:
                df = pd.read_excel(bench_file_path, sheet_name=sheet_name)
                
                # Czyszczenie technicznych przerw
                df = df.dropna(subset=['date', 'account_value'])
                df['date'] = pd.to_datetime(df['date'])
                df = df.sort_values('date')
                
                if df.empty: continue
                has_data = True

                # Zmiana wartości bezwzględnych (PLN) na Skumulowaną Stopę Zwrotu.
                initial_val = df['account_value'].iloc[0]
                df['cum_return'] = (df['account_value'] / initial_val) - 1
                
                ax.plot(df['date'], df['cum_return'], 
                        label=label, color=color, linestyle=style, linewidth=2.5)
                
            except ValueError:
                pass 

        if has_data:
            # Warunkowe rysowanie tytułu
            if SHOW_TITLES:
                ax.set_title(f"Porównanie Strategii Benchmarkowych\n({title_text})", fontsize=15, fontweight='bold', pad=15)
            
            # Etykiety osi (font 20) i wartości na osiach (font 15)
            ax.set_ylabel("Skumulowana Stopa Zwrotu (%)", fontsize=20)
            ax.set_xlabel("Data", fontsize=20)
            ax.tick_params(axis='both', labelsize=15)
            
            # Formatowanie osi OY
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
            ax.axhline(0, color='red', linewidth=1, linestyle=':', alpha=0.7)
            ax.grid(True, which='major', linestyle='--', alpha=0.5)
            
            # Legenda (font 15)
            ax.legend(loc='upper left', frameon=True, shadow=True, fontsize=15)
            
            plt.tight_layout()
            plt.savefig(output_img_path, dpi=300) 
            print(f"[SUKCES] Wykres zapisany jako: {img_name}")
            plt.show()
        else:
            plt.close()
            print("[INFO] Brak poprawnych danych do wyrysowania w wybranym pliku.")

    except Exception as e:
        print(f"❌ Błąd podczas rysowania {excel_name}: {e}")

Dostępne okresy: ['Q1 2020', 'Q2 2020', 'Q3 2020', 'Q4 2020']


Z jakiego początku analizować dane (np. 'Q1 2020') [Domyślnie Q1 2020]:  
Do jakiego końca analizować dane (np. 'Q4 2020') [Domyślnie Q4 2020]:  



=== WIZUALIZACJA BENCHMARKÓW ===
[PLOT] Przetwarzanie: Benchmarki_Q1_2020_Q4_2020.xlsx -> Wykres_Benchmarkow_Q1_2020_Q4_2020.png
[SUKCES] Wykres zapisany jako: Wykres_Benchmarkow_Q1_2020_Q4_2020.png


In [17]:
# ==============================================================================
# BENCHMARKI - TABELA METRYK
# ==============================================================================

# --- 1. KONFIGURACJA I INPUT ---

# Stopa wolna od ryzyka (dla modeli RL badacze z reguły przyjmuja 0, dlatego dla benchmarków należy przyjac tyle samo)
RISK_FREE_RATE = 0.00 

print(f"Dostępne okresy: {list(QUARTERS_MAP.keys())}")
user_start = input("Początek przedziału dla metryk (np. 'Q1 2020') [Domyślnie Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Koniec przedziału dla metryk (np. 'Q4 2020') [Domyślnie Q4 2020]: ").strip() or "Q4 2020"

start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

if start_idx > end_idx:
    start_idx, end_idx = end_idx, start_idx
    user_start, user_end = user_end, user_start

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")

if start_idx == end_idx:
    period_suffix = f"_{safe_start_str}"
    header_text = f"WYNIKI DLA {user_start}"
else:
    period_suffix = f"_{safe_start_str}_{safe_end_str}"
    header_text = f"WYNIKI DLA {user_start} - {user_end}"

excel_name = f"Benchmarki{period_suffix}.xlsx"
csv_name = f"Metryki{period_suffix}.csv"

# --- 2. FUNKCJA LICZĄCA ---
def calculate_metrics(df):
    """
    Kluczowy agregator statystyk oceny ryzyka do pracy magisterskiej.
    Oblicza zannualizowane wskaźniki efektywności używając standardu 252 dni handlowych.
    """
    # Usuwamy wiersze separatorów (NaN), które wstawia funkcja insert_period_breaks
    df = df.dropna(subset=['date', 'account_value']).copy()
    
    if df.empty or len(df) < 2:
        return None
    
    # 1. Total Return (Całkowita zmiana wartości portfela)
    start_val = df['account_value'].iloc[0]
    end_val = df['account_value'].iloc[-1]
    cum_ret = (end_val / start_val) - 1
    
    # 2. Daily Returns (Dzienne stopy zwrotu - baza matematyczna pod wskaźniki asymetryczne)
    df['daily_ret'] = df['account_value'].pct_change().fillna(0)
    mean_ret = df['daily_ret'].mean()
    std_ret = df['daily_ret'].std()
    
    # 3. Sharpe Ratio (Annualized)
    # Ocena zysku skorygowanego o całkowitą zmienność (karze również za duże wzrosty w krótkim czasie)
    if std_ret == 0:
        sharpe = 0.0
    else:
        sharpe = (mean_ret * 252 - RISK_FREE_RATE) / (std_ret * np.sqrt(252))
        
    # 4. Sortino Ratio (Annualized)
    # Złoty standard w handlu algorytmicznym. Ocenia zysk uwzględniając WYŁĄCZNIE zmienność 
    # portfela w dół (downside risk). Dni z zyskiem nie obniżają oceny działania modelu.
    downside_returns = df.loc[df['daily_ret'] < 0, 'daily_ret']
    downside_std = downside_returns.std()
    
    if downside_std == 0 or pd.isna(downside_std):
        sortino = sharpe if mean_ret < 0 else 0.0 # Zabezpieczenie przed dzieleniem przez zero
    else:
        sortino = (mean_ret * 252 - RISK_FREE_RATE) / (downside_std * np.sqrt(252))

    # 5. Max Drawdown (Maksymalne obsunięcie kapitału)
    # Wyznaczenie najgłębszego procentowego dołka od historycznego szczytu wyceny
    df['cum_max'] = df['account_value'].cummax()
    df['drawdown'] = (df['account_value'] - df['cum_max']) / df['cum_max']
    max_dd = df['drawdown'].min()
    
    # 6. Calmar Ratio (POPRAWIONE NA BAZIE CAGR)
    # Stosunek skumulowanej rocznej stopy zwrotu (CAGR) do najgorszego zanotowanego dołka (Max DD).
    # Idealny wskaźnik mierzący odporność portfela na krachy rynkowe (np. Covid-19 w 2020 r.).
    total_days = len(df)
    total_years = total_days / 252.0
    
    if total_years > 0:
        # Skumulowana Roczna Stopa Zwrotu (CAGR)
        cagr = ((1 + cum_ret) ** (1 / total_years)) - 1
    else:
        cagr = 0.0
        
    if max_dd == 0:
        calmar = 0.0
    else:
        calmar = cagr / abs(max_dd)
    
    return {
        'Total Return': cum_ret,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Calmar Ratio': calmar,
        'Max Drawdown': max_dd
    }

# --- 3. ANALIZA ---
file_path = os.path.join(BENCHMARK_DIR, excel_name)

if not os.path.exists(file_path):
    print(f"❌ BŁĄD: Nie znaleziono pliku: {excel_name}")
else:
    print(f"\n{'='*70}")
    print(f"   {header_text}")
    print(f"{'='*70}")
    
    try:
        excel_file = pd.ExcelFile(file_path)
        results = []
        
        for sheet_name in excel_file.sheet_names:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            metrics = calculate_metrics(df)
            
            if metrics:
                results.append({
                    'Strategy': sheet_name,
                    'Total Return': metrics['Total Return'],
                    'Sharpe Ratio': metrics['Sharpe Ratio'],
                    'Sortino Ratio': metrics['Sortino Ratio'],
                    'Calmar Ratio': metrics['Calmar Ratio'],
                    'Max Drawdown': metrics['Max Drawdown']
                })
        
        if results:
            results_df = pd.DataFrame(results).set_index('Strategy')
            
            # Sortowanie tabeli po współczynniku Sortino Ratio (zgodnie z logiką funkcji nagrody w RL)
            results_df = results_df.sort_values('Sortino Ratio', ascending=False)
            
            # Formatowanie tabeli wyłącznie w celu czytelnego wyświetlania w konsoli
            display_df = results_df.copy()
            display_df['Total Return'] = display_df['Total Return'].apply(lambda x: f"{x:+.2%}")
            display_df['Max Drawdown'] = display_df['Max Drawdown'].apply(lambda x: f"{x:.2%}")
            display_df['Sharpe Ratio'] = display_df['Sharpe Ratio'].apply(lambda x: f"{x:.4f}")
            display_df['Sortino Ratio'] = display_df['Sortino Ratio'].apply(lambda x: f"{x:.4f}")
            display_df['Calmar Ratio'] = display_df['Calmar Ratio'].apply(lambda x: f"{x:.4f}")
            
            print(display_df.to_string())
            
            # Zapis twardych, niefilterowanych danych na dysk do późniejszych analiz
            out_csv = os.path.join(BENCHMARK_DIR, csv_name)
            results_df.to_csv(out_csv)
            print(f"\n[SUKCES] Metryki zapisano w: {csv_name}")
        else:
            print("[INFO] Brak danych do analizy w arkuszach.")
            
    except Exception as e:
        print(f"❌ KRYTYCZNY BŁĄD ANALIZY: {e}")

Dostępne okresy: ['Q1 2020', 'Q2 2020', 'Q3 2020', 'Q4 2020']


Początek przedziału dla metryk (np. 'Q1 2020') [Domyślnie Q1 2020]:  
Koniec przedziału dla metryk (np. 'Q4 2020') [Domyślnie Q4 2020]:  



   WYNIKI DLA Q1 2020 - Q4 2020
             Total Return Sharpe Ratio Sortino Ratio Calmar Ratio Max Drawdown
Strategy                                                                      
MEAN_VAR           -4.37%       0.1899        0.2424      -0.0983      -44.41%
BUY_AND_HOLD       -8.87%      -0.0581       -0.0740      -0.1977      -44.86%
MIN_VAR           -13.57%      -0.2938       -0.3540      -0.3364      -40.33%

[SUKCES] Metryki zapisano w: Metryki_Q1_2020_Q4_2020.csv


### Preselekcja zmiennych dla modeli DRL 
Klasteryzacja hierarchiczna z Mutual Information + wartości SHAP z XGBoost

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression
from dateutil.relativedelta import relativedelta
import scipy.cluster.hierarchy as hc
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform
from collections import Counter

CUMULATIVE_THRESHOLD = 0.8  # <--- Zmienne muszą odpowiadać za 80% mocy modelu
SHOW_PLOT_TITLES = False     # <--- Zmień na False, jeśli nie chcesz tytułów na wykresach

print("=========================================================================")
print("=== PRESELEKCJA WSKAŹNIKÓW - WERSJA IN-SAMPLE (CAŁY ZBIÓR TRENINGOWY) ===")
print("=========================================================================")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Zakładam, że FI_DIR i df_global są zdefiniowane wcześniej w środowisku
FI_DIR = FI_DIR if 'FI_DIR' in locals() else "."

# --- KONFIGURACJA BAZOWA ---
base_start_date = pd.Timestamp("2014-01-01")
base_end_date = pd.Timestamp("2019-06-30")

horizon_weights = {5: 1/3, 21: 1/3, 63: 1/3}
horizons = list(horizon_weights.keys())

indicators_to_check = [
    # --- Analiza Techniczna: Odległości od średnich (Trend i Stacjonarność) ---
    'dist_close_5_sma', 'dist_close_10_sma', 'dist_close_20_sma',
    'dist_close_30_sma', 'dist_close_60_sma',
    'dist_close_120_sma', 'dist_close_200_sma',

    # === EMA ===
    'dist_close_3_ema', 'dist_close_5_ema', 'dist_close_8_ema', 
    'dist_close_10_ema', 'dist_close_12_ema', 'dist_close_15_ema', 
    'dist_close_30_ema', 'dist_close_35_ema', 'dist_close_40_ema', 
    'dist_close_45_ema', 'dist_close_50_ema', 'dist_close_60_ema',
    
    # --- Trend (MACD, PPO, TRIX) ---
    'ppo', 'ppos', 'ppoh',         
    'trix_9_sma', 'trix_15_sma',
    'trix_9_ema', 'trix_15_ema',

    # --- Kierunek Trendu i Siła (DMI, ADX) ---
    'pdi_14', 'ndi_14', 
    'dx_14_ema', 'dx_30_ema', 'dx_60_ema',

    # --- Momentum i Oscylatory ---
    'rsi_7', 'rsi_14', 'rsi_30', 'rsi_60', 
    'stochrsi_14', 'stochrsi_30', 
    'cci_14', 'cci_30', 'cci_60', 
    'kdjk', 'kdjd', 'kdjj', 
    'wr_7', 'wr_14', 'wr_30', 'wr_60',
    
    'close_5_roc', 'close_10_roc', 'close_20_roc', 'close_60_roc', 'close_120_roc', 'close_250_roc',
    'bop', 
    'cmo_14', 'cmo_30', 'cmo_60',
    'qqe_14,5', 'qqe_30,10', 'qqe_60,20',
    'coppock_10,11,14',
    'kst',

    # === Wskaźniki Trendu, Czystości i Prawdopodobieństwa ===
    'close_10_ker', 'close_30_ker', 'close_60_ker',
    'aroon_14', 'aroon_30', 'aroon_60',
    'close_12_psl', 'close_30_psl', 'close_60_psl',
    'ftr_14', 'ftr_30', 'ftr_60',
    'pgo_14', 'pgo_30', 'pgo_60',

    # --- Zmienność ---
    'boll_pct_b', 'boll_width', 
    'atr_7_pct', 'atr_14_pct', 'atr_30_pct', 'atr_60_pct',

    # --- Wolumen i Pieniądz ---
    'volume_rel', 'obv_zscore',
    'mfi_7', 'mfi_14', 'mfi_30', 
    'vwma_14', 'vwma_30', 'vwma_60',
    'pvo', 'pvos', 'pvoh',
    'vr_14', 'vr_30', 'vr_60',

    # --- Makroekonomia i Ryzyko Systemowe ---
    'wig20_volatility', 'usdpln_ret', 'bond_10y_diff', 'sp500_ret',

    # --- Fundamenty (Z-Score) ---
    'pb_ratio_zscore', 'pe_ratio_zscore', 'ps_ratio_zscore', 
    'p_fcf_ratio_zscore', 'ev_ebitda_zscore', 'ev_ebit_zscore',
    'roe_zscore', 'net_margin_zscore', 'debt_ratio_zscore', 'net_to_ocf_zscore'
]

indicators_dict = {}
indicators_scores_dict = {}

# ==============================================================================
# GŁÓWNA PĘTLA DLA 4 OKIEN CZASOWYCH
# ==============================================================================
for step in range(4):
    current_start = base_start_date + relativedelta(months=3 * step)
    current_end = base_end_date + relativedelta(months=3 * step)
    okres_name = f"okres_{step}"
    
    print(f"\n" + "="*80)
    print(f"=== ROZPOCZĘCIE ANALIZY: {okres_name} ({current_start.date()} do {current_end.date()}) ===")
    print("="*80)

    # 1. Filtrowanie danych dla bieżącego okna
    analysis_df = df_global[(df_global['date'] >= current_start) & (df_global['date'] <= current_end)].copy()
    X_cols = [col for col in indicators_to_check if col in analysis_df.columns]

    # 2. Inżynieria Targetu
    analysis_df['log_ret'] = np.log(analysis_df['close'] / analysis_df.groupby('tic')['close'].shift(1))
    
    for h in horizons:
        target_col = f'target_{h}d'
        analysis_df[target_col] = analysis_df.groupby('tic')['log_ret'].transform(
            lambda x: x.rolling(window=h, min_periods=h).sum().shift(-h)
        )
        
    ml_df = analysis_df.dropna(subset=X_cols + [f'target_{h}d' for h in horizons]).copy()
    ml_df = ml_df.sort_values('date').reset_index(drop=True)
    X_ml = ml_df[X_cols]

    # =========================================================================
    # KROK 1: KLASTERYZACJA HIERARCHICZNA (Within-Group Spearman) + MI
    # =========================================================================
    df_ranked = ml_df.groupby('tic')[X_cols].rank()
    corr_df = df_ranked.corr(method='pearson').fillna(0)
    corr_matrix = corr_df.values
    
    dist_matrix = 1 - np.abs(corr_matrix)
    np.fill_diagonal(dist_matrix, 0)
    condensed_dist = squareform(dist_matrix)
    dist_linkage = hc.complete(condensed_dist)
    
    # ---> WYKRES 1A: DENDROGRAM KLASTERYZACJI <---
    plt.figure(figsize=(20, 10))
    dendro = hc.dendrogram(dist_linkage, labels=X_cols, leaf_rotation=90, leaf_font_size=10, color_threshold=0.3)
    if SHOW_PLOT_TITLES:
        plt.title(f"Dendrogram Klasteryzacji Hierarchicznej (Within-Group Spearman) [IN-SAMPLE] - {okres_name}", fontsize=14)
    plt.ylabel("Odległość (1 - |Spearman|)", fontsize=15)
    plt.xticks(fontsize=15)
    plt.axhline(y=0.3, color='r', linestyle='--', label='Próg odcięcia (0.3)')
    plt.legend(fontsize=15)
    plt.tight_layout()
    plt.savefig(os.path.join(FI_DIR, f"{okres_name}_01a_dendrogram_insample.png"), dpi=300)
    if step == 0: plt.show()
    else: plt.close()

    # ---> WYKRES 1B: MACIERZ KORELACJI (POSORTOWANA WG KLASTRÓW) <---
    cluster_leaves = dendro['leaves']
    sorted_X_cols = [X_cols[i] for i in cluster_leaves]
    corr_df_sorted = corr_df.loc[sorted_X_cols, sorted_X_cols]
    
    plt.figure(figsize=(14, 12)) 
    sns.heatmap(corr_df_sorted, cmap='coolwarm', center=0, vmin=-1, vmax=1, 
                annot=False, square=True, linewidths=.5, cbar_kws={"shrink": .7})
    if SHOW_PLOT_TITLES:
        plt.title(f"Posortowana Macierz Korelacji Within-Group (Spearman) [IN-SAMPLE] - {okres_name}", fontsize=15)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(FI_DIR, f"{okres_name}_01b_correlation_matrix_insample.png"), dpi=300)
    if step == 0: plt.show()
    else: plt.close()
    
    # 1.4 Cięcie drzewa
    cluster_ids = hc.fcluster(dist_linkage, 0.3, criterion='distance')
    cluster_dict = {}
    for idx, cluster_id in enumerate(cluster_ids):
        cluster_dict.setdefault(cluster_id, []).append(X_cols[idx])

    # 1.5 Mutual Information
    mi_avg_scores = pd.Series(0.0, index=X_cols)
    X_mi_sample = X_ml.sample(n=min(len(X_ml), 10000), random_state=42) 
    
    for h, w in horizon_weights.items():
        y_h_sample = ml_df.loc[X_mi_sample.index, f'target_{h}d']
        mi = mutual_info_regression(X_mi_sample, y_h_sample, random_state=42)
        mi_avg_scores += pd.Series(mi, index=X_cols) * w

    # ---> WYKRES 2: WYNIKI MUTUAL INFORMATION <---
    plt.figure(figsize=(14, 6))
    mi_avg_scores.sort_values(ascending=False).plot(kind='bar', color='cadetblue', edgecolor='black')
    if SHOW_PLOT_TITLES:
        plt.title(f"Ważone Mutual Information (Przed Redukcją) [IN-SAMPLE] - {okres_name}", fontsize=14)
    plt.ylabel("MI Score")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(FI_DIR, f"{okres_name}_02_mutual_info_insample.png"), dpi=300)
    if step == 0: plt.show()
    else: plt.close()

    # 1.6 Wybór liderów klastrów i przygotowanie danych do wykresu
    cluster_records = []
    X_cols_clean = []
    for cluster_id, features in cluster_dict.items():
        features_sorted = mi_avg_scores[features].sort_values(ascending=False)
        best_feature = features_sorted.index[0]
        X_cols_clean.append(best_feature)
        
        for feat, score in features_sorted.items():
            cluster_records.append({
                'Cluster_ID': cluster_id,
                'Cluster_Name': f"Klaster {cluster_id}",
                'Feature': feat,
                'MI_Score': score,
                'Is_Leader': feat == best_feature
            })

    df_mi_clusters = pd.DataFrame(cluster_records)
    df_mi_clusters = df_mi_clusters.sort_values(by=['Cluster_ID', 'MI_Score'], ascending=[True, False])
    df_mi_clusters = df_mi_clusters.groupby('Cluster_ID').head(3).reset_index(drop=True)

    # ---> WYKRES 2B: MUTUAL INFORMATION W OBRĘBIE KLASTRÓW <---
    fig_height = max(6, len(df_mi_clusters) * 0.18)
    plt.figure(figsize=(10, fig_height))
    
    ax = sns.barplot(
        data=df_mi_clusters, 
        x='MI_Score', 
        y='Feature', 
        hue='Cluster_Name', 
        dodge=False, 
        palette='tab20'
    )
    
    current_cluster = df_mi_clusters.iloc[0]['Cluster_ID']
    for i, row in enumerate(df_mi_clusters.itertuples()):
        if row.Is_Leader:
            ax.text(row.MI_Score + 0.001, i, ' ★ LIDER', color='red', va='center', fontweight='bold', fontsize=10)
        if row.Cluster_ID != current_cluster:
            plt.axhline(i - 0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)
            current_cluster = row.Cluster_ID
            
    if SHOW_PLOT_TITLES:
        plt.title(f"Wybór Liderów Klastrów (Ważone MI) [IN-SAMPLE] - {okres_name}", fontsize=16, fontweight='bold')
    plt.xlabel("Ważone Mutual Information Score", fontsize=12)
    plt.ylabel("Wskaźniki", fontsize=12)
    plt.legend(title='Przynależność', bbox_to_anchor=(1.02, 1), loc='upper left', ncol=1, fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FI_DIR, f"{okres_name}_02b_mi_by_cluster_insample.png"), dpi=300, bbox_inches='tight')
    if step == 0: plt.show()
    else: plt.close()

    X_ml_clean = ml_df[X_cols_clean]
    print(f"Zredukowano wymiarowość z {len(X_cols)} do {len(X_cols_clean)} za pomocą klasteryzacji.")
    
    # =========================================================================
    # KROK 2: IN-SAMPLE SHAP VALUES (Budowa na całym zbiorze)
    # =========================================================================
    shap_results_weighted = pd.Series(0.0, index=X_cols_clean)
    shap_results_individual = {} 
    
    print("\n[INFO] Budowa modelu XGBoost na CAŁYM zbiorze i ekstrakcja SHAP...")

    for h, w in horizon_weights.items():
        y_h = ml_df[f'target_{h}d']
        
        X_train = X_ml_clean
        y_train = y_h
            
        xgb = XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
        
        xgb.fit(X_train, y_train, verbose=False)
        
        explainer = shap.TreeExplainer(xgb)
        shap_values = explainer.shap_values(X_train)
        
        mean_shap_h = pd.Series(np.abs(shap_values).mean(axis=0), index=X_cols_clean)

        shap_results_individual[h] = mean_shap_h
        shap_results_weighted += mean_shap_h * w
    
    # =========================================================================
    # KROK 3: OSTATECZNY RANKING, SKUMULOWANA WAŻNOŚĆ I WYKRESY SHAP
    # =========================================================================
    master_ranking = pd.DataFrame({'SHAP_Weighted': shap_results_weighted})
    master_ranking = master_ranking.sort_values(by='SHAP_Weighted', ascending=False)

    master_ranking['Final_Score_Normalized'] = master_ranking['SHAP_Weighted'] / master_ranking['SHAP_Weighted'].max()
    total_shap_importance = master_ranking['SHAP_Weighted'].sum()
    master_ranking['Cumulative_Importance'] = master_ranking['SHAP_Weighted'].cumsum() / total_shap_importance

    cutoff_idx = (master_ranking['Cumulative_Importance'] >= CUMULATIVE_THRESHOLD).idxmax()
    cutoff_pos = master_ranking.index.get_loc(cutoff_idx)
    
    top_dynamic_features = master_ranking.iloc[:cutoff_pos + 1].index.tolist()
    
    print(f"[{okres_name}] Wybrano {len(top_dynamic_features)} zmiennych, które odpowiadają za {CUMULATIVE_THRESHOLD*100}% mocy modelu.")

    # -------------------------------------------------------------------------
    # WYKRES 3: FINAŁOWY RANKING SHAP Z LINIĄ ODCIĘCIA (Wszystkie zmienne)
    # -------------------------------------------------------------------------
    plot_df = master_ranking.copy() 
    plot_df['Status'] = ['Wybrane (Top dynamiczny)' if x in top_dynamic_features else 'Odrzucone (Szum)' for x in plot_df.index]

    plt.figure(figsize=(12, 14)) 
    sns.barplot(
        x='Final_Score_Normalized', 
        y=plot_df.index, 
        data=plot_df, 
        hue='Status', 
        dodge=False, 
        palette={'Wybrane (Top dynamiczny)': '#21918c', 'Odrzucone (Szum)': 'lightgray'}
    )
    plt.axhline(y=len(top_dynamic_features) - 0.5, color='red', linestyle='--', linewidth=2, label=f'Linia odcięcia ({CUMULATIVE_THRESHOLD*100}% sumy SHAP)')
    if SHOW_PLOT_TITLES:
        plt.title(f"In-Sample Ranking SHAP (Wszystkie, z linią odcięcia) - {okres_name}", fontsize=14)
    plt.xlabel("Znormalizowana Waga", fontsize=14)
    plt.ylabel("Wskaźnik", fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(FI_DIR, f"{okres_name}_03_final_shap_ranking_insample_cutoff.png"), dpi=300)
    if step == 0: plt.show()
    else: plt.close()

    # -------------------------------------------------------------------------
    # WYKRES 3b: FINAŁOWY RANKING SHAP - WSZYSTKIE, JEDEN #E377C2, BEZ ODCIĘCIA
    # -------------------------------------------------------------------------
    single_color_plot_df = master_ranking.copy()

    plt.figure(figsize=(12, 14))
    sns.barplot(
        x='Final_Score_Normalized',   
        y=single_color_plot_df.index, 
        data=single_color_plot_df,
        color='#007481'               
    )
    if SHOW_PLOT_TITLES:
        plt.title(f"In-Sample Ranking SHAP (Wszystkie wskaźniki) - {okres_name}", fontsize=14)
    
    plt.xlabel("Znormalizowana Waga", fontsize=20)
    plt.ylabel("Wskaźnik", fontsize=20)
    
    # --- DODANE LINIE: Zmiana wielkości wartości na osiach ---
    plt.xticks(fontsize=15) # wartości liczbowe na dole
    plt.yticks(fontsize=15) # nazwy wskaźników po lewej stronie
    # ---------------------------------------------------------

    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    out_path_single_color_all = os.path.join(FI_DIR, f"{okres_name}_03b_final_shap_ranking_insample_all.png")
    plt.savefig(out_path_single_color_all, dpi=300)
    if step == 0: plt.show()
    else: plt.close()

    # -------------------------------------------------------------------------
    # WYKRES 3c: Dodatkowe wykresy SHAP dla poszczególnych horyzontów
    # -------------------------------------------------------------------------
    for h, shap_series in shap_results_individual.items():
        df_h = pd.DataFrame({'SHAP': shap_series})
        df_h = df_h.sort_values(by='SHAP', ascending=False)
        
        max_val = df_h['SHAP'].max()
        if max_val > 0:
            df_h['SHAP_Normalized'] = df_h['SHAP'] / max_val
        else:
            df_h['SHAP_Normalized'] = 0.0
            
        plt.figure(figsize=(12, 14))
        sns.barplot(
            x='SHAP_Normalized',   
            y=df_h.index, 
            data=df_h,
            color='#d95f02' 
        )

        if SHOW_PLOT_TITLES:
            plt.title(f"In-Sample Ranking SHAP (Target: {h} dni) - {okres_name}", fontsize=14)
        plt.xlabel("Znormalizowana Waga", fontsize=14)
        plt.ylabel("Wskaźnik", fontsize=14)
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        
        out_path_horizon = os.path.join(FI_DIR, f"{okres_name}_03c_shap_horizon_insample_{h}d.png")
        plt.savefig(out_path_horizon, dpi=300)
        
        if step == 0: plt.show()
        else: plt.close()

    # Zapisanie danych logicznych pozostaje na końcu
    indicators_dict[okres_name] = top_dynamic_features
    indicators_scores_dict[okres_name] = master_ranking.loc[top_dynamic_features, 'Final_Score_Normalized'].to_dict()   

# ==============================================================================
# PODSUMOWANIE WYNIKÓW
# ==============================================================================
print("\n" + "="*80)
print("=== PODSUMOWANIE PRESELEKCJI WSKAŹNIKÓW DLA 4 OKIEN (IN-SAMPLE) ===")
print("="*80)
for k, v in indicators_dict.items():
    print(f"{k}: {v}")

print("\n" + "="*80)
print("=== ANALIZA STABILNOŚCI I RÓŻNIC WE WSKAŹNIKACH ===")
print("="*80)

all_features = []
for features in indicators_dict.values():
    all_features.extend(features)

feature_counts = Counter(all_features)
df_counts = pd.DataFrame.from_dict(feature_counts, orient='index', columns=['Liczba_wystapien'])
df_counts['Procent_obecnosci'] = (df_counts['Liczba_wystapien'] / len(indicators_dict)) * 100
df_counts = df_counts.sort_values(by='Liczba_wystapien', ascending=False)

print("\n[1] Najbardziej uniwersalne wskaźniki w oknach In-Sample:")
print(df_counts.head(20).to_string())

unique_features = list(df_counts.index)
heatmap_data = pd.DataFrame(0.0, index=unique_features, columns=indicators_dict.keys())

for period, scores in indicators_scores_dict.items():
    for feat, score in scores.items():
        heatmap_data.loc[feat, period] = score

plt.figure(figsize=(16, 14)) 
mask = (heatmap_data == 0)

ax = sns.heatmap(
    heatmap_data, 
    cmap="viridis",        
    mask=mask,            
    cbar=True,            
    vmin=0, vmax=1,      
    linewidths=0.5, 
    linecolor='white',   
    square=False,        
    cbar_kws={"shrink": .7, "label": "Znormalizowana Siła SHAP"} 
)
ax.set_facecolor('#e8e8e8') 

if SHOW_PLOT_TITLES:
    plt.title(f"Ewolucja i siła wyselekcjonowanych wskaźników In-Sample (> {CUMULATIVE_THRESHOLD*100}% SHAP)", fontsize=14, pad=20)
plt.xlabel("Okres", fontsize=20)
plt.ylabel("Wskaźnik", fontsize=20)
plt.xticks(rotation=45, ha='right', fontsize=15)
plt.yticks(fontsize=15)
plt.tight_layout()

out_path_heatmap = os.path.join(FI_DIR, '06_indicator_evolution_heatmap_insample.png')
plt.savefig(out_path_heatmap, dpi=300, facecolor='white')
if step == 0: plt.show()
else: plt.close()

print(f"\n[INFO] Wszystkie wykresy IN-SAMPLE zostały zapisane w folderze: {os.path.abspath(FI_DIR)}")

# =====================================================================
# ANALIZA 3: ROTACJA (Jak bardzo rynek zmieniał się z kwartału na kwartał?)
# =====================================================================
print("\n[3] Rotacja wskaźników między kolejnymi oknami In-Sample:")
periods = list(indicators_dict.keys())

for i in range(1, len(periods)):
    prev_period = periods[i-1]
    curr_period = periods[i]
    
    prev_set = set(indicators_dict[prev_period])
    curr_set = set(indicators_dict[curr_period])
    
    common = prev_set.intersection(curr_set)
    dropped = prev_set - curr_set
    added = curr_set - prev_set
    
    print(f"\nPrzejście z {prev_period} ({len(prev_set)} zm.) -> {curr_period} ({len(curr_set)} zm.):")
    print(f"  -> Utrzymano: {len(common)} wskaźników.")
    if dropped:
        print(f"  -> Wypadły: {', '.join(dropped)}")
    if added:
        print(f"  -> Nowe (weszły): {', '.join(added)}")

=== PRESELEKCJA WSKAŹNIKÓW - WERSJA IN-SAMPLE (CAŁY ZBIÓR TRENINGOWY) ===

=== ROZPOCZĘCIE ANALIZY: okres_0 (2014-01-01 do 2019-06-30) ===
Zredukowano wymiarowość z 110 do 40 za pomocą klasteryzacji.

[INFO] Budowa modelu XGBoost na CAŁYM zbiorze i ekstrakcja SHAP...
[okres_0] Wybrano 17 zmiennych, które odpowiadają za 80.0% mocy modelu.

=== ROZPOCZĘCIE ANALIZY: okres_1 (2014-04-01 do 2019-09-30) ===
Zredukowano wymiarowość z 110 do 40 za pomocą klasteryzacji.

[INFO] Budowa modelu XGBoost na CAŁYM zbiorze i ekstrakcja SHAP...
[okres_1] Wybrano 16 zmiennych, które odpowiadają za 80.0% mocy modelu.

=== ROZPOCZĘCIE ANALIZY: okres_2 (2014-07-01 do 2019-12-30) ===
Zredukowano wymiarowość z 110 do 39 za pomocą klasteryzacji.

[INFO] Budowa modelu XGBoost na CAŁYM zbiorze i ekstrakcja SHAP...
[okres_2] Wybrano 17 zmiennych, które odpowiadają za 80.0% mocy modelu.

=== ROZPOCZĘCIE ANALIZY: okres_3 (2014-10-01 do 2020-03-30) ===
Zredukowano wymiarowość z 110 do 40 za pomocą klasteryzacji.

[

### Klasa środowiska symulacyjnego

In [5]:
class StockTradingEnvCustom(StockTradingEnv):
    """
    Wrapper na domyślne środowisko StockTradingEnv z biblioteki FinRL.
    Zoptymalizowany pod kątem pamięci RAM i stabilności w Optunie.
    """
    def __init__(self, reward_type="log", risk_penalty=0.1, **kwargs):
        # =================================================================
        # FIX: INICJALIZACJA NUMPY PRZED SUPER()
        # Wyciągamy 'df' oraz 'tech_indicator_list' z argumentów wejściowych,
        # aby zbudować macierze ZANIM oryginalny FinRL wywoła _initiate_state()
        # =================================================================
        df = kwargs.get('df')
        tech_indicators = kwargs.get('tech_indicator_list', [])

        # 1. Ustalanie kolumny ceny wykonania
        try:
            exec_price_col_idx = df.columns.get_loc('next_day_open')
        except:
            exec_price_col_idx = df.columns.get_loc('close')
            
        self.exec_price_col_idx = exec_price_col_idx
        
        # A. Ceny Egzekucji
        self.np_exec_prices = df.iloc[:, self.exec_price_col_idx].values.astype(np.float32)
        
        # B. Ceny Zamknięcia 
        close_col_idx = df.columns.get_loc('close')
        self.np_close_prices = df.iloc[:, close_col_idx].values.astype(np.float32)

        # C. Wskaźniki Techniczne
        if tech_indicators:
            valid_indicators = [ind for ind in tech_indicators if ind in df.columns]
            if valid_indicators:
                self.np_indicators = df[valid_indicators].values.astype(np.float32)
            else:
                self.np_indicators = None
        else:
            self.np_indicators = None

        # =================================================================
        # Teraz bezpiecznie wywołujemy konstruktor bazowy FinRL. 
        # Gdy wywoła on _initiate_state(), macierze NumPy będą już gotowe!
        # =================================================================
        super().__init__(**kwargs)
        
        # 2. XTB Wymogi
        self.min_trade_amount = 10.0 
        
        # 3. Konfiguracja Eksperymentalna
        self.reward_type = reward_type  
        self.risk_penalty = risk_penalty
        self.return_memory = []
        self.peak_account_value = self.initial_amount # <--- DODANO (do Drawdown)

    def _get_execution_price(self, index):
        data_idx = self.day * self.stock_dim + index
        return self.np_exec_prices[data_idx]

    def _calculate_cost(self, amount):
        if amount == 0: return 0.0
        return amount * self.buy_cost_pct[0]

    # =================================================================
    # NADPISANIE FUNKCJI _initiate_state (Optymalizacja wektora)
    # Zastępuje powolne operacje na listach szybkimi alokacjami w NumPy
    # =================================================================
    def _initiate_state(self):
        """Przebudowana funkcja natywna omijająca Pandas przy startowym stanie"""
        if self.initial:
            self.peak_account_value = self.initial_amount # <--- DODANO
            # Epizod startowy: Stan gotówki i zera akcji
            state = [self.initial_amount] + \
                    self.np_close_prices[self.day * self.stock_dim : (self.day + 1) * self.stock_dim].tolist() + \
                    [0] * self.stock_dim
        else:
            # Kolejny dzień: Używamy zapamiętanej ilości akcji
            state = [self.state[0]] + \
                    self.np_close_prices[self.day * self.stock_dim : (self.day + 1) * self.stock_dim].tolist() + \
                    list(self.state[(self.stock_dim + 1) : (self.stock_dim * 2 + 1)])

        # Pobranie bloku wskaźników dla wszystkich spółek dla danego dnia:
        if self.np_indicators is not None:
            # Wycina blok macierzy odpowiadający danemu dniowi: kształt [stock_dim, num_indicators]
            # Następnie 'flatten' spłaszcza to do wektora 1D i dodaje do stanu.
            inds_flat = self.np_indicators[self.day * self.stock_dim : (self.day + 1) * self.stock_dim].flatten()
            state = state + inds_flat.tolist()
            
        return state

    # =================================================================
    # NADPISANIE FUNKCJI _update_state (Optymalizacja Kroku)
    # Wywoływana kilkaset tysięcy razy w jednym treningu
    # =================================================================
    def _update_state(self):
        """Przebudowana funkcja omijająca Pandas przy aktualizacji stanu po kroku"""
        state = [self.state[0]] + \
                self.np_close_prices[self.day * self.stock_dim : (self.day + 1) * self.stock_dim].tolist() + \
                list(self.state[(self.stock_dim + 1) : (self.stock_dim * 2 + 1)])

        if self.np_indicators is not None:
            inds_flat = self.np_indicators[self.day * self.stock_dim : (self.day + 1) * self.stock_dim].flatten()
            state = state + inds_flat.tolist()
            
        return state
 
    def _sell_stock(self, index, action):
        is_panic_sell = False 
        
        if self.turbulence_threshold is not None:
            if self.turbulence >= self.turbulence_threshold:
                is_panic_sell = True
                if self.state[index + self.stock_dim + 1] > 0: 
                    action = self.state[index + self.stock_dim + 1]
                else: return 0
        
        price = self._get_execution_price(index)
        
        if self.state[index + self.stock_dim + 1] > 0:
            # Upewniamy się, że nie sprzedajemy więcej niż mamy
            action = min(abs(float(action)), self.state[index + self.stock_dim + 1])
            transaction_val = price * action
            
            # =================================================================
            # FIX: IDENTYFIKACJA CAŁKOWITEJ WYPRZEDAŻY (LIQUIDATE ALL)
            # Jeśli różnica między akcją a posiadanymi akcjami jest bliska zeru
            # =================================================================
            is_liquidate_all = abs(action - self.state[index + self.stock_dim + 1]) < 1e-5
            
            # Agent może sprzedać poniżej limitu TYLKO wtedy, gdy to panic_sell lub pozbywa się wszystkiego
            if not is_panic_sell and not is_liquidate_all and transaction_val < self.min_trade_amount:
                action = 0
                transaction_val = 0
            
            if action > 0:
                commission = self._calculate_cost(transaction_val)
                self.state[0] += (transaction_val - commission)
                self.state[index + self.stock_dim + 1] -= action
                self.trades += 1
                self.cost += commission 
        else: 
            action = 0
            
        return action

    def _buy_stock(self, index, action):
        if self.turbulence_threshold is not None:
            if self.turbulence >= self.turbulence_threshold: return 0

        price = self._get_execution_price(index)
        available_cash = self.state[0]
        
        if price > 0:
            max_afford_shares = available_cash / (price * (1 + self.buy_cost_pct[0]))
        else:
            max_afford_shares = 0

        current_hmax = self.hmax[index] if isinstance(self.hmax, list) else self.hmax
        
        action = float(action)
        action = min(action, max_afford_shares)
        action = min(action, float(current_hmax))
        
        transaction_val = price * action
        
        if transaction_val < self.min_trade_amount:
            action = 0
            transaction_val = 0
        
        if action > 0:
            comm = self._calculate_cost(transaction_val)
            self.state[0] -= (transaction_val + comm)
            self.state[index + self.stock_dim + 1] += action
            self.trades += 1
            self.cost += comm 
        else: 
            action = 0
        return action

    def _get_custom_reward(self):
        if self.day == 0: return 0
        current_value = self.asset_memory[-1]
        prev_value = self.asset_memory[-2] if len(self.asset_memory) > 1 else self.initial_amount
        
        if prev_value <= 1e-5: return 0.0
        safe_current_value = max(current_value, 1e-5)

        if self.reward_type == "simple":
            return current_value - prev_value
            
        elif self.reward_type == "log":
            return np.log(safe_current_value / prev_value)
            
        elif self.reward_type == "sharpe":
            step_return = (safe_current_value / prev_value) - 1
            self.return_memory.append(step_return)
            
            # WĄSKIE GARDŁO NAPRAWIONE: Skracamy listę, by nie puchła w pamięci
            if len(self.return_memory) > 30:
                self.return_memory.pop(0) # Utrzymujemy stałą długość O(1)
                
            volatility = np.std(self.return_memory) if len(self.return_memory) == 30 else 0.0
            return step_return - (self.risk_penalty * volatility)
            
        elif self.reward_type == "drawdown":
            step_return = (safe_current_value / prev_value) - 1
            
            # WĄSKIE GARDŁO NAPRAWIONE: Bieżąca aktualizacja "High Watermark" (Złożoność O(1))
            if current_value > self.peak_account_value:
                self.peak_account_value = current_value
                
            drawdown = (self.peak_account_value - current_value) / self.peak_account_value
            return step_return - (self.risk_penalty * drawdown)
            
        return 0

    def step(self, actions):
        actions = np.array(actions, dtype=float)

        if self.turbulence_threshold is not None and self.turbulence >= self.turbulence_threshold:
            # W trybie awaryjnym i tak wszystko sprzedajemy, więc pomijamy skalowanie
            if isinstance(self.hmax, (list, np.ndarray)):
                actions = -np.array(self.hmax, dtype=float)
            else:
                actions = np.array([-self.hmax] * self.stock_dim, dtype=float)
                
            temp = self.turbulence_threshold
            self.turbulence_threshold = None
            step_result = super().step(actions)
            self.turbulence_threshold = temp
        else:
            # =================================================================
            # PROPORCJONALNE SKALOWANIE BUDŻETU (SYNC Z LOGIKĄ TESTOWĄ)
            # =================================================================
            current_cash = self.state[0]
            expected_cash_from_sells = 0.0
            needed_cash_for_buys = 0.0

            # 1. Pobieramy ceny i posiadane akcje dla obecnego kroku
            current_prices = self.np_exec_prices[self.day * self.stock_dim : (self.day + 1) * self.stock_dim]
            current_holdings = np.array(self.state[(self.stock_dim + 1) : (self.stock_dim * 2 + 1)])
            
            # Konwersja hmax do ujednoliconego wektora NumPy
            hmax_arr = np.array(self.hmax, dtype=float) if isinstance(self.hmax, (list, np.ndarray)) else np.array([self.hmax] * self.stock_dim, dtype=float)
            
            # Wektoryzacja zamierzonego wolumenu (tak jak robi to FinRL w tle)
            target_volumes = actions * hmax_arr

            # 2. Szybka symulacja finansowa wyprzedzająca ruch środowiska
            for i in range(self.stock_dim):
                price = current_prices[i]
                if target_volumes[i] < 0:
                    # Sprzedaż: możemy sprzedać max to, co posiadamy
                    sell_vol = min(abs(target_volumes[i]), current_holdings[i])
                    val = sell_vol * price
                    if val >= self.min_trade_amount:
                        expected_cash_from_sells += val - self._calculate_cost(val)
                elif target_volumes[i] > 0:
                    # Zakup
                    val = target_volumes[i] * price
                    if val >= self.min_trade_amount:
                        needed_cash_for_buys += val + self._calculate_cost(val)

            total_available = current_cash + expected_cash_from_sells

            # 3. Właściwe skalowanie sygnałów kupna w dół
            if needed_cash_for_buys > total_available and needed_cash_for_buys > 0:
                scale_ratio = total_available / needed_cash_for_buys
                # Modyfikujemy główny wektor akcji (tylko wartości dodatnie)
                actions = np.where(actions > 0, actions * scale_ratio, actions)
            # =================================================================

            step_result = super().step(actions)

        # =================================================================
        # FIX ZAPOBIEGAJĄCY SORTINO = 0.0 (GWARANCJA ZWROTU TOTAL_ASSETS)
        # =================================================================
        if len(step_result) == 5:
            obs, _, terminated, truncated, info = step_result
            terminal = terminated or truncated
        else:
            obs, _, terminal, info = step_result
            truncated = False

        # Zabezpieczenie na wypadek, gdyby bazowe środowisko zwróciło np. info = None
        if info is None or not isinstance(info, dict):
            info = {}

        current_assets = self.asset_memory[-1]
        info['total_assets'] = float(current_assets) 

        if self.reward_type != "simple":
            self.reward = self._get_custom_reward()
            self.reward = self.reward * self.reward_scaling
        
        # Zwracamy spójnie w zależności od wersji bazowego GYM
        if len(step_result) == 5:
            return obs, self.reward, terminated, truncated, info
        else:
            return obs, self.reward, terminal, info

### Tuning hiperparametrów z wykorzystaniem estymatora TPE (Optuna)

In [6]:
SB3_MODELS = {
    "ppo": PPO,
    "a2c": A2C,
    "ddpg": DDPG,
    "td3": TD3,
    "sac": SAC
}

# --- 0. WYBÓR OKRESU I KONFIGURACJA ---
try:
    max_period = len(GLOBAL_DATES) - 1
    PERIOD_ID = int(input(f"Podaj numer okresu do tuningu (Period ID) [0-{max_period}]: ").strip() or 0)
    if PERIOD_ID < 0 or PERIOD_ID > max_period:
        print(f"[WARN] Podano ID poza zakresem. Ustawiam domyślnie na Period 0.")
        PERIOD_ID = 0
except ValueError:
    PERIOD_ID = 0
    print("[WARN] Nieprawidłowy input. Ustawiam domyślnie na Period 0.")

# --- DODANE: Dynamiczne mapowanie wskaźników ---
period_key = f"okres_{PERIOD_ID}"
# Pobieramy wskaźniki dla danego okresu. Jeśli nie ma klucza w słowniku, pobieramy dla "okres_0", a jak nie ma "okres_0", dajemy pustą listę.
indicators = indicators_dict.get(period_key, indicators_dict.get("okres_0", [])) 

print(f"\n=== START TUNINGU OPTUNA (RESUMABLE & CONTINUOUS) DLA PERIOD_{PERIOD_ID} ===")

def set_seeds(seed=1):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# --- 1. KONFIGURACJA TUNINGU ---
# Dodajemy interaktywny wybór modelu, aby móc odpalać skrypt w wielu terminalach
wybrany_model = input("Podaj model do tuningu (ppo, a2c, ddpg, td3, sac) lub wciśnij Enter dla wszystkich: ").strip().lower()

if wybrany_model in SB3_MODELS:
    TUNE_MODELS = [wybrany_model]
    print(f"[INFO] Wybrano pojedynczy model: {wybrany_model.upper()}")
else:
    TUNE_MODELS = ["ppo", "a2c", "ddpg", "td3", "sac"]
    if wybrany_model != "":
        print(f"[WARN] Nie rozpoznano modelu '{wybrany_model}'.")
    print("[INFO] Odpalam pętlę dla wszystkich modeli sekwencyjnie.")

N_TRIALS = 50            
EVAL_SEEDS = [1, 2, 3] # <--- LISTA NASION DO MULTI-SEED EVALUATION
REWARD_SCALING = 100
MAX_POSITION_PCT = 1

TUNING_DIR = os.path.join(OUTPUT_MODELS_DIR if 'OUTPUT_MODELS_DIR' in locals() else os.getcwd(), f"Optuna_Tuning_Period{PERIOD_ID}")
os.makedirs(TUNING_DIR, exist_ok=True)

# --- 2. PRZYGOTOWANIE DANYCH ---
unique_tickers = sorted(df_global.tic.unique())
stock_dim = len(unique_tickers)
# Wymiar środowiska wyliczany dynamicznie na podstawie zmiennej `indicators`
state_dim = 1 + 2 * stock_dim + len(indicators) * stock_dim

print(f"[SYSTEM] Stock Dim: {stock_dim}")
print(f"[SYSTEM] Indicators count: {len(indicators)}")
print(f"[SYSTEM] State Dim: {state_dim}")

test_start = GLOBAL_DATES[PERIOD_ID]
train_start = test_start - relativedelta(years=TRAIN_YEARS)
train_end = test_start - relativedelta(months=VAL_MONTHS)
val_start = train_end

print(f"[SYSTEM] Tuning dla zakresu: Trening [{train_start.date()} - {train_end.date()}), Walidacja [{val_start.date()} - {test_start.date()})")

train = df_global[(df_global.date >= train_start) & (df_global.date < train_end)].copy()
val = df_global[(df_global.date >= val_start) & (df_global.date < test_start)].copy()

train.index = train.date.factorize()[0]
val.index = val.date.factorize()[0]

unique_tickers = sorted(FILES_CONFIG.keys()) 
target_exposure = INITIAL_CAPITAL * MAX_POSITION_PCT
hmax_list = []

for tic in unique_tickers:
    tic_data = train[train.tic == tic]
    max_price = tic_data['close'].max() if not tic_data.empty else 100.0
    if pd.isna(max_price) or max_price <= 0: max_price = 1.0
    limit = target_exposure / (max_price * (1 + COST))
    hmax_list.append(round(float(limit), 4))

env_kwargs = {
    "hmax": hmax_list, "initial_amount": INITIAL_CAPITAL, "buy_cost_pct": [COST]*stock_dim, 
    "sell_cost_pct": [COST]*stock_dim, "state_space": state_dim, "stock_dim": stock_dim, 
    "tech_indicator_list": indicators, "action_space": stock_dim, "reward_scaling": REWARD_SCALING, 
    "reward_type": "log", "risk_penalty": 0.1,  
    "num_stock_shares": [0]*stock_dim, "risk_indicator_col": "turbulence"
}

# --- 2B. KALKULACJA KROKÓW I EWALUACJI ---
episode_length = len(train.index.unique())
steps = int(100 * episode_length)  
eval_freq = steps // 10  
dynamic_starts = steps // 10

print(f"[SYSTEM] Tuned Episode Length: {episode_length} days")
print(f"[SYSTEM] Total Timesteps: {steps} | Eval Freq: {eval_freq}")

# --- 3. SPECJALNY CALLBACK Z PRUNINGIEM (OPTUNA - SORTINO) ---
class TuningEvalCallback(BaseCallback):
    def __init__(self, trial, eval_env, eval_freq, initial_capital=100000, 
                 reward_scaling=100, normalization_source=None, 
                 seed_idx=0, total_steps_per_seed=0): # <--- DODANO INDEKSACJĘ NASIONA
        super(TuningEvalCallback, self).__init__(verbose=0)
        self.trial = trial
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.initial_capital = initial_capital
        self.reward_scaling = reward_scaling
        self.normalization_source = normalization_source
        self.best_sortino = -np.inf 
        self.seed_idx = seed_idx
        self.total_steps_per_seed = total_steps_per_seed

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            if self.normalization_source is not None and isinstance(self.eval_env, VecNormalize):
                self.eval_env.obs_rms = copy.deepcopy(self.normalization_source.obs_rms)
                self.eval_env.training = False 
                self.eval_env.norm_reward = False 
                
            obs = self.eval_env.reset()
            done = False
            portfolio_values = [self.initial_capital]
            
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, rewards, dones, info = self.eval_env.step(action)
                current_total_assets = info[0].get('total_assets', portfolio_values[-1])
                portfolio_values.append(current_total_assets)
                done = dones[0]

            if isinstance(self.eval_env, VecNormalize):
                self.eval_env.training = True

            df_account = pd.DataFrame(portfolio_values, columns=['value'])
            returns = df_account['value'].pct_change().dropna()
            
            if len(returns) < 2 or returns.std() == 0:
                sortino = 0.0
            else:
                neg_ret = returns[returns < 0]
                downside_std = neg_ret.std(ddof=0) if len(neg_ret) > 1 else 1e-6
                expected_return = returns.mean() * 252
                annual_downside = downside_std * (np.sqrt(252))
                sortino = expected_return / (annual_downside + 1e-9)

            if sortino > self.best_sortino:
                self.best_sortino = sortino

            # Skalowanie kroku, aby pruner widział ciągły postęp dla wszystkich 3 nasion
            global_step = (self.seed_idx * self.total_steps_per_seed) + self.num_timesteps
            
            self.trial.report(sortino, global_step)
            if self.trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return True

'''

def sample_params(trial, model_name):
    """
    Definiuje wielowymiarową przestrzeń poszukiwań. Zwraca unikalną kombinację 
    hiperparametrów zaproponowaną w danym kroku przez optymalizator bayesowski (TPE).
    """
    # ---------------------------------------------------------
    # WSPÓLNE PARAMETRY DLA WSZYSTKICH MODELI
    # ---------------------------------------------------------
    net_arch = trial.suggest_categorical("net_arch", [64, 128, 256, 512])
    activation_fn_name = trial.suggest_categorical("activation_fn", ["tanh", "relu"])
    activation_fn = {"tanh": torch.nn.Tanh, "relu": torch.nn.ReLU}[activation_fn_name]

    # Regularyzacja L2 chroniąca wagi przed przeuczeniem
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    
    # [ZMIANA]: gamma z przedziału [0.8, 0.999] (logarytmiczny odwrócony)
    # 1 - 0.2 = 0.8 oraz 1 - 0.001 = 0.999
    gamma = 1.0 - trial.suggest_float("one_minus_gamma", 0.001, 0.2, log=True)
    
    # Rozszerzone przedziały wspólne (według specyfikacji tabelarycznej)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 3e-3, log=True)
    
    # [ZMIANA]: Losujemy batch_size potęgi od 6 do 10 (czyli od 64 do 1024)
    if model_name != "a2c":
        batch_size = 2 ** trial.suggest_int("batch_size_pow", 6, 10)

    # ---------------------------------------------------------
    # PARAMETRY SPECYFICZNE DLA ALGORYTMÓW
    # ---------------------------------------------------------
    if model_name == "ppo":
        n_steps = 2 ** trial.suggest_int("n_steps_pow", 5, 12)
        
        # Zabezpieczenie PPO: batch_size nigdy nie może przekroczyć n_steps
        ppo_batch_size = min(batch_size, n_steps)

        return {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
            "weight_decay": weight_decay,
            "gamma": gamma,
            "learning_rate": learning_rate,
            "batch_size": ppo_batch_size,
            # [ZMIANA]: ent_coef zaktualizowane na 1e-8
            "ent_coef": trial.suggest_float("ent_coef", 1e-8, 0.1, log=True),
            "clip_range": trial.suggest_categorical("clip_range", [0.1, 0.2, 0.3]),
            "n_epochs": trial.suggest_int("n_epochs", 3, 20),
            "gae_lambda": 1.0 - trial.suggest_float("one_minus_gae_lambda", 0.001, 0.2, log=True),
            # [ZMIANA]: max_grad_norm do 5.0 oraz z rozkładem logarytmicznym
            "max_grad_norm": trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True), 
            "n_steps": n_steps
        }
        
    elif model_name == "a2c":
        return {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
            "weight_decay": weight_decay,
            "gamma": gamma,
            "learning_rate": learning_rate,
            # [ZMIANA]: ent_coef zaktualizowane na 1e-8
            "ent_coef": trial.suggest_float("ent_coef", 1e-8, 0.1, log=True),
            # [ZMIANA]: max_grad_norm do 5.0 oraz z rozkładem logarytmicznym
            "max_grad_norm": trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True),
            "gae_lambda": 1.0 - trial.suggest_float("one_minus_gae_lambda", 0.001, 0.2, log=True),
            # [ZMIANA]: n_steps od 5 do 63 (od tygodnia do kwartału giełdowego)
            "n_steps": trial.suggest_int("n_steps", 5, 63),
            # [DODANE]: Normalizacja wariancji sygnału - kluczowe dla rynków finansowych
            "normalize_advantage": trial.suggest_categorical("normalize_advantage", [False, True])
        }
        
    elif model_name in ["ddpg", "td3", "sac"]:
        train_freq_val = trial.suggest_int("train_freq", 1, 10) 
        
        # --- ZMIANA W BUFORZE ---
        # 1. Losujemy mnożnik określający procent całkowitego treningu
        buffer_multiplier = trial.suggest_categorical("buffer_multiplier", [0.2, 0.4, 0.6, 0.8, 1.0])
        # 2. Obliczamy fizyczną wielkość bufora i wymuszamy typ integer
        dynamic_buffer_size = int(steps * buffer_multiplier)

        params = {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
            "weight_decay": weight_decay,
            "gamma": gamma,
            "learning_rate": learning_rate,
            "batch_size": batch_size, 
            
            # 3. Przypisujemy wyliczoną wartość
            "buffer_size": dynamic_buffer_size, 
            
            "tau": trial.suggest_float("tau", 0.001, 0.08, log=True), 
            "train_freq": train_freq_val, 
            "gradient_steps": train_freq_val 
        }
        
        if model_name in ["ddpg", "td3"]:
            params["noise_type"] = "normal"
            # [ZMIANA]: noise_std rozkład logarytmiczny
            params["noise_std"] = trial.suggest_float("noise_std", 0.01, 0.3, log=True)
            
        if model_name == "td3":
            params["policy_delay"] = trial.suggest_int("policy_delay", 2, 5)
            
        if model_name == "sac":
            params["ent_coef"] = trial.suggest_categorical("ent_coef_sac", ["auto"])
            
        return params

'''
def sample_params(trial, model_name):
    # ---------------------------------------------------------
    # WSPÓLNE PARAMETRY DLA WSZYSTKICH MODELI
    # ---------------------------------------------------------
    net_arch = trial.suggest_categorical("net_arch", [64, 128, 256, 512])
    activation_fn_name = trial.suggest_categorical("activation_fn", ["tanh", "relu"])
    activation_fn = {"tanh": torch.nn.Tanh, "relu": torch.nn.ReLU}[activation_fn_name]
    
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    gamma = 1.0 - trial.suggest_float("one_minus_gamma", 0.001, 0.2, log=True)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)

    # ---------------------------------------------------------
    # PARAMETRY SPECYFICZNE DLA ALGORYTMÓW
    # ---------------------------------------------------------
    if model_name == "ppo":
        n_steps = 2 ** trial.suggest_int("n_steps_pow", 5, 11) 
        batch_size = 2 ** trial.suggest_int("batch_size_pow", 6, 10)
        
        return {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
            "weight_decay": weight_decay,
            "gamma": gamma,
            "learning_rate": learning_rate,
            "batch_size": min(batch_size, n_steps),
            "ent_coef": trial.suggest_float("ent_coef", 1e-8, 0.1, log=True),
            "clip_range": trial.suggest_categorical("clip_range", [0.1, 0.2, 0.3]),
            "n_epochs": trial.suggest_int("n_epochs", 3, 20),
            "gae_lambda": 1.0 - trial.suggest_float("one_minus_gae_lambda", 0.001, 0.2, log=True),
            "n_steps": n_steps,
            # [PRZYWRÓCONE Z NOWYM ZAKRESEM] Ochrona przed wybuchem gradientu
            "max_grad_norm": trial.suggest_float("max_grad_norm", 0.1, 1.0, log=True)
        }
        
    elif model_name == "a2c":
        return {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
            "weight_decay": weight_decay,
            "gamma": gamma,
            "learning_rate": learning_rate, 
            "ent_coef": trial.suggest_float("ent_coef", 1e-8, 0.1, log=True),
            "gae_lambda": 1.0 - trial.suggest_float("one_minus_gae_lambda", 0.001, 0.2, log=True),
            "n_steps": trial.suggest_categorical("n_steps", [8, 16, 32, 64, 128, 256]),
            # [PRZYWRÓCONE Z NOWYM ZAKRESEM] Ochrona przed wybuchem gradientu
            "max_grad_norm": trial.suggest_float("max_grad_norm", 0.1, 1.0, log=True)
        }
        
    elif model_name in ["ddpg", "td3", "sac"]:
        buffer_multiplier = trial.suggest_categorical("buffer_multiplier", [0.2, 0.4, 0.6, 0.8, 1.0])
        dynamic_buffer_size = int(steps * buffer_multiplier)

        params = {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
            "weight_decay": weight_decay,
            "gamma": gamma,
            "learning_rate": learning_rate,
            "batch_size": 2 ** trial.suggest_int("batch_size_pow", 6, 10), 
            "buffer_size": dynamic_buffer_size, 
            "tau": trial.suggest_float("tau", 0.001, 0.02, log=True)
        }
        
        if model_name in ["ddpg", "td3"]:
            params["noise_type"] = "normal"
            params["noise_std"] = trial.suggest_float("noise_std", 0.01, 0.1, log=True)
            
        if model_name == "td3":
            # [PRZYWRÓCONE] Sprawdzanie opóźnienia aktora dla trudnych rynków
            params["policy_delay"] = trial.suggest_int("policy_delay", 2, 5)
            
        if model_name == "sac":
            params["ent_coef"] = trial.suggest_categorical("ent_coef_sac", ["auto"])
            
        return params
        
# --- 5. BAZOWE PARAMETRY SIECI ---
base_model_params = {
    "ppo": {}, "a2c": {},
    "ddpg": {"learning_starts": dynamic_starts},
    "td3": {"learning_starts": dynamic_starts},
    "sac": {"learning_starts": dynamic_starts, "ent_coef": "auto"}
}

# --- 6. GŁÓWNA PĘTLA TUNINGU ---
best_hyperparams_dict = {}

for name in TUNE_MODELS:
    print(f"\n{'='*50}\n[{name.upper()}] Inicjalizacja Optuny dla PERIOD_{PERIOD_ID}...\n{'='*50}")
    
    model_tune_dir = os.path.join(TUNING_DIR, name)
    os.makedirs(model_tune_dir, exist_ok=True)
    
    # Baza Optuny zagnieżdżona na poziomie pliku binarnego SQLite pozwala 
    # kontynuować trening po ewentualnym zaniku zasilania lub błędzie systemu
    model_db_path = os.path.join(model_tune_dir, f"optuna_{name}_period{PERIOD_ID}.db")
    model_storage_url = f"sqlite:///{model_db_path}"
    model_csv_path = os.path.join(model_tune_dir, f"trials_{name}_period{PERIOD_ID}.csv")
    tuned_flag_path = os.path.join(model_tune_dir, f"tuned_period{PERIOD_ID}.flag")
    model_best_json = os.path.join(model_tune_dir, f"best_{name}_period{PERIOD_ID}.json")

    if os.path.exists(tuned_flag_path):
        print(f"[{name.upper()}] Wykryto plik 'tuned_period{PERIOD_ID}.flag'. Pomijam tuning dla tego okresu.")
        if os.path.exists(model_best_json):
            with open(model_best_json, 'r') as f:
                best_hyperparams_dict[name] = json.load(f)
        continue

    # Zmodyfikowany nagłówek CSV (dodana kolumna na wyniki cząstkowe poszczególnych nasion)
    if not os.path.exists(model_csv_path):
        with open(model_csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["Model", "Trial", "Status", "Median_Sortino_std_penalty", "Seed_Scores", "Params"])

    def objective(trial):
        # 1. Losowanie parametrów z przestrzeni poszukiwań (jeden raz dla wszystkich nasion)
        sampled_kwargs = sample_params(trial, name)
        
        for past_trial in trial.study.trials:
            if past_trial.state == optuna.trial.TrialState.COMPLETE and past_trial.number != trial.number:
                if past_trial.params == trial.params:
                    print(f"  [INFO] Wykryto powtórkę parametrów z próby {past_trial.number}! Pomijam obliczenia.")
                    return past_trial.value 
        
        # 2. Inżynieria słownika hiperparametrów
        net_size = sampled_kwargs.pop("net_arch")
        activation_fn = sampled_kwargs.pop("activation_fn")
        w_decay = sampled_kwargs.pop("weight_decay")
        
        if name in ["ppo", "a2c"]:
            policy_kw = dict(net_arch=dict(pi=[net_size, net_size], vf=[net_size, net_size]), 
                             activation_fn=activation_fn, optimizer_kwargs=dict(weight_decay=w_decay))
        else:
            policy_kw = dict(net_arch=dict(pi=[net_size, net_size], qf=[net_size, net_size]), 
                             activation_fn=activation_fn, optimizer_kwargs=dict(weight_decay=w_decay))
            
        kwargs = base_model_params[name].copy()
        kwargs.update(sampled_kwargs)
        
        if name in ['ddpg', 'td3']:
            kwargs.pop("noise_type", None) 
            noise_std = kwargs.pop("noise_std", 0.1)
            kwargs["action_noise"] = NormalActionNoise(mean=np.zeros(stock_dim), sigma=noise_std * np.ones(stock_dim))
            
        kwargs.update({"device": "cpu"})

        # 3. PĘTLA PO NASIONACH (Multi-seed evaluation)
        sortino_scores = []
        
        for seed_idx, current_seed in enumerate(EVAL_SEEDS):
            set_seeds(current_seed)

            def make_train_env():
                e = StockTradingEnvCustom(df=train, **{**env_kwargs, "turbulence_threshold": None})
                e.action_space.seed(current_seed) # Zabezpieczenie losowania learning_starts
                return Monitor(e)
                
            trial_env_train = DummyVecEnv([make_train_env])
            trial_env_train = VecNormalize(trial_env_train, norm_obs=True, norm_reward=True, clip_obs=10., gamma=0.99)
            trial_env_train.seed(current_seed)
            
            trial_eval_env = DummyVecEnv([lambda: StockTradingEnvCustom(df=val, **{**env_kwargs, "turbulence_threshold": None})])
            trial_eval_env = VecNormalize(trial_eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)

            model = SB3_MODELS[name](
                policy="MlpPolicy",
                env=trial_env_train,
                policy_kwargs=policy_kw,
                seed=current_seed,
                **kwargs
            )

            tuning_callback = TuningEvalCallback(
                trial=trial, 
                eval_env=trial_eval_env, 
                eval_freq=eval_freq, 
                initial_capital=INITIAL_CAPITAL,
                reward_scaling=REWARD_SCALING, 
                normalization_source=trial_env_train,
                seed_idx=seed_idx,               # Indeks (0, 1, 2)
                total_steps_per_seed=steps       # Całkowita liczba kroków epizodu
            )

            try:
                model.learn(total_timesteps=steps, callback=tuning_callback)
            except optuna.exceptions.TrialPruned:
                # Jeśli Median Pruner utnie proces w trakcie uczenia nasiona, ratujemy to, co zdążyliśmy zapisać
                trial.set_user_attr("seed_scores", sortino_scores)
                raise optuna.exceptions.TrialPruned()

            # Zapis ostatecznego wyniku Sortino dla pojedynczego nasiona
            seed_sortino = tuning_callback.best_sortino
            sortino_scores.append(seed_sortino)
            
            # Wewnętrzny ręczny Pruning (oszczędność czasu)
            # Jeśli pierwsze ziarno ma tragiczny wynik, nie trać czasu na kolejne dwa!
            if seed_idx == 0 and seed_sortino < -10:
                trial.set_user_attr("seed_scores", sortino_scores)
                raise optuna.exceptions.TrialPruned()

        # Po przetrenowaniu na wszystkich nasionach
        trial.set_user_attr("seed_scores", sortino_scores)
        
        # 1. Obliczenie miar statystycznych
        median_sortino = np.median(sortino_scores)
        std_sortino = np.std(sortino_scores)
        
        # 2. Współczynnik kary (lambda)
        # 1 to dobry kompromis. Zbyt duże (np. 2.0) sprawi, że Optuna będzie 
        # promować modele, które nic nie robią (bo wtedy wariancja wynosi 0).
        penalty_weight = 1
        
        # 3. Finalna metryka celu
        final_metric = median_sortino - (penalty_weight * std_sortino)
        
        # Zapiszmy sobie to odchylenie w bazie, żeby móc je analizować w CSV
        trial.set_user_attr("std_dev", float(std_sortino))
        
        return final_metric

    def optuna_csv_logger(study, trial):
        """Zapisuje ślad rewizyjny po wykonaniu każdej próby w formacie CSV."""
        with open(model_csv_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            val_score = round(trial.value, 4) if trial.value is not None else "PRUNED"
            
            # Pobranie tablicy wyników cząstkowych z atrybutów użytkownika
            seed_scores = trial.user_attrs.get("seed_scores", [])
            seed_scores_str = str([round(s, 4) for s in seed_scores])
            
            writer.writerow([name.upper(), trial.number, trial.state.name, val_score, seed_scores_str, json.dumps(trial.params)])

    # Tworzenie obiektu "study"
    study_name = f"study_{name}_period{PERIOD_ID}"
    study = optuna.create_study(
        study_name=study_name,
        storage=model_storage_url,   
        load_if_exists=True,         
        direction="maximize", 
        sampler=TPESampler(seed=42, n_startup_trials=5), # Zostawiamy ziarno dla optymalizatora TPE (reprodukowalność doboru parametrów)
        # Median Pruner na 3 nasiona po 10 kroków (eval_freq) to 30 ewaluacji na kombinację.
        # n_warmup_steps=3 oznacza, że algorytm pozwala na swobodny bieg przez pierwsze 3 punkty sprawdzające.
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    )

    completed_trials = len(study.trials)
    trials_to_run = max(0, N_TRIALS - completed_trials)

    if trials_to_run > 0:
        print(f"[{name.upper()}] Zakończono: {completed_trials}/{N_TRIALS}. Uruchamiam {trials_to_run} prób...")
        study.optimize(objective, n_trials=trials_to_run, callbacks=[optuna_csv_logger])
    
    if len(study.trials) >= N_TRIALS:
        print(f"[{name.upper()}] Optymalizacja UKOŃCZONA dla Period {PERIOD_ID}.")
        print(f"[{name.upper()}] Najlepsza Mediana Sortino: {study.best_value:.4f}")
        
        best_hyperparams_dict[name] = study.best_params
        with open(model_best_json, 'w') as f:
            json.dump(study.best_params, f, indent=4)
            
        with open(tuned_flag_path, 'w') as f:
            f.write("DONE")

# --- 7. ZAPIS FINALNYCH WYNIKÓW ---
# Konsolidacja najlepszych wyników dla wszystkich badanych architektur do jednego centralnego pliku
output_json_path = os.path.join(TUNING_DIR, f"best_hyperparameters_continuous_period{PERIOD_ID}.json")
with open(output_json_path, 'w') as f:
    json.dump(best_hyperparams_dict, f, indent=4)

print(f"\n[SUKCES] Tuning całej grupy dla PERIOD_{PERIOD_ID} zakończony pomyślnie.")
print(f"Główny plik z parametrami dla Treningu zapisano w: {output_json_path}")

Podaj numer okresu do tuningu (Period ID) [0-3]:  0



=== START TUNINGU OPTUNA (RESUMABLE & CONTINUOUS) DLA PERIOD_0 ===


Podaj model do tuningu (ppo, a2c, ddpg, td3, sac) lub wciśnij Enter dla wszystkich:  a2c


[INFO] Wybrano pojedynczy model: A2C
[SYSTEM] Stock Dim: 16
[SYSTEM] Indicators count: 15
[SYSTEM] State Dim: 273
[SYSTEM] Tuning dla zakresu: Trening [2014-01-01 - 2019-10-01), Walidacja [2019-10-01 - 2020-01-01)
[SYSTEM] Tuned Episode Length: 1436 days
[SYSTEM] Total Timesteps: 143600 | Eval Freq: 14360

[A2C] Inicjalizacja Optuny dla PERIOD_0...


[I 2026-05-24 15:53:27,608] A new study created in RDB with name: study_a2c_period0


[A2C] Zakończono: 0/50. Uruchamiam 50 prób...
day: 1435, episode: 10
begin_total_asset: 100000.00
end_total_asset: 26720.96
total_reward: -73279.04
total_cost: 68390.66
total_trades: 16193
Sharpe: -1.085
day: 1435, episode: 20
begin_total_asset: 100000.00
end_total_asset: 29168.71
total_reward: -70831.29
total_cost: 74917.80
total_trades: 16257
Sharpe: -0.999
day: 1435, episode: 30
begin_total_asset: 100000.00
end_total_asset: 22850.05
total_reward: -77149.95
total_cost: 58873.18
total_trades: 15944
Sharpe: -1.210
day: 1435, episode: 40
begin_total_asset: 100000.00
end_total_asset: 28778.13
total_reward: -71221.87
total_cost: 69089.17
total_trades: 16210
Sharpe: -1.004
day: 1435, episode: 50
begin_total_asset: 100000.00
end_total_asset: 58883.13
total_reward: -41116.87
total_cost: 98807.98
total_trades: 16349
Sharpe: -0.365
day: 1435, episode: 60
begin_total_asset: 100000.00
end_total_asset: 56998.29
total_reward: -43001.71
total_cost: 85999.08
total_trades: 16290
Sharpe: -0.340
day: 1

[I 2026-05-24 15:59:59,026] Trial 0 finished with value: 2.2387339462746016 and parameters: {'net_arch': 128, 'activation_fn': 'tanh', 'weight_decay': 1.493656855461763e-06, 'one_minus_gamma': 0.09842315738502598, 'learning_rate_a2c': 0.00010502105436744271, 'ent_coef_a2c': 0.00416043964525661, 'max_grad_norm': 0.31788672591451017, 'one_minus_gae_lambda': 0.17052641538983093, 'n_steps': 54, 'normalize_advantage': False}. Best is trial 0 with value: 2.2387339462746016.


day: 1435, episode: 100
begin_total_asset: 100000.00
end_total_asset: 236609.58
total_reward: 136609.58
total_cost: 150829.25
total_trades: 16423
Sharpe: 0.773
day: 1435, episode: 10
begin_total_asset: 100000.00
end_total_asset: 33636.78
total_reward: -66363.22
total_cost: 72698.98
total_trades: 16017
Sharpe: -0.853
day: 1435, episode: 20
begin_total_asset: 100000.00
end_total_asset: 28046.37
total_reward: -71953.63
total_cost: 66361.42
total_trades: 16198
Sharpe: -1.041
day: 1435, episode: 30
begin_total_asset: 100000.00
end_total_asset: 21658.55
total_reward: -78341.45
total_cost: 60067.79
total_trades: 16015
Sharpe: -1.308
day: 1435, episode: 40
begin_total_asset: 100000.00
end_total_asset: 41146.10
total_reward: -58853.90
total_cost: 73889.23
total_trades: 16234
Sharpe: -0.688
day: 1435, episode: 50
begin_total_asset: 100000.00
end_total_asset: 32914.32
total_reward: -67085.68
total_cost: 52792.62
total_trades: 16128
Sharpe: -0.872
day: 1435, episode: 60
begin_total_asset: 100000.0

AssertionError: Should not reach.

### Obliczenia analogiczne do tuningu Optuna, ale dla domyślnych hiperparametrów z biblioteki Stable Baselines 3 (w celu porównania wyników)

In [17]:
SB3_MODELS = {
    "ppo": PPO,
    "a2c": A2C,
    "ddpg": DDPG,
    "td3": TD3,
    "sac": SAC
}

# --- 0. KONFIGURACJA BAZOWA ---
EVAL_SEEDS = [1, 2, 3] # Nasze 3 przebiegi losowości
REWARD_SCALING = 100
MAX_POSITION_PCT = 1

OUTPUT_DIR = os.path.join(os.getcwd(), "Default_Hyperparams_Evaluation")
os.makedirs(OUTPUT_DIR, exist_ok=True)
RESULTS_CSV_PATH = os.path.join(OUTPUT_DIR, "default_baselines_results.csv")

def set_seeds(seed=1):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# --- 1. CALLBACK DO ZAPISYWANIA WYNIKÓW BEZ OPTUNY ---
class DefaultEvalCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq, initial_capital=100000, normalization_source=None):
        super(DefaultEvalCallback, self).__init__(verbose=0)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.initial_capital = initial_capital
        self.normalization_source = normalization_source
        self.best_sortino = -np.inf 

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            # Kopiowanie parametrów normalizacji (ważne, aby nie zepsuć statystyk treningowych)
            if self.normalization_source is not None and isinstance(self.eval_env, VecNormalize):
                self.eval_env.obs_rms = copy.deepcopy(self.normalization_source.obs_rms)
                self.eval_env.training = False 
                self.eval_env.norm_reward = False 
                
            obs = self.eval_env.reset()
            done = False
            portfolio_values = [self.initial_capital]
            
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, rewards, dones, info = self.eval_env.step(action)
                current_total_assets = info[0].get('total_assets', portfolio_values[-1])
                portfolio_values.append(current_total_assets)
                done = dones[0]

            df_account = pd.DataFrame(portfolio_values, columns=['value'])
            returns = df_account['value'].pct_change().dropna()
            
            if len(returns) < 2 or returns.std() == 0:
                sortino = 0.0
            else:
                neg_ret = returns[returns < 0]
                downside_std = neg_ret.std(ddof=0) if len(neg_ret) > 1 else 1e-6
                expected_return = returns.mean() * 252
                annual_downside = downside_std * (np.sqrt(252))
                sortino = expected_return / (annual_downside + 1e-9)

            if sortino > self.best_sortino:
                self.best_sortino = sortino

        return True

# --- 2. INICJALIZACJA PLIKU WYNIKOWEGO ---
if not os.path.exists(RESULTS_CSV_PATH):
    with open(RESULTS_CSV_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["Period", "Model", "Seed_Scores", "Median", "Std", "Median_minus_Std"])

print(f"=== START EWALUACJI BAZOWYCH HIPERPARAMETRÓW (SB3 DEFAULTS) ===")

# --- 3. GŁÓWNA PĘTLA: OKRESY -> MODELE -> NASIONA ---
# Iterujemy na sztywno po 4 okresach (0, 1, 2, 3)
for PERIOD_ID in range(4):
    print(f"\n{'='*60}\nPRZETWARZAM OKRES: {PERIOD_ID}\n{'='*60}")
    
    # Przygotowanie wskaźników dla okresu
    period_key = f"okres_{PERIOD_ID}"
    indicators = indicators_dict.get(period_key, indicators_dict.get("okres_0", [])) 
    
    unique_tickers = sorted(df_global.tic.unique())
    stock_dim = len(unique_tickers)
    state_dim = 1 + 2 * stock_dim + len(indicators) * stock_dim

    test_start = GLOBAL_DATES[PERIOD_ID]
    train_start = test_start - relativedelta(years=TRAIN_YEARS)
    train_end = test_start - relativedelta(months=VAL_MONTHS)
    val_start = train_end

    train = df_global[(df_global.date >= train_start) & (df_global.date < train_end)].copy()
    val = df_global[(df_global.date >= val_start) & (df_global.date < test_start)].copy()

    train.index = train.date.factorize()[0]
    val.index = val.date.factorize()[0]

    target_exposure = INITIAL_CAPITAL * MAX_POSITION_PCT
    hmax_list = []

    for tic in unique_tickers:
        tic_data = train[train.tic == tic]
        max_price = tic_data['close'].max() if not tic_data.empty else 100.0
        if pd.isna(max_price) or max_price <= 0: max_price = 1.0
        limit = target_exposure / (max_price * (1 + COST))
        hmax_list.append(round(float(limit), 4))

    env_kwargs = {
        "hmax": hmax_list, "initial_amount": INITIAL_CAPITAL, "buy_cost_pct": [COST]*stock_dim, 
        "sell_cost_pct": [COST]*stock_dim, "state_space": state_dim, "stock_dim": stock_dim, 
        "tech_indicator_list": indicators, "action_space": stock_dim, "reward_scaling": REWARD_SCALING, 
        "reward_type": "log", "risk_penalty": 0.1,  
        "num_stock_shares": [0]*stock_dim, "risk_indicator_col": "turbulence"
    }

    episode_length = len(train.index.unique())
    steps = int(100 * episode_length)  
    eval_freq = steps // 10  
    dynamic_starts = steps // 10

    # Baza parametrów: Pozostawiamy TYLKO to co absolutnie niezbędne do działania algorytmu 
    # (learning_starts zapobiega crashom DDPG/SAC przy krótkich epizodach, reszta przejmuje domyślne ustawienia SB3)
    base_model_params = {
        "ppo": {}, "a2c": {},
        "ddpg": {"learning_starts": dynamic_starts},
        "td3": {"learning_starts": dynamic_starts},
        "sac": {"learning_starts": dynamic_starts, "ent_coef": "auto"}
    }

    # Pętla po 5 modelach
    for model_name, model_class in SB3_MODELS.items():
        print(f"\n---> Model: {model_name.upper()}")
        seed_sortino_scores = []
        
        # Pętla po 3 nasionach
        for current_seed in EVAL_SEEDS:
            set_seeds(current_seed)
            print(f"     Trenowanie - Ziarno {current_seed}...", end=" ", flush=True)
            
            def make_train_env():
                e = StockTradingEnvCustom(df=train, **{**env_kwargs, "turbulence_threshold": None})
                e.action_space.seed(current_seed)
                return Monitor(e)
                
            env_train = DummyVecEnv([make_train_env])
            env_train = VecNormalize(env_train, norm_obs=True, norm_reward=True, clip_obs=10., gamma=0.99)
            env_train.seed(current_seed)
            
            eval_env = DummyVecEnv([lambda: StockTradingEnvCustom(df=val, **{**env_kwargs, "turbulence_threshold": None})])
            eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)
            
            kwargs = base_model_params[model_name].copy()
            kwargs["device"] = "cpu"
            
            # Wymagany szum dla ciągłej przestrzeni akcji przy ustawieniach domyślnych DDPG/TD3
            if model_name in ['ddpg', 'td3']:
                noise_std = 0.1  # Domyślny noise SB3
                kwargs["action_noise"] = NormalActionNoise(mean=np.zeros(stock_dim), sigma=noise_std * np.ones(stock_dim))
            
            # Inicjalizacja modelu TYLKO z domyślnymi parametrami polityki (brak modyfikacji net_arch, gae, gamma, itp.)
            model = model_class(
                policy="MlpPolicy",
                env=env_train,
                seed=current_seed,
                **kwargs
            )
            
            default_callback = DefaultEvalCallback(
                eval_env=eval_env, 
                eval_freq=eval_freq, 
                initial_capital=INITIAL_CAPITAL,
                normalization_source=env_train
            )
            
            # Uczenie modelu z domyślnymi ustawieniami
            model.learn(total_timesteps=steps, callback=default_callback)
            
            best_seed_score = default_callback.best_sortino
            seed_sortino_scores.append(best_seed_score)
            print(f"Najlepszy Sortino (Walidacja): {best_seed_score:.4f}")
            
        # Obliczenie statystyk dla wpisania do docelowej tabeli (Opcja "D")
        median_score = np.median(seed_sortino_scores)
        std_score = np.std(seed_sortino_scores)
        final_metric = median_score - std_score
        
        print(f"   [WYNIK {model_name.upper()}] Mediana: {median_score:.4f} | Odch. Std: {std_score:.4f} | Mediana - Odch: {final_metric:.4f}")
        
        # Zapis do CSV
        with open(RESULTS_CSV_PATH, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            scores_str = str([round(s, 4) for s in seed_sortino_scores])
            writer.writerow([PERIOD_ID, model_name.upper(), scores_str, round(median_score, 4), round(std_score, 4), round(final_metric, 4)])

print(f"\n[SUKCES] Zakończono generowanie wyników Baseline (D). Zapisano do: {RESULTS_CSV_PATH}")

=== START EWALUACJI BAZOWYCH HIPERPARAMETRÓW (SB3 DEFAULTS) ===

PRZETWARZAM OKRES: 0

---> Model: PPO
     Trenowanie - Ziarno 1... day: 1435, episode: 10
begin_total_asset: 100000.00
end_total_asset: 39648.48
total_reward: -60351.52
total_cost: 87280.95
total_trades: 16178
Sharpe: -0.723
day: 1435, episode: 20
begin_total_asset: 100000.00
end_total_asset: 23383.88
total_reward: -76616.12
total_cost: 69391.59
total_trades: 15953
Sharpe: -1.177
day: 1435, episode: 30
begin_total_asset: 100000.00
end_total_asset: 66670.22
total_reward: -33329.78
total_cost: 94788.48
total_trades: 15952
Sharpe: -0.231
day: 1435, episode: 40
begin_total_asset: 100000.00
end_total_asset: 118538.30
total_reward: 18538.30
total_cost: 109786.62
total_trades: 16179
Sharpe: 0.244
day: 1435, episode: 50
begin_total_asset: 100000.00
end_total_asset: 149292.87
total_reward: 49292.87
total_cost: 115016.14
total_trades: 15997
Sharpe: 0.430
day: 1435, episode: 60
begin_total_asset: 100000.00
end_total_asset: 345689.4

KeyboardInterrupt: 

### Trening algorytmów 
Przeprowadzany wielokrotnie, na wielu ziarnach losowości, z wykorzystaniem hiperparametrów wyznaczonych w procesie strojenia

In [ ]:
def build_tuned_kwargs(model_name, raw_params, stock_dim, steps, episode_length):
    """
    Kluczowa funkcja tłumacząca płaską strukturę hiperparametrów wygenerowaną 
    przez estymator Optuna (TPE) na obiekty klasowe zgodne z wymogami Stable Baselines3 (SB3).
    Dba o zagnieżdżone obiekty, w tym funkcje aktywacji, logikę szumu i regularyzację L2.
    """
    if model_name not in raw_params:
        return None 
        
    rp = raw_params[model_name]
    p = {} 
    
    act_fn = torch.nn.Tanh if rp.get("activation_fn") == "tanh" else torch.nn.ReLU
    net_arch_raw = rp.get("net_arch", 128)
    net_arch_param = [net_arch_raw, net_arch_raw] if isinstance(net_arch_raw, int) else net_arch_raw
    
    w_decay = rp.get("weight_decay", 0.0)
    opt_kwargs = dict(weight_decay=w_decay)

    if model_name in ["ppo", "a2c"]:
        p["policy_kwargs"] = dict(net_arch=dict(pi=net_arch_param, vf=net_arch_param), activation_fn=act_fn, optimizer_kwargs=opt_kwargs)
    else:
        p["policy_kwargs"] = dict(net_arch=dict(pi=net_arch_param, qf=net_arch_param), activation_fn=act_fn, optimizer_kwargs=opt_kwargs)
        
    # [POPRAWKA]: Odczyt gotowych wartości wprost z JSON
    if "gamma" in rp: p["gamma"] = rp["gamma"]
    if "learning_rate" in rp: p["learning_rate"] = rp["learning_rate"]
    
    if model_name != "a2c":
        if "batch_size" in rp:
            p["batch_size"] = rp["batch_size"]

    # [POPRAWKA]: Wczytanie wspólnych parametrów (w tym 'auto' dla SAC)
    if "ent_coef" in rp: p["ent_coef"] = rp["ent_coef"]
    if "max_grad_norm" in rp: p["max_grad_norm"] = rp["max_grad_norm"]
        
    if "gae_lambda" in rp: 
        p["gae_lambda"] = rp["gae_lambda"]

    if model_name == "ppo":
        if "n_steps" in rp: 
            p["n_steps"] = min(rp["n_steps"], episode_length)
            
        if "batch_size" in p:
            p["batch_size"] = min(p["batch_size"], p.get("n_steps", p["batch_size"]))
        
        if "n_epochs" in rp: p["n_epochs"] = rp["n_epochs"]
        if "clip_range" in rp: p["clip_range"] = rp["clip_range"]
        
    elif model_name == "a2c":
        if "n_steps" in rp: p["n_steps"] = rp["n_steps"]

    if model_name in ["ddpg", "td3", "sac"]:
        p["learning_starts"] = steps // 10
        if "buffer_size" in rp: p["buffer_size"] = rp["buffer_size"]
        if "tau" in rp: p["tau"] = rp["tau"]
        # Parametry train_freq i gradient_steps zostały wyeliminowane z Optuny, SB3 użyje optymalnych domyślnych
            
    if model_name == "td3" and "policy_delay" in rp: p["policy_delay"] = rp["policy_delay"]

    if model_name in ["ddpg", "td3"]:
        n_std = rp.get("noise_std", 0.1)
        p["action_noise"] = NormalActionNoise(mean=np.zeros(stock_dim), sigma=n_std * np.ones(stock_dim))

    return p

def build_tuned_kwargs(model_name, raw_params, stock_dim, steps, episode_length):
    """
    Kluczowa funkcja tłumacząca płaską strukturę hiperparametrów wygenerowaną 
    przez estymator Optuna (TPE) na obiekty klasowe zgodne z wymogami Stable Baselines3 (SB3).
    Dba o zagnieżdżone obiekty, w tym funkcje aktywacji, logikę szumu i regularyzację L2.
    """
    if model_name not in raw_params:
        return None 
        
    rp = raw_params[model_name]
    p = {} 
    
    act_fn = torch.nn.Tanh if rp.get("activation_fn") == "tanh" else torch.nn.ReLU
    net_arch_raw = rp.get("net_arch", 128)
    net_arch_param = [net_arch_raw, net_arch_raw] if isinstance(net_arch_raw, int) else net_arch_raw
    
    w_decay = rp.get("weight_decay", 0.0)
    opt_kwargs = dict(weight_decay=w_decay)

    if model_name in ["ppo", "a2c"]:
        p["policy_kwargs"] = dict(net_arch=dict(pi=net_arch_param, vf=net_arch_param), activation_fn=act_fn, optimizer_kwargs=opt_kwargs)
    else:
        p["policy_kwargs"] = dict(net_arch=dict(pi=net_arch_param, qf=net_arch_param), activation_fn=act_fn, optimizer_kwargs=opt_kwargs)
        
    # [POPRAWKA]: Odczyt i kalkulacja transformowanych wartości zmiennoprzecinkowych z JSON
    if "one_minus_gamma" in rp: 
        p["gamma"] = 1.0 - rp["one_minus_gamma"]
    elif "gamma" in rp: 
        p["gamma"] = rp["gamma"]

    if "learning_rate" in rp: 
        p["learning_rate"] = rp["learning_rate"]
    
    # [POPRAWKA]: Kalkulacja wielkości batcha na podstawie potęgi dwójki
    if model_name != "a2c":
        if "batch_size_pow" in rp:
            p["batch_size"] = 2 ** rp["batch_size_pow"]
        elif "batch_size" in rp:
            p["batch_size"] = rp["batch_size"]

    # [POPRAWKA]: Obsługa współczynnika entropii zarówno dla PPO/A2C, jak i SAC
    if "ent_coef" in rp: 
        p["ent_coef"] = rp["ent_coef"]
    if "ent_coef_sac" in rp: 
        p["ent_coef"] = rp["ent_coef_sac"]
        
    if "max_grad_norm" in rp: 
        p["max_grad_norm"] = rp["max_grad_norm"]
        
    # [POPRAWKA]: Odwrócenie wartości GAE Lambda
    if "one_minus_gae_lambda" in rp: 
        p["gae_lambda"] = 1.0 - rp["one_minus_gae_lambda"]
    elif "gae_lambda" in rp:
        p["gae_lambda"] = rp["gae_lambda"]

    # Obliczanie n_steps dla PPO i A2C
    n_steps_calc = None
    if "n_steps_pow" in rp:
        n_steps_calc = 2 ** rp["n_steps_pow"]
    elif "n_steps" in rp:
        n_steps_calc = rp["n_steps"]

    if model_name == "ppo":
        if n_steps_calc is not None: 
            p["n_steps"] = min(n_steps_calc, episode_length)
            
        if "batch_size" in p:
            p["batch_size"] = min(p["batch_size"], p.get("n_steps", p["batch_size"]))
        
        if "n_epochs" in rp: p["n_epochs"] = rp["n_epochs"]
        if "clip_range" in rp: p["clip_range"] = rp["clip_range"]
        
    elif model_name == "a2c":
        if n_steps_calc is not None: 
            p["n_steps"] = n_steps_calc

    if model_name in ["ddpg", "td3", "sac"]:
        p["learning_starts"] = steps // 10
        if "buffer_size" in rp: p["buffer_size"] = rp["buffer_size"]
        if "tau" in rp: p["tau"] = rp["tau"]
            
    if model_name == "td3" and "policy_delay" in rp: p["policy_delay"] = rp["policy_delay"]

    if model_name in ["ddpg", "td3"]:
        n_std = rp.get("noise_std", 0.1)
        p["action_noise"] = NormalActionNoise(mean=np.zeros(stock_dim), sigma=n_std * np.ones(stock_dim))

    return p
    
MODELS = ["ppo", "a2c", "ddpg", "td3", "sac"]

# ==============================================================================
# KONFIGURACJA
# ==============================================================================
stock_dim = len(FILES_CONFIG)
MAX_POSITION_PCT = 1
REWARD_SCALING = 100

SB3_MODELS = {
    "ppo": PPO,
    "a2c": A2C,
    "ddpg": DDPG,
    "td3": TD3,
    "sac": SAC
}

print(f"[SYSTEM] Global COST: {COST} (Linear/Spread)")
print(f"[SYSTEM] Mode: FRACTIONAL SHARES (Float)")
print(f"[SYSTEM] Target Max Position: {INITIAL_CAPITAL * MAX_POSITION_PCT:.0f} PLN per stock")
print("[SYSTEM] Env Wrapper: Monitor -> StockTradingEnvCustom (XTB) -> DummyVecEnv -> VecNormalize")

print("\n=== START TRENINGU - WALK-FORWARD WITH VEC_NORMALIZE ===")

# --- 0. FUNKCJA RYSOWANIA WYNIKÓW TRENINGU ---
def plot_learning_process(path, period_id, model_names, episode_length):
    """Generuje wykresy diagnostyczne po zakończeniu treningu danego okresu Walk-Forward."""
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))
    
    # 1. Sortino Ratio (Ewaluacja)
    ax = axes[0]
    for name in model_names:
        log_file = os.path.join(path, name, "evaluations.csv")
        if os.path.exists(log_file):
            try:
                df = pd.read_csv(log_file)
                if not df.empty and 'sortino' in df.columns:
                    ax.plot(df['step'] / episode_length, df['sortino'], label=name, marker='o', markersize=3)
            except: pass
    ax.set_title(f"Period {period_id}: Validation Sortino Ratio")
    ax.set_xlabel("Episodes"); ax.set_ylabel("Sortino Ratio"); ax.grid(True, alpha=0.3); ax.legend()

    # 2. Action Variance (Diagnoza kolapsu akcji - Action Collapse)
    ax = axes[1]
    for name in model_names:
        log_file = os.path.join(path, name, "evaluations.csv")
        if os.path.exists(log_file):
            try:
                df = pd.read_csv(log_file)
                if not df.empty and 'action_var' in df.columns:
                    ax.plot(df['step'] / episode_length, df['action_var'], label=name)
            except: pass
    ax.set_title(f"Period {period_id}: Action Variance")
    ax.set_xlabel("Episodes"); ax.set_ylabel("Variance"); ax.grid(True, alpha=0.3); ax.legend()

    # 3. Training Reward (Zbieżność funkcji celu)
    ax = axes[2]
    for name in model_names:
        log_file = os.path.join(path, name, "train_log", "monitor.csv")
        if os.path.exists(log_file):
            try:
                df = pd.read_csv(log_file, skiprows=1)
                x_axis = df.index + 1
                window = 10 
                y_axis = df['r'].rolling(window=window, min_periods=1).mean()
                ax.plot(x_axis, y_axis, label=f"{name} (MA {window})")
            except Exception as e: 
                print(f"Błąd rysowania reward dla {name}: {e}")
                pass
                
    ax.set_title(f"Period {period_id}: Training Reward (Cumulative)")
    ax.set_xlabel("Episodes"); ax.set_ylabel("Episode Reward"); ax.grid(True, alpha=0.3); ax.legend()

    img_path = os.path.join(path, "learning_curve_full.png")
    plt.tight_layout()
    plt.savefig(img_path)
    plt.close()

# --- CALLBACK (STATS SYNC Z SORTINO) ---
class SortinoEvalCallback(BaseCallback):
    """
    Monitor szkolenia, który zatrzymuje i weryfikuje środowisko treningowe.
    Ocenia jakość modelu opierając się na wskaźniku Sortino w oknie walidacyjnym (Out-of-Sample).
    """
    def __init__(self, eval_env, eval_freq, log_path, best_model_save_path, 
                 initial_capital=100000, reward_scaling=100, verbose=1, 
                 normalization_source=None):
        super(SortinoEvalCallback, self).__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.log_path = log_path
        self.best_model_save_path = best_model_save_path
        self.best_sortino = -np.inf
        self.initial_capital = initial_capital
        self.reward_scaling = reward_scaling
        self.eval_history = [] 
        self.normalization_source = normalization_source

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            
            # Bezpieczny transfer statystyk standaryzujących. Wagi z dynamicznie aktualizowanego
            # wektora uczącego (VecNormalize) są "zamrażane" i kopiowane do wektora testowego. 
            if self.normalization_source is not None and isinstance(self.eval_env, VecNormalize):
                self.eval_env.obs_rms = copy.deepcopy(self.normalization_source.obs_rms)
                self.eval_env.training = False 
                self.eval_env.norm_reward = False 

            obs = self.eval_env.reset()
            done = False
            portfolio_values = [self.initial_capital]
            all_actions = [] 
            total_reward = 0.0
            
            while not done:
                action, _state = self.model.predict(obs, deterministic=True)
                all_actions.append(action[0])
                obs, rewards, dones, info = self.eval_env.step(action)
                
                # Zabezpieczenie na wypadek niespójnej krotki informacyjnej pomiędzy bibliotekami
                current_total_assets = info[0].get('total_assets', portfolio_values[-1])
                portfolio_values.append(current_total_assets)
                
                total_reward += rewards[0]
                done = dones[0]
            
            # Przywrócenie trybu uczenia dla oryginalnego wektora środowiska
            if isinstance(self.eval_env, VecNormalize):
                self.eval_env.training = True

            # Obliczenie statystyk asymetrycznego ryzyka spadku (Downside Deviation)
            df_account = pd.DataFrame(portfolio_values, columns=['value'])
            returns = df_account['value'].pct_change().dropna()
            
            if len(returns) < 2 or returns.std() == 0: 
                sortino = 0.0
            else: 
                neg_returns = returns[returns < 0]
                downside_std = neg_returns.std(ddof=0) if len(neg_returns) > 1 else 1e-6
                expected_return = returns.mean() * 252 
                annual_downside = downside_std * (252 ** 0.5)
                sortino = expected_return / (annual_downside + 1e-9)
            
            # Detekcja wariancji akcji zabezpiecza przed patologią modelu (stały wektor odpowiedzi)
            actions_array = np.array(all_actions)
            step_action_variance = np.var(actions_array, axis=0).mean() if len(actions_array) > 0 else 0.0
            
            self.eval_history.append([self.num_timesteps, sortino, total_reward, step_action_variance])
            pd.DataFrame(self.eval_history, columns=['step', 'sortino', 'total_reward', 'action_var'])\
              .to_csv(os.path.join(self.log_path, "evaluations.csv"), index=False)

            # Rejestracja nowego lidera, jeśli aktualny Sortino przebił dotychczasowe maksimum
            if sortino > self.best_sortino:
                self.best_sortino = sortino
                self.model.save(os.path.join(self.best_model_save_path, "best_model"))
                # Trwały zapis zamrożonych statystyk wektora normalizującego, niezbędny do 
                # prawidłowego odtworzenia percepcji agenta na etapie symulacji handlowej (Out-of-Sample).
                if hasattr(self.model.get_env(), "save"):
                    self.model.get_env().save(os.path.join(self.best_model_save_path, "best_model_vecnorm.pkl"))
                
        return True

# --- 3. KONFIGURACJA ŚRODOWISKA URUCHOMIENIOWEGO ---
# Zdolność do uruchomienia wielu izolowanych instancji treningowych (Run ID). 
raw_input = input("Podaj RUN_ID (np. pojedyncza liczba '0', przedział '0-3' lub lista '0,2,4') [0]: ").strip()

# Parsowanie wejścia użytkownika
run_ids = []
if not raw_input:
    run_ids = [0]
elif "-" in raw_input:
    try:
        start_id, end_id = map(int, raw_input.split("-"))
        run_ids = list(range(start_id, end_id + 1))
    except:
        run_ids = [0]
elif "," in raw_input:
    try:
        run_ids = [int(x.strip()) for x in raw_input.split(",")]
    except:
        run_ids = [0]
else:
    try:
        run_ids = [int(raw_input)]
    except:
        run_ids = [0]

# --- NOWY BLOK: WYBÓR MODELI DO TRENOWANIA ---
AVAILABLE_MODELS = ["ppo", "a2c", "ddpg", "td3", "sac"]
raw_models_input = input(f"Podaj modele do trenowania oddzielone przecinkiem {AVAILABLE_MODELS} [Enter = wszystkie]: ").strip().lower()

if not raw_models_input:
    MODELS = AVAILABLE_MODELS
else:
    # Rozbicie po przecinku, usunięcie spacji i weryfikacja z listą dostępnych
    input_models = [m.strip() for m in raw_models_input.split(",")]
    MODELS = [m for m in input_models if m in AVAILABLE_MODELS]
    
    # Zabezpieczenie na wypadek podania całkowicie błędnych nazw
    if not MODELS:
        print("[SYSTEM WARN] Nie rozpoznano poprawnych modeli. Uruchamiam wszystkie domyślne.")
        MODELS = AVAILABLE_MODELS
# ---------------------------------------------

# --- NOWY BLOK: WYBÓR OKRESÓW DO TRENOWANIA ---
max_period = len(GLOBAL_DATES) - 1
raw_periods_input = input(f"Podaj okresy do trenowania (np. '0-{max_period}', '0,2' lub Enter dla wszystkich): ").strip()

selected_periods = []
if not raw_periods_input:
    selected_periods = list(range(max_period + 1))
elif "-" in raw_periods_input:
    try:
        start_id, end_id = map(int, raw_periods_input.split("-"))
        selected_periods = list(range(start_id, end_id + 1))
    except:
        selected_periods = list(range(max_period + 1))
elif "," in raw_periods_input:
    try:
        selected_periods = [int(x.strip()) for x in raw_periods_input.split(",")]
    except:
        selected_periods = list(range(max_period + 1))
else:
    try:
        selected_periods = [int(raw_periods_input)]
    except:
        selected_periods = list(range(max_period + 1))

# Oczyszczenie z wartości spoza dostępnego zakresu (np. jeśli użytkownik wpisze 99)
selected_periods = [p for p in selected_periods if 0 <= p <= max_period]
if not selected_periods:
    selected_periods = list(range(max_period + 1))
# ---------------------------------------------

print(f"[SYSTEM] Aktywne modele: {MODELS}")
print(f"[SYSTEM] Zaplanowane RUN_ID do wykonania: {run_ids}")
print(f"[SYSTEM] Zaplanowane OKRESY (Periods) do wykonania: {selected_periods}")

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# --- DODANE: Wczytanie wyników Baseline z CSV (z kodu 2) ---
import pandas as pd # Upewnij się, że pandas jest zaimportowane na górze skryptu
baseline_metrics = {}
baseline_results_path = os.path.join(os.getcwd(), "Default_Hyperparams_Evaluation", "default_baselines_results.csv")

if os.path.exists(baseline_results_path):
    print("[SYSTEM] Wczytuję metryki bazowe (Baseline) dla domyślnych hiperparametrów...")
    df_base = pd.read_csv(baseline_results_path)
    for _, row in df_base.iterrows():
        period_idx = int(row["Period"])
        m_name = row["Model"].lower()
        metric_val = float(row["Median_minus_Std"])
        
        if period_idx not in baseline_metrics:
            baseline_metrics[period_idx] = {}
        baseline_metrics[period_idx][m_name] = metric_val
else:
    print("[SYSTEM WARN] Nie znaleziono pliku default_baselines_results.csv. Baseline będzie traktowany jako -nieskończoność.")
# -----------------------------------------------------------

# ==============================================================================
# GŁÓWNA PĘTLA PO WSZYSTKICH RUN_ID
# ==============================================================================
for CURRENT_RUN_ID in run_ids:
    print(f"\n{'='*60}\n[SYSTEM] ROZPOCZYNAM RUN_ID: {CURRENT_RUN_ID}\n{'='*60}")
    
    current_seed = 42 + CURRENT_RUN_ID
    set_seeds(current_seed)

    RUN_DIR = os.path.join(OUTPUT_MODELS_DIR if 'OUTPUT_MODELS_DIR' in locals() else os.getcwd(), f"run_{CURRENT_RUN_ID}")
    if not os.path.exists(RUN_DIR): os.makedirs(RUN_DIR)

    # --- 4. GŁÓWNA PĘTLA PO OKNACH CZASOWYCH (WALK-FORWARD) ---
    for i, test_start in enumerate(GLOBAL_DATES):
        
        # --- DODANE: Pomijanie niewybranych okresów ---
        if i not in selected_periods:
            continue
        
        # --- DODANE: Dynamiczne wskaźniki i wymiar środowiska ---
        period_key = f"okres_{i}"
        indicators = indicators_dict.get(period_key, indicators_dict.get("okres_0", []))
        state_dim = 1 + 2 * stock_dim + len(indicators) * stock_dim
        print(f"\n[SYSTEM] Rozpoczynam Period {i} | Wskaźniki: {len(indicators)} | State Dim: {state_dim}")

        # === INTELIGENTNE ŁADOWANIE PARAMETRÓW OPTUNY ===
        optuna_best_params = {}
        base_dir = OUTPUT_MODELS_DIR if 'OUTPUT_MODELS_DIR' in locals() else os.getcwd()
        
        tuning_dir_current = os.path.join(base_dir, f"Optuna_Tuning_Period{i}")
        json_path_current = os.path.join(tuning_dir_current, f"best_hyperparameters_continuous_period{i}.json")
        
        tuning_dir_fallback = os.path.join(base_dir, "Optuna_Tuning_Period0")
        json_path_fallback = os.path.join(tuning_dir_fallback, "best_hyperparameters_continuous_period0.json")

        # Krok 1: Wczytanie zoptymalizowanego pliku dla aktualnie przerabianego okresu
        if os.path.exists(json_path_current):
            with open(json_path_current, 'r') as f: optuna_best_params = json.load(f)
            print(f"\n[OPTUNA] Znaleziono zoptymalizowane parametry dla Period {i}.")
            
        # Krok 2: Wczytanie pojedynczych wyciągów dla bieżącego okresu
        elif os.path.exists(tuning_dir_current):
            found_any = False
            for m_name in MODELS:
                ind_path = os.path.join(tuning_dir_current, m_name, f"best_{m_name}_period{i}.json")
                if os.path.exists(ind_path):
                    with open(ind_path, 'r') as f: optuna_best_params[m_name] = json.load(f)
                    found_any = True
            if found_any: print(f"\n[OPTUNA] Wczytano indywidualne parametry z folderów dla Period {i}.")
        
        # Krok 3: Fallback – Wczytanie wyników uzyskanych dla Okresu '0' (jeśli brak innych)
        if not optuna_best_params and os.path.exists(json_path_fallback):
            with open(json_path_fallback, 'r') as f: optuna_best_params = json.load(f)
            print(f"\n[OPTUNA WARN] Brak parametrów dla Period {i}. Wczytano FALLBACK z Period 0.")
            
        # Krok 4: Fallback – Indywidualne parametry dla Okresu '0'
        if not optuna_best_params and os.path.exists(tuning_dir_fallback):
            found_any = False
            for m_name in MODELS:
                ind_path = os.path.join(tuning_dir_fallback, m_name, f"best_{m_name}_period0.json")
                if os.path.exists(ind_path):
                    with open(ind_path, 'r') as f: optuna_best_params[m_name] = json.load(f)
                    found_any = True
            if found_any: print(f"\n[OPTUNA WARN] Brak parametrów dla Period {i}. Wczytano indywidualne jako FALLBACK z Period 0.")
            
        if not optuna_best_params:
            print(f"\n[OPTUNA ERROR] Całkowity brak parametrów! Użyte zostaną wartości DOMYŚLNE dla Period {i}.")
            
        # --- DODANE: Pobranie wartości metryki celu z CSV Optuny (z kodu 1) ---
        optuna_metrics = {}
        for m_name in MODELS:
            csv_optuna_path = os.path.join(tuning_dir_current, m_name, f"trials_{m_name}_period{i}.csv")
            optuna_metrics[m_name] = -float('inf') # Inicjalizacja bardzo słabym wynikiem
            
            if os.path.exists(csv_optuna_path):
                try:
                    df_opt = pd.read_csv(csv_optuna_path)
                    completed_trials = df_opt[df_opt["Status"] == "COMPLETE"]
                    if not completed_trials.empty:
                        # Wyciągamy maksymalną wartość z kolumny metryki celu
                        optuna_metrics[m_name] = completed_trials["Median_Sortino_std_penalty"].max()
                except Exception as e:
                    print(f"[OPTUNA ERROR] Błąd parsowania CSV dla {m_name.upper()} w Period {i}: {e}")
        # =========================================================

        train_start = test_start - relativedelta(years=TRAIN_YEARS)
        train_end = test_start - relativedelta(months=VAL_MONTHS)
        val_start = train_end
        
        train = df_global[(df_global.date >= train_start) & (df_global.date < train_end)]
        val = df_global[(df_global.date >= val_start) & (df_global.date < test_start)]

        # Konwersja wektora dat na wymiar indeksacyjny dla sieci neuronowych
        train.index = train.date.factorize()[0]
        val.index = val.date.factorize()[0]
        
        unique_tickers = sorted(FILES_CONFIG.keys()) 
        target_exposure = INITIAL_CAPITAL * MAX_POSITION_PCT
        hmax_list = []
        
        print(f"\n[KALIBRACJA] Obliczanie limitów HMAX dla Periodu {i}...")
        
        # Budowa wektora HMAX dla każdej poszczególnej spółki oparta o jej szczyt notowań.
        for tic in unique_tickers:
            tic_data = train[train.tic == tic]
            if tic_data.empty: max_price = 100.0
            else: max_price = tic_data['close'].max()
            if pd.isna(max_price) or max_price <= 0: max_price = 1.0
            
            limit = target_exposure / (max_price * (1 + COST))
            limit = round(float(limit), 4)
            hmax_list.append(limit)
            
            if tic in unique_tickers[:3]:
                print(f"   -> {tic:12}: Cena Ref ~{max_price:7.2f} PLN | HMAX = {limit} szt.")

        # Inicjalizacja hiperparametrów wejściowych dla obiektów fizycznych środowisk giełdy
        env_kwargs = {
            "hmax": hmax_list, "initial_amount": INITIAL_CAPITAL, "buy_cost_pct": [COST]*stock_dim, 
            "sell_cost_pct": [COST]*stock_dim, "state_space": state_dim, "stock_dim": stock_dim, 
            "tech_indicator_list": indicators, "action_space": stock_dim, "reward_scaling": REWARD_SCALING, "reward_type": "log", "risk_penalty": 0.1,  
            "num_stock_shares": [0]*stock_dim, "risk_indicator_col": "turbulence"
        }
        
        print(f"--- Period {i}: Train len={len(train)}, Val len={len(val)} ---")
        
        # Utworzenie instancji walidacyjnej nadzorowanej przez dynamiczny wektor normujący (VecNormalize)
        eval_env = DummyVecEnv([lambda: StockTradingEnvCustom(df=val, **{**env_kwargs, "turbulence_threshold": None})])
        eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.)

        steps = int(100 * len(train.index.unique()))

        # Słownik wbudowanych konfiguracji domyślnych (wywoływany gdy brak strojenia)
        model_params = {
            "ppo": {"n_steps": 2048, "ent_coef": 0.01, "learning_rate": 0.00025, "batch_size": 128, "policy_kwargs": dict(net_arch=dict(pi=[128, 128], vf=[128, 128]), optimizer_kwargs=dict(weight_decay=0.0))},
            "a2c": {"ent_coef": 0.01, "learning_rate": 0.0007, "n_steps": 5, "policy_kwargs": dict(net_arch=dict(pi=[128, 128], vf=[128, 128]), optimizer_kwargs=dict(weight_decay=0.0))},
            "ddpg": {"learning_starts": steps//10, "buffer_size": 100000, "batch_size": 256, "learning_rate": 0.0001, "policy_kwargs": dict(net_arch=dict(pi=[256, 256], qf=[256, 256]), optimizer_kwargs=dict(weight_decay=0.0))},
            "td3": {"learning_starts": steps//10, "buffer_size": 100000, "batch_size": 256, "learning_rate": 0.0003, "policy_delay": 2, "policy_kwargs": dict(net_arch=dict(pi=[256, 256], qf=[256, 256]), optimizer_kwargs=dict(weight_decay=0.0))},
            "sac": {"learning_starts": steps//10, "buffer_size": 100000, "batch_size": 256, "learning_rate": 0.0003, "ent_coef": "auto", "policy_kwargs": dict(net_arch=dict(pi=[256, 256], qf=[256, 256]), optimizer_kwargs=dict(weight_decay=0.0))}
        }

        # Uczenie konkretnych architektur dla danego okna
        for name in MODELS:
            save_path = os.path.join(RUN_DIR, f"period_{i}", name)
            os.makedirs(save_path, exist_ok=True)
            os.makedirs(os.path.join(save_path, "train_log"), exist_ok=True)
            done_flag = os.path.join(save_path, "done.flag")

            # Flaga pomijająca wyuczone wcześniej modele
            if not os.path.exists(done_flag):
                eval_freq = steps//100

                try:
                    monitor_path = os.path.join(save_path, "train_log", "monitor.csv")

                    def make_train_env():
                        e = StockTradingEnvCustom(df=train, **{**env_kwargs, "turbulence_threshold": None})
                        # Zabezpieczenie przed płaską wariancją w fazie learning_starts!
                        e.action_space.seed(current_seed) 
                        return Monitor(e, filename=monitor_path)

                    env_train = DummyVecEnv([make_train_env])
                    env_train = VecNormalize(env_train, norm_obs=True, norm_reward=True, clip_obs=10., gamma=0.99)
                    env_train.seed(current_seed)
                    
                    default_kwargs = model_params[name].copy()

                    # Oszacowanie długości pojedynczego epizodu
                    ep_length = len(train.index.unique())

                    # Przekazanie jej do parsera, aby zreplikować logikę PPO z Optuny
                    tuned_kwargs = build_tuned_kwargs(name, optuna_best_params, stock_dim, steps, ep_length)

                    # --- DODANE: Pojedynek parametrów Baseline vs Optuna ---
                    b_metric = baseline_metrics.get(i, {}).get(name, -float('inf'))
                    o_metric = optuna_metrics.get(name, -float('inf'))

                    print(f"\n[{name.upper()}] DECYZJA O HIPERPARAMETRACH (Period {i}):")
                    print(f"  -> Tuning Optuna (Mediana - Std): {o_metric:.4f}")
                    print(f"  -> Domyślne SB3  (Mediana - Std): {b_metric:.4f}")

                    if tuned_kwargs is not None and (o_metric > b_metric or b_metric == -float('inf')):
                        print(f"  => WYGRYWA TUNING. Używam hiperparametrów zoptymalizowanych Optuną.")
                        k = tuned_kwargs
                    else:
                        if tuned_kwargs is None:
                            print(f"  => WYGRYWA DOMYŚLNY SB3 (brak poprawnego pliku JSON z Optuny).")
                        else:
                            print(f"  => WYGRYWA DOMYŚLNY SB3 (parametry domyślne dały lepszy wynik na walidacji).")
                        k = default_kwargs
                    # -------------------------------------------------------
                    
                    p = k.pop('policy_kwargs', None)
                    current_seed = 42 + CURRENT_RUN_ID

                    set_seeds(current_seed)

                    # Eliminacja błędów logiki dewiacji standardowej biblioteki (SDE wyłączone)
                    if name in ['ddpg','td3'] and p: p.pop('use_sde', None)
                    k.update({"device": "cpu"})
                    
                    SB3_MODELS = {"ppo": PPO, "a2c": A2C, "ddpg": DDPG, "td3": TD3, "sac": SAC}
                    
                    # Budowa instancji agenta na wyciągniętych obiektach parametryzujących (sieć, szum, optim)
                    model = SB3_MODELS[name](
                        policy="MlpPolicy",
                        env=env_train,
                        policy_kwargs=p,
                        seed=current_seed,
                        **k
                    )
                    
                    sortino_callback = SortinoEvalCallback(
                        eval_env=eval_env, 
                        eval_freq=eval_freq, 
                        log_path=save_path, 
                        best_model_save_path=save_path,
                        initial_capital=INITIAL_CAPITAL,
                        reward_scaling=REWARD_SCALING, 
                        verbose=0,
                        normalization_source=env_train  
                    )

                    print(f"Start treningu {name.upper()} (VecNormalize, steps: {steps})...")
                    model.learn(total_timesteps=steps, tb_log_name=name, callback=sortino_callback)
                    
                    with open(done_flag, "w") as f: f.write("DONE")
                    
                except Exception as e:
                    print(f"Error {name}: {e}")
                    import traceback
                    traceback.print_exc()

        try:
            ep_length = len(train.index.unique())
            plot_learning_process(os.path.join(RUN_DIR, f"period_{i}"), i, MODELS, ep_length)
            print(f"[PLOT] Wygenerowano wykres dla Period {i}")
        except Exception as e:
            print(f"[PLOT ERROR] {e}")

    print(f"[KONIEC] RUN {CURRENT_RUN_ID} zakończony.")

print("\n[SYSTEM] Wszystkie zaplanowane RUN_ID (z pętli) zostały zakończone.")

[SYSTEM] Global COST: 0.001 (Linear/Spread)
[SYSTEM] Mode: FRACTIONAL SHARES (Float)
[SYSTEM] Target Max Position: 100000 PLN per stock
[SYSTEM] Env Wrapper: Monitor -> StockTradingEnvCustom (XTB) -> DummyVecEnv -> VecNormalize

=== START TRENINGU - WALK-FORWARD WITH VEC_NORMALIZE ===


Podaj RUN_ID (np. pojedyncza liczba '0', przedział '0-3' lub lista '0,2,4') [0]:  78-83
Podaj modele do trenowania oddzielone przecinkiem ['ppo', 'a2c', 'ddpg', 'td3', 'sac'] [Enter = wszystkie]:  
Podaj okresy do trenowania (np. '0-3', '0,2' lub Enter dla wszystkich):  


[SYSTEM] Aktywne modele: ['ppo', 'a2c', 'ddpg', 'td3', 'sac']
[SYSTEM] Zaplanowane RUN_ID do wykonania: [78, 79, 80, 81, 82, 83]
[SYSTEM] Zaplanowane OKRESY (Periods) do wykonania: [0, 1, 2, 3]
[SYSTEM WARN] Nie znaleziono pliku default_baselines_results.csv. Baseline będzie traktowany jako -nieskończoność.

[SYSTEM] ROZPOCZYNAM RUN_ID: 78

[SYSTEM] Rozpoczynam Period 0 | Wskaźniki: 14 | State Dim: 257

[OPTUNA] Znaleziono zoptymalizowane parametry dla Period 0.

[KALIBRACJA] Obliczanie limitów HMAX dla Periodu 0...
   -> ALIOR_BANK  : Cena Ref ~  76.50 PLN | HMAX = 1305.9502 szt.
   -> CCC         : Cena Ref ~ 304.78 PLN | HMAX = 327.7777 szt.
   -> CD_PROJECT  : Cena Ref ~ 245.52 PLN | HMAX = 406.8985 szt.
--- Period 0: Train len=22976, Val len=960 ---

[PPO] DECYZJA O HIPERPARAMETRACH (Period 0):
  -> Tuning Optuna (Mediana - Std): 5.4775
  -> Domyślne SB3  (Mediana - Std): -inf
  => WYGRYWA TUNING. Używam hiperparametrów zoptymalizowanych Optuną.
Start treningu PPO (VecNormalize, s

### Symulacja testowa wielu pojedynczych przebiegów modeli A2C, PPO, DDPG, TD3, SAC
Na jej bazie rozkład przebiegu krzywych kapitału w czasie i metryk

In [9]:
# ==============================================================================
# 1. KONFIGURACJA I INPUTY
# ==============================================================================
INITIAL_CAPITAL = 100000.0
COST = 0.001
MAX_MONTHLY_TURNOVER = 400000.0
MIN_TRADE_VAL = 10.0
REWARD_SCALING = 100
MODEL_CLASSES = {"ppo": PPO, "a2c": A2C, "ddpg": DDPG, "td3": TD3, "sac": SAC}
MODELS_TO_BAG = ['ppo', 'a2c', 'ddpg', 'td3', 'sac']

PRICE_LOOKUP = df_global.set_index(['date', 'tic'])['close'].to_dict()
OPEN_LOOKUP = df_global.set_index(['date', 'tic'])['open'].to_dict() if 'open' in df_global.columns else PRICE_LOOKUP
NEXT_OPEN_LOOKUP = df_global.set_index(['date', 'tic'])['next_day_open'].to_dict() if 'next_day_open' in df_global.columns else PRICE_LOOKUP

print(f"Dostępne okresy: {list(QUARTERS_MAP.keys())}")
user_start = input("Początek (np. 'Q1 2020'): ").strip() or "Q1 2020"
user_end = input("Koniec (np. 'Q4 2020'): ").strip() or "Q4 2020"
start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

# Sztywno ustawiona wartość N
CURRENT_N = 1
NUM_BOOTSTRAP_ITERATIONS = int(input("\nIle razy zbudować każdy model (B)? [np. 100]: ") or 100)

SUMMARY_FILE = os.path.join(AGG_DIR, "wyniki_podsumowanie_statystyczne_N1.csv")
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_N1_B{NUM_BOOTSTRAP_ITERATIONS}.csv")

# ==============================================================================
# 2. KLASA ŚRODOWISKA I FUNKCJE POMOCNICZE
# ==============================================================================
class StockTradingEnvCustom(StockTradingEnv):
    def __init__(self, **kwargs):
        self.hmax = kwargs.get('hmax', 100)
        super().__init__(**kwargs)
    def step(self, actions):
        res = super().step(actions)
        if len(res) == 5: return res
        return res[0], res[1], res[2], False, res[3]

def calculate_sortino(df_log, target_return=0.0):
    if df_log is None or 'account_value' not in df_log.columns: return 0.0
    returns = df_log['account_value'].pct_change().dropna()
    if len(returns) < 2: return 0.0
    ann_ret = returns.mean() * 252
    downside = returns[returns < target_return]
    ann_std = downside.std() * np.sqrt(252) if len(downside) >= 2 else returns.std() * np.sqrt(252)
    return ann_ret / (ann_std + 1e-9)

def get_dynamic_hmax(current_cash, current_holdings, current_prices, ref_prices, cost_pct):
    stock_val = np.sum(np.array(current_holdings) * np.array(current_prices))
    account_val = current_cash + stock_val
    if account_val <= 0: return [0]*len(current_holdings), account_val
    target_exposure = account_val
    hmax_list = [int(target_exposure / (max(p, 1.0) * (1 + cost_pct))) for p in ref_prices]
    return hmax_list, account_val

# ==============================================================================
# 3. SILNIK SYMULACJI
# ==============================================================================
def run_simulation_engine_aggregated(vec_env, model_or_list, 
                                  initial_cash=100000, 
                                  initial_holdings=None, initial_turnover=0.0, initial_month_str="",
                                  weights=None): 
    obs = vec_env.reset()
    inner_env = vec_env.venv.envs[0] if isinstance(vec_env, VecNormalize) else vec_env.envs[0]
    unique_tickers = sorted(FILES_CONFIG.keys())
    stock_dim = len(unique_tickers)
    
    current_cash = float(initial_cash)
    current_holdings = np.array(initial_holdings, dtype=float) if initial_holdings is not None else np.zeros(stock_dim, dtype=float)
    pending_orders = np.zeros(stock_dim, dtype=float)
    current_month_str = str(initial_month_str)
    accumulated_monthly_turnover = initial_turnover

    logs = {'date': [], 'account_value': []}
    days_list = inner_env.df.index.unique()
    
    for i in range(len(days_list)):
        current_date = pd.to_datetime(inner_env.df.iloc[i * stock_dim]['date']).normalize()
        
        # Reset obrotu na początku nowego miesiąca
        if current_date.strftime("%Y-%m") != current_month_str:
            accumulated_monthly_turnover = 0.0
            current_month_str = current_date.strftime("%Y-%m")

        # --- EGZEKUCJA ZLECEŃ Z POPRZEDNIEGO DNIA (PORANEK DNIA T) ---
        if i > 0:
            open_prices = np.array([OPEN_LOOKUP.get((current_date, t), 0.0) for t in unique_tickers])
            
            # 1. Kalkulacja planowanego obrotu dla porannych zleceń
            planned_day_turnover = 0.0
            for j in range(stock_dim):
                if open_prices[j] <= 0 or pending_orders[j] == 0: continue
                order = pending_orders[j]
                if order < 0:
                    qty = max(order, -current_holdings[j])
                    planned_day_turnover += abs(qty) * open_prices[j]
                elif order > 0:
                    cost = order * open_prices[j] * (1 + COST)
                    if cost > current_cash: 
                        order = current_cash / (open_prices[j] * (1 + COST))
                    planned_day_turnover += order * open_prices[j]

            # 2. Proporcjonalne skalowanie w dół, jeśli grozi nam przekroczenie 400k PLN darmowego limitu
            turnover_scale = 1.0
            if accumulated_monthly_turnover + planned_day_turnover > MAX_MONTHLY_TURNOVER:
                allowed_turnover = MAX_MONTHLY_TURNOVER - accumulated_monthly_turnover
                if allowed_turnover > 0 and planned_day_turnover > 0:
                    turnover_scale = allowed_turnover / planned_day_turnover
                else:
                    turnover_scale = 0.0  # Całkowita blokada handlu w tym miesiącu

            # 3. Właściwa realizacja zleceń z uwzględnieniem wyznaczonego skalowania obrotu
            for j in range(stock_dim):
                if open_prices[j] <= 0 or pending_orders[j] == 0: continue
                order = pending_orders[j] * turnover_scale  # Skalowanie anty-prowizyjne
                
                if order < 0:
                    qty = max(order, -current_holdings[j])
                    trade_val = abs(qty) * open_prices[j]
                    current_cash += trade_val * (1 - COST)
                    current_holdings[j] += qty
                    accumulated_monthly_turnover += trade_val  # Aktualizacja licznika obrotu
                elif order > 0:
                    cost = order * open_prices[j] * (1 + COST)
                    if cost > current_cash: 
                        order = current_cash / (open_prices[j] * (1 + COST))
                    trade_val = order * open_prices[j]
                    current_cash -= trade_val * (1 + COST)
                    current_holdings[j] += order
                    accumulated_monthly_turnover += trade_val  # Aktualizacja licznika obrotu

        # --- WYCENA PORTFELA NA KONIEC DNIA T I GENEROWANIE SYGNAŁÓW ---
        close_prices = np.array([PRICE_LOOKUP.get((current_date, t), 0.0) for t in unique_tickers])
        account_value = current_cash + np.sum(current_holdings * close_prices)
        
        acts = []
        for m_obj, v_norm in model_or_list:
            a, _ = m_obj.predict(v_norm.normalize_obs(obs), deterministic=True)
            acts.append(a[0] if len(a.shape) > 1 else a)
        raw_act = np.average(acts, axis=0, weights=weights) if (weights is not None) else np.mean(acts, axis=0)
        target_action = np.round(raw_act * np.array(inner_env.hmax), 4)

        pending_orders = target_action
        logs['date'].append(current_date)
        logs['account_value'].append(account_value)
        vec_env.step([np.zeros(stock_dim)])

    return pd.DataFrame(logs), current_cash, current_holdings, accumulated_monthly_turnover, current_month_str
    
# ==============================================================================
# 4. GŁÓWNA PĘTLA ANALIZY (DLA N = 1)
# ==============================================================================

# Czyścimy stare pliki
master_summary_data = []
for f in [SUMMARY_FILE, TRACES_FILE]:
    if os.path.exists(f): os.remove(f)

unique_tickers = sorted(FILES_CONFIG.keys())
stock_dim = len(unique_tickers)
run_dirs = sorted(glob.glob(os.path.join(OUTPUT_MODELS_DIR, "run_*")))
valid_runs = [d for d in run_dirs if os.path.basename(d).split('_')[1].isdigit()]
periods_count = len(glob.glob(os.path.join(valid_runs[0], "period_*")))

print(f"\n\n{'='*35} START ANALIZY DLA N = 1 {'='*35}")

final_keys = [f"{m.upper()}_AVG" for m in MODELS_TO_BAG]

equity_traces = {k: [] for k in final_keys}
tournament_results = {k: [] for k in final_keys}

# Pre-generowanie kombinacji RUN_ID
pre_sampled_paths = {m: [] for m in MODELS_TO_BAG}
for m_name in MODELS_TO_BAG:
    possible = [r for r in valid_runs if os.path.exists(os.path.join(r, "period_0", m_name))]
    n_actual = min(CURRENT_N, len(possible))
    
    max_comb = math.comb(len(possible), n_actual)
    target_unique = min(NUM_BOOTSTRAP_ITERATIONS, max_comb)
    
    unique_combos = set()
    while len(unique_combos) < target_unique:
        unique_combos.add(tuple(sorted(random.sample(possible, n_actual))))
    
    combo_list = list(unique_combos)
    random.shuffle(combo_list)
    
    while len(combo_list) < NUM_BOOTSTRAP_ITERATIONS:
        combo_list.append(random.choice(combo_list[:target_unique]))
    pre_sampled_paths[m_name] = [list(c) for c in combo_list]

# Pętla iteracji (B)
for b_iter in range(NUM_BOOTSTRAP_ITERATIONS):
    print(f"🚀 N=1 | Postęp: {b_iter+1}/{NUM_BOOTSTRAP_ITERATIONS}    ", end='\r')
    
    sampled_paths = {m: pre_sampled_paths[m][b_iter] for m in MODELS_TO_BAG}
    iter_states = {k: {'cash': INITIAL_CAPITAL, 'holdings': [0.0]*stock_dim, 'to': 0.0, 'mo': ""} for k in final_keys}
    iter_logs = {k: [] for k in final_keys}

    for i in range(periods_count):
        if i < start_idx or i > end_idx: continue
        
        p_key = f"okres_{i}"
        inds = indicators_dict.get(p_key, indicators_dict.get("okres_0", []))
        s_dim = 1 + 2 * stock_dim + len(inds) * stock_dim
        test_df = df_global[(df_global.date >= GLOBAL_DATES[i]) & (df_global.date < GLOBAL_DATES[i] + relativedelta(months=TEST_MONTHS))].copy()
        test_df.index = test_df.date.factorize()[0]
        
        # HMAX
        train_start_calib = GLOBAL_DATES[i] - relativedelta(years=TRAIN_YEARS)
        df_calib = df_global[(df_global.date >= train_start_calib) & (df_global.date < GLOBAL_DATES[i])]
        ref_prices_per_ticker = [max(df_calib[df_calib.tic == t]['close'].max(), 1.0) if not df_calib[df_calib.tic == t].empty else 1.0 for t in unique_tickers]
        cur_spot = [PRICE_LOOKUP.get((GLOBAL_DATES[i], t), 1.0) for t in unique_tickers]

        period_params = {
            "stock_dim": stock_dim, "initial_amount": INITIAL_CAPITAL, "num_stock_shares": [0.0]*stock_dim,
            "buy_cost_pct": [COST]*stock_dim, "sell_cost_pct": [COST]*stock_dim, "reward_scaling": REWARD_SCALING,
            "action_space": stock_dim, "tech_indicator_list": inds, "state_space": s_dim, "df": test_df, "risk_indicator_col": "turbulence"
        }

        loaded_by_type = {m: [] for m in MODELS_TO_BAG}

        for m_name in MODELS_TO_BAG:
            dummy_v = DummyVecEnv([lambda: StockTradingEnvCustom(**{**period_params, "hmax": 100})])
            for r_path in sampled_paths[m_name]:
                p_mod = os.path.join(r_path, f"period_{i}", m_name, "best_model.zip")
                p_vec = os.path.join(r_path, f"period_{i}", m_name, "best_model_vecnorm.pkl")
                if os.path.exists(p_mod) and os.path.exists(p_vec):
                    try:
                        m_obj = MODEL_CLASSES[m_name].load(p_mod, device='cpu')
                        v_obj = VecNormalize.load(p_vec, dummy_v)
                        v_obj.training = False; v_obj.norm_reward = False
                        loaded_by_type[m_name].append((m_obj, v_obj))
                    except: continue

        def execute_sim(key, agents, weights=None):
            st = iter_states[key]
            d_hmax, _ = get_dynamic_hmax(st['cash'], st['holdings'], cur_spot, ref_prices_per_ticker, COST)
            sim_p = {**period_params, "num_stock_shares": list(st['holdings']), "initial_amount": st['cash'], "hmax": d_hmax}
            v = DummyVecEnv([lambda: StockTradingEnvCustom(**sim_p)])
            res_df, f_c, f_h, t_o, m_o = run_simulation_engine_aggregated(v, agents, st['cash'], st['holdings'], st['to'], st['mo'], weights=weights)
            iter_logs[key].append(res_df)
            iter_states[key].update({'cash': f_c, 'holdings': f_h, 'to': t_o, 'mo': m_o})

        # Tylko proste uśrednianie wewnątrz każdej klasy algorytmu
        for m_name in MODELS_TO_BAG:
            if loaded_by_type[m_name]: execute_sim(f"{m_name.upper()}_AVG", loaded_by_type[m_name])

    # Zbieranie wyników po zakończeniu roku
    for k in final_keys:
        if iter_logs[k]:
            full_df = pd.concat(iter_logs[k]).drop_duplicates('date').sort_values('date')
            tournament_results[k].append(calculate_sortino(full_df))
            equity_traces[k].append(full_df[['date', 'account_value']].copy())

# ZAPIS DANYCH DLA N = 1
print(f"\n💾 Zapisywanie danych do CSV dla N=1...")
for k, values in tournament_results.items():
    if values:
        master_summary_data.append({
            'Model': k, 'N': CURRENT_N,
            'Mean_Sortino': np.mean(values), 'Median_Sortino': np.median(values),
            'Std_Sortino': np.std(values), 'Min': np.min(values), 'Max': np.max(values)
        })

all_traces_flat = []
for model_name, traces_list in equity_traces.items():
    for iteration_idx, df_trace in enumerate(traces_list):
        temp_df = df_trace.copy()
        temp_df['Model'] = model_name
        temp_df['N'] = CURRENT_N
        temp_df['Iteration'] = iteration_idx + 1
        all_traces_flat.append(temp_df[['Model', 'N', 'Iteration', 'date', 'account_value']])

if all_traces_flat:
    df_chunk = pd.concat(all_traces_flat, ignore_index=True)
    write_header = not os.path.exists(TRACES_FILE)
    df_chunk.to_csv(TRACES_FILE, mode='a', index=False, sep=';', decimal=',', header=write_header)

del equity_traces
del tournament_results

# ==============================================================================
# 5. FINALIZACJA
# ==============================================================================
df_master_summary = pd.DataFrame(master_summary_data)
df_master_summary.to_csv(SUMMARY_FILE, index=False, sep=';', decimal=',')

print(f"\n{'='*50}\n✅ PROCES ZAKOŃCZONY POMYŚLNIE")
print(f"📊 Wyniki zapisane w: {SUMMARY_FILE}")
print(f"📈 Ścieżki kapitału zapisane w: {TRACES_FILE}")

Dostępne okresy: ['Q1 2020', 'Q2 2020', 'Q3 2020', 'Q4 2020']


Początek (np. 'Q1 2020'):  
Koniec (np. 'Q4 2020'):  

Ile razy zbudować każdy model (B)? [np. 100]:  100




=================================== START ANALIZY DLA N = 1 ===================================
🚀 N=1 | Postęp: 100/100    
💾 Zapisywanie danych do CSV dla N=1...

✅ PROCES ZAKOŃCZONY POMYŚLNIE
📊 Wyniki zapisane w: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/wyniki_podsumowanie_statystyczne_N1.csv
📈 Ścieżki kapitału zapisane w: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/szczegoly_account_value_N1_B100.csv


In [10]:
# --- 1. KONFIGURACJA I PARAMETRY ---

MODEL_COLORS = {
    'A2C': '#2CA02C', 'PPO': '#DAA520', 'DDPG': '#D62728', 
    'TD3': '#1F77B4', 'SAC': '#9467BD'
}
MODELS_TO_PLOT = ['A2C_AVG', 'PPO_AVG', 'DDPG_AVG', 'TD3_AVG', 'SAC_AVG']

liczba = int(input("Ile było iteracji dla każdego modelu?: [100]") or 100)

# Ścieżki do plików
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_N1_B{liczba}.csv")

# --- 2. FUNKCJE DO OBLICZANIA METRYK ---

def calculate_metrics(series):
    """Oblicza zestaw metryk dla pojedynczej krzywej kapitału."""
    returns = series.pct_change().dropna()
    if len(returns) < 2:
        return [0] * 5
    
    # 1. Skumulowana stopa zwrotu
    cum_ret = (series.iloc[-1] / series.iloc[0]) - 1
    
    # 2. Sharpe Ratio (Roczny, rf=0)
    sharpe = (returns.mean() / (returns.std() + 1e-9)) * np.sqrt(252)
    
    # 3. Sortino Ratio
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252) if len(downside_returns) >= 2 else returns.std() * np.sqrt(252)
    sortino = (returns.mean() * 252) / (downside_std + 1e-9)
    
    # 4. Max Drawdown
    cum_max = series.cummax()
    drawdown = (series - cum_max) / cum_max
    max_dd = drawdown.min()
    
    # 5. Calmar Ratio
    calmar = (returns.mean() * 252) / (abs(max_dd) + 1e-9)
    
    return [cum_ret, sharpe, sortino, max_dd, calmar]

# --- 3. WCZYTYWANIE DANYCH ---

print("⏳ Wczytywanie danych zbiorczych... Proszę czekać.")
if not os.path.exists(TRACES_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku {TRACES_FILE}")
    exit()

df_all = pd.read_csv(TRACES_FILE, sep=';', decimal=',')
df_all['date'] = pd.to_datetime(df_all['date'])

# Filtrowanie tylko dla N=1
df_n1 = df_all[df_all['N'] == 1].copy()

# --- 4. PRZYGOTOWANIE SIATKI WYKRESÓW ---

fig = plt.figure(figsize=(18, 16), facecolor='white')
gs = GridSpec(3, 4, figure=fig)
ax_base = fig.add_subplot(gs[0, 0:2]) 
axes = [
    ax_base,
    fig.add_subplot(gs[0, 2:4], sharex=ax_base, sharey=ax_base),
    fig.add_subplot(gs[1, 0:2], sharex=ax_base, sharey=ax_base),
    fig.add_subplot(gs[1, 2:4], sharex=ax_base, sharey=ax_base),
    fig.add_subplot(gs[2, 1:3], sharex=ax_base, sharey=ax_base)
]

all_metrics_storage = []

# --- 5. PĘTLA ANALITYCZNA ---

for idx, model_key in enumerate(MODELS_TO_PLOT):
    ax = axes[idx]
    base_name = model_key.replace('_AVG', '')
    
    # POPRAWKA 1: Wyciąganie dedykowanego #E377C2u dla danego modelu!
    current_color = MODEL_COLORS.get(base_name, '#000000') 
    
    print(f"📊 Przetwarzanie modelu: {base_name} (N=1)...")
    
    model_data = df_n1[df_n1['Model'] == model_key]
    iterations = model_data['Iteration'].unique()
    
    model_iteration_series = [] 
    iteration_metrics = []      
    runs_for_median = []        

    for it in iterations:
        it_df = model_data[model_data['Iteration'] == it].sort_values('date')
        it_series = it_df.set_index('date')['account_value']
        
        if it_series.empty: continue
            
        norm_series = (it_series / it_series.iloc[0]) - 1
        model_iteration_series.append(norm_series)
        
        metrics = calculate_metrics(it_series)
        iteration_metrics.append(metrics)
        
        # Zapamiętanie krzywej wraz ze wskaźnikiem Sortino (indeks 2)
        runs_for_median.append({
            'iteration': it,
            'sortino': metrics[2], 
            'curve': norm_series
        })

    if not model_iteration_series:
        ax.set_title(f"{base_name} (Brak Danych)", fontsize=14, color='red')
        ax.axis('off')
        continue

    # POPRAWKA 2 (Zatwierdzona): Szukanie medianowego przebiegu wg Sortino
    runs_for_median.sort(key=lambda x: x['sortino'])
    n_runs = len(runs_for_median)
    
    if n_runs % 2 != 0:
        mid_idx = n_runs // 2
        median_curve = runs_for_median[mid_idx]['curve']
    else:
        mid1 = n_runs // 2 - 1
        mid2 = n_runs // 2
        curve1 = runs_for_median[mid1]['curve']
        curve2 = runs_for_median[mid2]['curve']
        median_curve = (curve1 + curve2) / 2.0

    # Rozrzut całkowity (Min-Max)
    combined_df = pd.concat(model_iteration_series, axis=1)
    min_curve = combined_df.min(axis=1)
    max_curve = combined_df.max(axis=1)
    
    # --- RYSOWANIE ---
    dates = combined_df.index
    
    # POPRAWKA 3: Użycie `current_color` zamiast BRAND_COLOR
    ax.fill_between(dates, min_curve, max_curve, 
                    color=current_color, alpha=0.2, zorder=2)
    
    ax.plot(dates, median_curve, color=current_color, linewidth=2.5, zorder=5)
    
    ax.set_title(f"{base_name}", fontsize=24, fontweight='bold', color=current_color)
    ax.set_ylabel("Skumulowana stopa zwrotu (%)", fontsize=12)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, which='major', linestyle='--', alpha=0.5)
    ax.axhline(0, color='black', linewidth=1, alpha=0.3)

    # --- AGREGACJA METRYK ---
    metrics_df = pd.DataFrame(iteration_metrics, columns=['Ret', 'Sharpe', 'Sortino', 'MDD', 'Calmar'])
    for col in metrics_df.columns:
        all_metrics_storage.append({
            'Model': base_name,
            'Metryka': col,
            'Średnia': metrics_df[col].mean(),
            'Mediana': metrics_df[col].median(),
            'Min': metrics_df[col].min(),
            'Max': metrics_df[col].max(),
            'Std': metrics_df[col].std()
        })

# --- 6. KOŃCOWE FORMATOWANIE WYKRESU I ZAPIS ---

plt.tight_layout(rect=[0, 0.08, 1, 0.98])
custom_legend_elements = [
    Patch(facecolor='gray', alpha=0.3, label='Rozpiętość wyników'),
    Line2D([0], [0], color='black', lw=2.5, label='Przebieg medianowy (wg wskaźnika Sortino)')
]
fig.legend(handles=custom_legend_elements, loc='lower center', 
           ncol=2, fontsize=18, bbox_to_anchor=(0.5, 0.02), 
           frameon=True, shadow=True)

plot_filename = f"Zestawienie_Modeli_N1_B{liczba}.png"
save_path = os.path.join(AGG_DIR, plot_filename)

plt.savefig(save_path, dpi=300, bbox_inches='tight') 
print(f"✅ Wykres został zapisany w: {save_path}")

# --- 7. TWORZENIE I WYŚWIETLANIE TABELI METRYK ---

final_table = pd.DataFrame(all_metrics_storage)
metric_name_map = {
    'Ret': 'Skumulowany Zwrot',
    'Sharpe': 'Wskaźnik Sharpe',
    'Sortino': 'Wskaźnik Sortino',
    'MDD': 'Max Drawdown',
    'Calmar': 'Wskaźnik Calmar'
}
final_table['Metryka'] = final_table['Metryka'].map(metric_name_map)

pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print("\n" + "="*100)
print("📈 TABELA METRYK PORÓWNAWCZYCH (N=1, Rozkład Bootstrap)")
print("="*100)
print(final_table.pivot(index=['Model', 'Metryka'], columns=[]).sort_index())

final_table.to_csv(os.path.join(AGG_DIR, "Raport_Metryk_N1.csv"), sep=';', decimal=',', index=False)

Ile było iteracji dla każdego modelu?: [100] 100


⏳ Wczytywanie danych zbiorczych... Proszę czekać.
📊 Przetwarzanie modelu: A2C (N=1)...
📊 Przetwarzanie modelu: PPO (N=1)...
📊 Przetwarzanie modelu: DDPG (N=1)...
📊 Przetwarzanie modelu: TD3 (N=1)...
📊 Przetwarzanie modelu: SAC (N=1)...
✅ Wykres został zapisany w: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Zestawienie_Modeli_N1_B100.png

📈 TABELA METRYK PORÓWNAWCZYCH (N=1, Rozkład Bootstrap)
                         Średnia  Mediana     Min     Max    Std
Model Metryka                                                   
A2C   Max Drawdown       -0.4170  -0.4096 -0.5737 -0.3014 0.0616
      Skumulowany Zwrot   0.1715   0.0803 -0.3322  1.4824 0.3749
      Wskaźnik Calmar     0.5845   0.4172 -0.6495  2.9646 0.7749
      Wskaźnik Sharpe     0.4667   0.3965 -0.7304  2.1478 0.6340
      Wskaźnik Sortino    0.6779   0.5475 -1.0088  3.1023 0.9166
DDPG  Max Drawdown       -0.5294  -0.5349 -0.7016 -0.3159 0.0925
      Skumulowany Zwrot  -0.0106  -0.09

### Symulacja testowa wielu przebiegów dla komitetów
Różne liczby agentów głosujacych w komitetach, każdy komitet w określonej liczebości agentów tworzony wielokrotnie (bazujac na próbach bootstrapowych agentów)

In [6]:
# ==============================================================================
# 1. KONFIGURACJA I INPUTY
# ==============================================================================
INITIAL_CAPITAL = 100000.0
COST = 0.001
MAX_MONTHLY_TURNOVER = 400000.0
MIN_TRADE_VAL = 10.0
REWARD_SCALING = 100
MODEL_CLASSES = {"ppo": PPO, "a2c": A2C, "ddpg": DDPG, "td3": TD3, "sac": SAC}
MODELS_TO_BAG = ['ppo', 'a2c', 'ddpg', 'td3', 'sac']

PRICE_LOOKUP = df_global.set_index(['date', 'tic'])['close'].to_dict()
OPEN_LOOKUP = df_global.set_index(['date', 'tic'])['open'].to_dict() if 'open' in df_global.columns else PRICE_LOOKUP
NEXT_OPEN_LOOKUP = df_global.set_index(['date', 'tic'])['next_day_open'].to_dict() if 'next_day_open' in df_global.columns else PRICE_LOOKUP

print(f"Dostępne okresy: {list(QUARTERS_MAP.keys())}")
user_start = input("Początek (np. 'Q1 2020'): ").strip() or "Q1 2020"
user_end = input("Koniec (np. 'Q4 2020'): ").strip() or "Q4 2020"
start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

NUM_BOOTSTRAP_ITERATIONS = int(input("\nPodaj liczbę powtórzeń Monte Carlo (B) dla każdego N [np. 30]: ") or 30)
max_n_to_check = int(input("Do jakiego maksymalnego N sprawdzić komitety? [np. 100]: ") or 100)

density_choice = input("Sposób testowania N: (C)iągły [1,2,3...], czy (R)zadki [1,2,5,10,20,50,100]? (C/r) [C]: ").strip().upper()

if density_choice == 'R':
    sparse_list = [1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
    N_VARIANTS = [n for n in sparse_list if n <= max_n_to_check]
else:
    N_VARIANTS = list(range(1, max_n_to_check + 1))

file_suffix = "_RZADKIE" if density_choice == 'R' else ""

SUMMARY_FILE = os.path.join(AGG_DIR, f"wyniki_podsumowanie_NASYCENIE_N1_{max_n_to_check}_B{NUM_BOOTSTRAP_ITERATIONS}{file_suffix}.csv")
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_NASYCENIE_N1_{max_n_to_check}_B{NUM_BOOTSTRAP_ITERATIONS}{file_suffix}.csv")

# [NOWE] Plik na surowe sygnały z sieci
SIGNALS_FILE = os.path.join(AGG_DIR, f"sygnaly_surowe_NASYCENIE_N1_{max_n_to_check}_B{NUM_BOOTSTRAP_ITERATIONS}{file_suffix}.csv")

METRIC_CHOICE = input("Metryka dla WIN/SMTH (median/iqm)? [iqm]: ").strip().lower() or "iqm"

ALL_COMBINATIONS = []
for r in range(2, 6):
    for combo in itertools.combinations(MODELS_TO_BAG, r):
        ALL_COMBINATIONS.append(sorted(list(combo)))

# ==============================================================================
# 2. KLASA ŚRODOWISKA I FUNKCJE POMOCNICZE
# ==============================================================================
class StockTradingEnvCustom(StockTradingEnv):
    def __init__(self, **kwargs):
        self.hmax = kwargs.get('hmax', 100)
        super().__init__(**kwargs)
    def step(self, actions):
        res = super().step(actions)
        if len(res) == 5: return res
        return res[0], res[1], res[2], False, res[3]

def calculate_iqm(metrics_list):
    if len(metrics_list) < 4: return np.median(metrics_list)
    q25, q75 = np.percentile(metrics_list, [25, 75])
    vals = [m for m in metrics_list if q25 <= m <= q75]
    return np.mean(vals) if vals else np.median(metrics_list)

def calculate_sortino(df_log, target_return=0.0):
    if df_log is None or 'account_value' not in df_log.columns: return 0.0
    returns = df_log['account_value'].pct_change().dropna()
    if len(returns) < 2: return 0.0
    ann_ret = returns.mean() * 252
    downside = returns[returns < target_return]
    ann_std = downside.std() * np.sqrt(252) if len(downside) >= 2 else returns.std() * np.sqrt(252)
    return ann_ret / (ann_std + 1e-9)

def get_dynamic_hmax(current_cash, current_holdings, current_prices, ref_prices, cost_pct):
    stock_val = np.sum(np.array(current_holdings) * np.array(current_prices))
    account_val = current_cash + stock_val
    if account_val <= 0: return [0]*len(current_holdings), account_val
    target_exposure = account_val
    hmax_list = [int(target_exposure / (max(p, 1.0) * (1 + cost_pct))) for p in ref_prices]
    return hmax_list, account_val

# ==============================================================================
# 3. SILNIK SYMULACJI
# ==============================================================================
def run_simulation_engine_aggregated(vec_env, model_or_list, 
                                  initial_cash=100000, 
                                  initial_holdings=None, initial_turnover=0.0, initial_month_str="",
                                  weights=None): 
    obs = vec_env.reset()
    inner_env = vec_env.venv.envs[0] if isinstance(vec_env, VecNormalize) else vec_env.envs[0]
    unique_tickers = sorted(FILES_CONFIG.keys())
    stock_dim = len(unique_tickers)
    
    current_cash = float(initial_cash)
    current_holdings = np.array(initial_holdings, dtype=float) if initial_holdings is not None else np.zeros(stock_dim, dtype=float)
    pending_orders = np.zeros(stock_dim, dtype=float)
    current_month_str = str(initial_month_str)
    accumulated_monthly_turnover = initial_turnover

    # [ZMIANA] Dodajemy 'raw_act' do zbierania logów
    logs = {'date': [], 'account_value': [], 'raw_act': []}
    days_list = inner_env.df.index.unique()
    
    for i in range(len(days_list)):
        current_date = pd.to_datetime(inner_env.df.iloc[i * stock_dim]['date']).normalize()
        
        # Reset limitu darmowego obrotu XTB na początku nowego miesiąca
        if current_date.strftime("%Y-%m") != current_month_str:
            accumulated_monthly_turnover = 0.0
            current_month_str = current_date.strftime("%Y-%m")

        if i > 0:
            open_prices = np.array([OPEN_LOOKUP.get((current_date, t), 0.0) for t in unique_tickers])
            
            # --- ETAP 1: Kalkulacja planowanego obrotu dla porannych zleceń ---
            planned_day_turnover = 0.0
            for j in range(stock_dim):
                if open_prices[j] <= 0 or pending_orders[j] == 0: continue
                order = pending_orders[j]
                if order < 0:
                    qty = max(order, -current_holdings[j])
                    planned_day_turnover += abs(qty) * open_prices[j]
                elif order > 0:
                    cost = order * open_prices[j] * (1 + COST)
                    if cost > current_cash: 
                        order = current_cash / (open_prices[j] * (1 + COST))
                    planned_day_turnover += order * open_prices[j]

            # --- ETAP 2: Proporcjonalne skalowanie (Obrona przed prowizją XTB) ---
            turnover_scale = 1.0
            if accumulated_monthly_turnover + planned_day_turnover > MAX_MONTHLY_TURNOVER:
                allowed_turnover = MAX_MONTHLY_TURNOVER - accumulated_monthly_turnover
                if allowed_turnover > 0 and planned_day_turnover > 0:
                    turnover_scale = allowed_turnover / planned_day_turnover
                else:
                    turnover_scale = 0.0  # Całkowita blokada handlu w tym miesiącu po przekroczeniu limitu

            # --- ETAP 3: Właściwa realizacja zleceń (z uwzględnieniem skalowania) ---
            for j in range(stock_dim):
                if open_prices[j] <= 0 or pending_orders[j] == 0: continue
                order = pending_orders[j] * turnover_scale  # Aplikacja mnożnika
                
                if order < 0:
                    qty = max(order, -current_holdings[j])
                    trade_val = abs(qty) * open_prices[j]
                    current_cash += trade_val * (1 - COST)
                    current_holdings[j] += qty
                    accumulated_monthly_turnover += trade_val  # Aktualizacja licznika obrotu XTB
                    
                elif order > 0:
                    cost = order * open_prices[j] * (1 + COST)
                    if cost > current_cash: 
                        order = current_cash / (open_prices[j] * (1 + COST))
                    trade_val = order * open_prices[j]
                    current_cash -= trade_val * (1 + COST)
                    current_holdings[j] += order
                    accumulated_monthly_turnover += trade_val  # Aktualizacja licznika obrotu XTB

        # Generowanie nowych sygnałów na koniec dnia
        close_prices = np.array([PRICE_LOOKUP.get((current_date, t), 0.0) for t in unique_tickers])
        account_value = current_cash + np.sum(current_holdings * close_prices)
        
        acts = []
        for m_obj, v_norm in model_or_list:
            a, _ = m_obj.predict(v_norm.normalize_obs(obs), deterministic=True)
            acts.append(a[0] if len(a.shape)>1 else a)
        raw_act = np.average(acts, axis=0, weights=weights) if (weights is not None) else np.mean(acts, axis=0)
        target_action = np.round(raw_act * np.array(inner_env.hmax), 4)

        pending_orders = target_action
        logs['date'].append(current_date)
        logs['account_value'].append(account_value)
        logs['raw_act'].append(raw_act.copy()) # Zapis surowego sygnału
        
        vec_env.step([np.zeros(stock_dim)])

    return pd.DataFrame(logs), current_cash, current_holdings, accumulated_monthly_turnover, current_month_str

# ==============================================================================
# 4. GŁÓWNA PĘTLA PO WARIANTACH N (NASYCENIE INFORMACYJNE)
# ==============================================================================

master_summary_data = []
# [ZMIANA] Czyścimy 3 pliki, wliczając nowy plik z sygnałami
for f in [SUMMARY_FILE, TRACES_FILE, SIGNALS_FILE]:
    if os.path.exists(f): os.remove(f)

unique_tickers = sorted(FILES_CONFIG.keys())
stock_dim = len(unique_tickers)
run_dirs = sorted(glob.glob(os.path.join(OUTPUT_MODELS_DIR, "run_*")))
valid_runs = [d for d in run_dirs if os.path.basename(d).split('_')[1].isdigit()]
periods_count = len(glob.glob(os.path.join(valid_runs[0], "period_*")))

for current_N in N_VARIANTS:
    print(f"\n{'='*40}")
    print(f" BADANIE NASYCENIA DLA N = {current_N}")
    print(f"{'='*40}")
    
    final_keys = [f"{m.upper()}_AVG" for m in MODELS_TO_BAG]
    for combo in ALL_COMBINATIONS:
        combo_str = "-".join([m.upper() for m in combo])
        final_keys += [f"A_AVG_{combo_str}", f"A_WIN_{combo_str}", f"A_SMTH_{combo_str}"]
    
    equity_traces = {k: [] for k in final_keys}
    tournament_results = {k: [] for k in final_keys}

    pre_sampled_paths = {m: [] for m in MODELS_TO_BAG}
    for m_name in MODELS_TO_BAG:
        possible = [r for r in valid_runs if os.path.exists(os.path.join(r, "period_0", m_name))]
        n_actual = min(current_N, len(possible))
        
        if current_N > len(possible):
            print(f"⚠️ Uwaga: N={current_N} dla {m_name.upper()} przekracza dostępną pulę ({len(possible)}). Użyto wszystkich.")
            
        for _ in range(NUM_BOOTSTRAP_ITERATIONS):
            sampled_run = list(random.sample(possible, n_actual))
            pre_sampled_paths[m_name].append(sampled_run)

    for b_iter in tqdm(range(NUM_BOOTSTRAP_ITERATIONS), desc=f"Postęp dla N={current_N}"):
        
        sampled_paths = {m: pre_sampled_paths[m][b_iter] for m in MODELS_TO_BAG}
        iter_states = {k: {'cash': INITIAL_CAPITAL, 'holdings': [0.0]*stock_dim, 'to': 0.0, 'mo': ""} for k in final_keys}
        iter_logs = {k: [] for k in final_keys}

        for i in range(periods_count):
            if i < start_idx or i > end_idx: continue
            
            p_key = f"okres_{i}"
            inds = indicators_dict.get(p_key, indicators_dict.get("okres_0", []))
            s_dim = 1 + 2 * stock_dim + len(inds) * stock_dim
            test_df = df_global[(df_global.date >= GLOBAL_DATES[i]) & (df_global.date < GLOBAL_DATES[i] + relativedelta(months=TEST_MONTHS))].copy()
            test_df.index = test_df.date.factorize()[0]
            
            train_start_calib = GLOBAL_DATES[i] - relativedelta(years=TRAIN_YEARS)
            df_calib = df_global[(df_global.date >= train_start_calib) & (df_global.date < GLOBAL_DATES[i])]
            ref_prices_per_ticker = [max(df_calib[df_calib.tic == t]['close'].max(), 1.0) if not df_calib[df_calib.tic == t].empty else 1.0 for t in unique_tickers]
            cur_spot = [PRICE_LOOKUP.get((GLOBAL_DATES[i], t), 1.0) for t in unique_tickers]

            period_params = {
                "stock_dim": stock_dim, "initial_amount": INITIAL_CAPITAL, "num_stock_shares": [0.0]*stock_dim,
                "buy_cost_pct": [COST]*stock_dim, "sell_cost_pct": [COST]*stock_dim, "reward_scaling": REWARD_SCALING,
                "action_space": stock_dim, "tech_indicator_list": inds, "state_space": s_dim, "df": test_df, "risk_indicator_col": "turbulence"
            }

            loaded_by_type = {m: [] for m in MODELS_TO_BAG}
            metrics_by_type = {m: [] for m in MODELS_TO_BAG}

            for m_name in MODELS_TO_BAG:
                dummy_v = DummyVecEnv([lambda: StockTradingEnvCustom(**{**period_params, "hmax": 100})])
                for r_path in sampled_paths[m_name]:
                    p_mod = os.path.join(r_path, f"period_{i}", m_name, "best_model.zip")
                    p_vec = os.path.join(r_path, f"period_{i}", m_name, "best_model_vecnorm.pkl")
                    p_eval = os.path.join(r_path, f"period_{i}", m_name, "evaluations.csv")
                    if os.path.exists(p_mod) and os.path.exists(p_vec):
                        try:
                            m_obj = MODEL_CLASSES[m_name].load(p_mod, device='cpu')
                            v_obj = VecNormalize.load(p_vec, dummy_v)
                            v_obj.training = False; v_obj.norm_reward = False
                            loaded_by_type[m_name].append((m_obj, v_obj))
                            if os.path.exists(p_eval): metrics_by_type[m_name].append(pd.read_csv(p_eval)['sortino'].max())
                        except: continue

            val_stats = {m: (calculate_iqm(metrics_by_type[m]) if METRIC_CHOICE=="iqm" else np.median(metrics_by_type[m])) if metrics_by_type[m] else -999 for m in MODELS_TO_BAG}

            def execute_sim(key, agents, weights=None):
                st = iter_states[key]
                d_hmax, _ = get_dynamic_hmax(st['cash'], st['holdings'], cur_spot, ref_prices_per_ticker, COST)
                sim_p = {**period_params, "num_stock_shares": list(st['holdings']), "initial_amount": st['cash'], "hmax": d_hmax}
                v = DummyVecEnv([lambda: StockTradingEnvCustom(**sim_p)])
                res_df, f_c, f_h, t_o, m_o = run_simulation_engine_aggregated(v, agents, st['cash'], st['holdings'], st['to'], st['mo'], weights=weights)
                iter_logs[key].append(res_df)
                iter_states[key].update({'cash': f_c, 'holdings': f_h, 'to': t_o, 'mo': m_o})

            for m_name in MODELS_TO_BAG:
                if loaded_by_type[m_name]: execute_sim(f"{m_name.upper()}_AVG", loaded_by_type[m_name])

            for combo in ALL_COMBINATIONS:
                c_str = "-".join([m.upper() for m in combo])
                
                valid_m = [m for m in combo if len(loaded_by_type[m]) > 0]
                if not valid_m: 
                    continue
                
                c_mods = []
                c_vals = []
                for m in valid_m:
                    c_mods.extend(loaded_by_type[m])
                    c_vals.append(val_stats[m])
                
                num_valid_algos = len(valid_m)
                avg_weights = []
                for m in valid_m:
                    n = len(loaded_by_type[m])
                    class_weight = 1.0 / num_valid_algos
                    agent_weight = class_weight / n
                    avg_weights.extend([agent_weight] * n)
                    
                execute_sim(f"A_AVG_{c_str}", c_mods, weights=avg_weights)

                clean_vals = np.nan_to_num(c_vals, nan=-9999)
                winner_idx = np.argmax(clean_vals)
                winner_name = valid_m[winner_idx]
                
                execute_sim(f"A_WIN_{c_str}", loaded_by_type[winner_name])

                w_norm = softmax(clean_vals)
                smth_weights = []
                for idx, m in enumerate(valid_m):
                    n = len(loaded_by_type[m])
                    agent_weight = w_norm[idx] / n
                    smth_weights.extend([agent_weight] * n)
                    
                execute_sim(f"A_SMTH_{c_str}", c_mods, weights=smth_weights)

        for k in final_keys:
            if iter_logs[k]:
                # [ZMIANA] Wymuszamy subset=['date'] bo kolumna 'raw_act' zawiera tablice (unhashable)
                full_df = pd.concat(iter_logs[k]).drop_duplicates(subset=['date']).sort_values('date')
                tournament_results[k].append(calculate_sortino(full_df))
                
                # [ZMIANA] Przekazujemy również wyliczone surowe sygnały z df
                equity_traces[k].append(full_df[['date', 'account_value', 'raw_act']].copy())

    print(f"💾 Zapisywanie wyników dla N={current_N}...")
    for k, values in tournament_results.items():
        if values:
            for b_idx, sortino_val in enumerate(values):
                master_summary_data.append({
                    'Model': k, 
                    'N': current_N, 
                    'Bootstrap_ID': b_idx + 1, 
                    'Sortino': sortino_val
                })

    all_traces_flat = []
    all_signals_flat = [] # [NOWE] Lista na zgrupowane ramki sygnałów
    
    for model_name, traces_list in equity_traces.items():
        for iteration_idx, df_trace in enumerate(traces_list):
            temp_df = df_trace.copy()
            
            # ------------------------------------------------------------------
            # [NOWE] Ekstrakcja Sygnałów (bardzo wydajny unnest tablic Numpy)
            # ------------------------------------------------------------------
            # Tablica ma kształt (liczba_dni, liczba_spółek)
            raw_acts_matrix = np.stack(temp_df['raw_act'].values)
            
            # Replikujemy daty z wektora aby zgadzały się wymiary
            dates_rep = np.repeat(temp_df['date'].values, stock_dim)
            # Powielamy listę tickerów przez ilość dni
            tickers_rep = np.tile(unique_tickers, len(temp_df))
            
            # Płaska ramka danych pod zapis CSV z sygnałami
            sig_df = pd.DataFrame({
                'Date': dates_rep,
                'Model': model_name,
                'N': current_N,
                'Bootstrap_ID': iteration_idx + 1,
                'Ticker': tickers_rep,
                'Raw_Signal': raw_acts_matrix.flatten()
            })
            all_signals_flat.append(sig_df)
            # ------------------------------------------------------------------

            # Przygotowanie klasycznego temp_df (wywalamy z niego tablice przed zrzutem)
            temp_df = temp_df.drop(columns=['raw_act'])
            temp_df['Model'] = model_name
            temp_df['N'] = current_N
            temp_df['Iteration'] = iteration_idx + 1
            all_traces_flat.append(temp_df[['Model', 'N', 'Iteration', 'date', 'account_value']])
    
    # [NOWE] Zapis nowej paczki spłaszczonych surowych sygnałów 
    if all_signals_flat:
        df_sig_chunk = pd.concat(all_signals_flat, ignore_index=True)
        write_header_sig = not os.path.exists(SIGNALS_FILE)
        df_sig_chunk.to_csv(SIGNALS_FILE, mode='a', index=False, sep=';', decimal=',', header=write_header_sig)

    if all_traces_flat:
        df_chunk = pd.concat(all_traces_flat, ignore_index=True)
        write_header = not os.path.exists(TRACES_FILE)
        df_chunk.to_csv(TRACES_FILE, mode='a', index=False, sep=';', decimal=',', header=write_header)
    
    del equity_traces
    del tournament_results

# ==============================================================================
# 5. FINALIZACJA
# ==============================================================================
df_master_summary = pd.DataFrame(master_summary_data)
df_master_summary.to_csv(SUMMARY_FILE, index=False, sep=';', decimal=',')

print(f"\n{'='*60}")
print(f"✅ BADANIE NASYCENIA ZAKOŃCZONE POMYŚLNIE")
print(f"📊 Plik z podsumowaniem statystycznym (N od 1 do {max_n_to_check}):\n   {SUMMARY_FILE}")
print(f"📈 Plik z dokładnymi krzywymi kapitału:\n   {TRACES_FILE}")
print(f"🧠 Plik ze zrzuconymi logami surowych sygnałów komitetów:\n   {SIGNALS_FILE}") # [NOWE]
print(f"{'='*60}")

Dostępne okresy: ['Q1 2020', 'Q2 2020', 'Q3 2020', 'Q4 2020']


Początek (np. 'Q1 2020'):  
Koniec (np. 'Q4 2020'):  

Podaj liczbę powtórzeń Monte Carlo (B) dla każdego N [np. 30]:  1
Do jakiego maksymalnego N sprawdzić komitety? [np. 100]:  15
Sposób testowania N: (C)iągły [1,2,3...], czy (R)zadki [1,2,5,10,20,50,100]? (C/r) [C]:  C
Metryka dla WIN/SMTH (median/iqm)? [iqm]:  median



 BADANIE NASYCENIA DLA N = 1


Postęp dla N=1: 100%|█████████████████████████████| 1/1 [00:22<00:00, 22.24s/it]


💾 Zapisywanie wyników dla N=1...

 BADANIE NASYCENIA DLA N = 2


...

💾 Zapisywanie wyników dla N=13...

 BADANIE NASYCENIA DLA N = 14


Postęp dla N=14: 100%|███████████████████████████| 1/1 [01:40<00:00, 100.25s/it]


💾 Zapisywanie wyników dla N=14...

 BADANIE NASYCENIA DLA N = 15


Postęp dla N=15: 100%|███████████████████████████| 1/1 [01:46<00:00, 106.49s/it]


💾 Zapisywanie wyników dla N=15...

✅ BADANIE NASYCENIA ZAKOŃCZONE POMYŚLNIE
📊 Plik z podsumowaniem statystycznym (N od 1 do 15):
   /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/wyniki_podsumowanie_NASYCENIE_N1_15_B1.csv
📈 Plik z dokładnymi krzywymi kapitału:
   /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/szczegoly_account_value_NASYCENIE_N1_15_B1.csv
🧠 Plik ze zrzuconymi logami surowych sygnałów komitetów:
   /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/sygnaly_surowe_NASYCENIE_N1_15_B1.csv


### Rozkład przebiegu krzywych kapitału modeli bazowych (wybrane N)

In [33]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ==============================================================================
# 1. KONFIGURACJA I PARAMETRY
# ==============================================================================

MODEL_COLORS = {
    'A2C': '#2CA02C', 'PPO': '#DAA520', 'DDPG': '#D62728', 
    'TD3': '#1F77B4', 'SAC': '#9467BD'
}
# Skupiamy się na modelach bazowych, tak jak w kodzie (3)
MODELS_TO_PLOT = ['A2C_AVG', 'PPO_AVG', 'DDPG_AVG', 'TD3_AVG', 'SAC_AVG']

print("--- KONFIGURACJA WYSZUKIWANIA PLIKU ---")
max_n_to_check = int(input("Do jakiego N był generowany plik? (max_n_to_check) [np. 100]: ") or 100)
NUM_BOOTSTRAP_ITERATIONS = int(input("Ile było powtórzeń Monte Carlo (B)? [np. 30]: ") or 30)
density_choice = input("Czy użyto rzadkiej siatki N w nazwie pliku? (T/N) [N]: ").strip().upper()

file_suffix = "_RZADKIE" if density_choice == 'T' else ""
filename = f"szczegoly_account_value_NASYCENIE_N1_{max_n_to_check}_B{NUM_BOOTSTRAP_ITERATIONS}{file_suffix}.csv"
TRACES_FILE = os.path.join(AGG_DIR, filename)

print("\n--- KONFIGURACJA WYKRESU ---")
target_n = int(input(f"Dla jakiego N chcesz wygenerować wykres? [np. 5]: ") or 5)

# ==============================================================================
# 2. FUNKCJE DO OBLICZANIA METRYK
# ==============================================================================

def calculate_metrics(series):
    """Oblicza zestaw metryk dla pojedynczej krzywej kapitału."""
    returns = series.pct_change().dropna()
    if len(returns) < 2:
        return [0] * 5
    
    # 1. Skumulowana stopa zwrotu
    cum_ret = (series.iloc[-1] / series.iloc[0]) - 1
    
    # 2. Sharpe Ratio (Roczny, rf=0)
    sharpe = (returns.mean() / (returns.std() + 1e-9)) * np.sqrt(252)
    
    # 3. Sortino Ratio
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252) if len(downside_returns) >= 2 else returns.std() * np.sqrt(252)
    sortino = (returns.mean() * 252) / (downside_std + 1e-9)
    
    # 4. Max Drawdown
    cum_max = series.cummax()
    drawdown = (series - cum_max) / cum_max
    max_dd = drawdown.min()
    
    # 5. Calmar Ratio
    calmar = (returns.mean() * 252) / (abs(max_dd) + 1e-9)
    
    return [cum_ret, sharpe, sortino, max_dd, calmar]

# ==============================================================================
# 3. WCZYTYWANIE I FILTROWANIE DANYCH
# ==============================================================================

print(f"\n⏳ Wczytywanie danych z pliku: {filename}...")
if not os.path.exists(TRACES_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku {TRACES_FILE}")
    exit()

df_all = pd.read_csv(TRACES_FILE, sep=';', decimal=',')
df_all['date'] = pd.to_datetime(df_all['date'])

# Filtrowanie dla wybranego N
df_target = df_all[df_all['N'] == target_n].copy()

if df_target.empty:
    print(f"❌ BŁĄD: Brak danych dla N = {target_n} w tym pliku!")
    exit()

# ==============================================================================
# 4. PRZYGOTOWANIE SIATKI WYKRESÓW
# ==============================================================================

fig = plt.figure(figsize=(18, 16), facecolor='white')
gs = GridSpec(3, 4, figure=fig)
ax_base = fig.add_subplot(gs[0, 0:2]) 
axes = [
    ax_base,
    fig.add_subplot(gs[0, 2:4], sharex=ax_base, sharey=ax_base),
    fig.add_subplot(gs[1, 0:2], sharex=ax_base, sharey=ax_base),
    fig.add_subplot(gs[1, 2:4], sharex=ax_base, sharey=ax_base),
    fig.add_subplot(gs[2, 1:3], sharex=ax_base, sharey=ax_base)
]

all_metrics_storage = []

# ==============================================================================
# 5. PĘTLA ANALITYCZNA (RYSOWANIE I OBLICZANIE)
# ==============================================================================

for idx, model_key in enumerate(MODELS_TO_PLOT):
    ax = axes[idx]
    base_name = model_key.replace('_AVG', '')
    
    current_color = MODEL_COLORS.get(base_name, '#000000') 
    
    print(f"📊 Przetwarzanie modelu: {base_name} (Dla N={target_n})...")
    
    model_data = df_target[df_target['Model'] == model_key]
    iterations = model_data['Iteration'].unique()
    
    model_iteration_series = [] 
    iteration_metrics = []      
    runs_for_median = []        

    for it in iterations:
        it_df = model_data[model_data['Iteration'] == it].sort_values('date')
        it_series = it_df.set_index('date')['account_value']
        
        if it_series.empty: continue
            
        norm_series = (it_series / it_series.iloc[0]) - 1
        model_iteration_series.append(norm_series)
        
        metrics = calculate_metrics(it_series)
        iteration_metrics.append(metrics)
        
        # Zapamiętanie krzywej wraz ze wskaźnikiem Sortino
        runs_for_median.append({
            'iteration': it,
            'sortino': metrics[2], 
            'curve': norm_series
        })

    if not model_iteration_series:
        ax.set_title(f"{base_name} (Brak Danych)", fontsize=14, color='red')
        ax.axis('off')
        continue

    # Szukanie medianowego przebiegu wg Sortino
    runs_for_median.sort(key=lambda x: x['sortino'])
    n_runs = len(runs_for_median)
    
    if n_runs % 2 != 0:
        mid_idx = n_runs // 2
        median_curve = runs_for_median[mid_idx]['curve']
    else:
        mid1 = n_runs // 2 - 1
        mid2 = n_runs // 2
        curve1 = runs_for_median[mid1]['curve']
        curve2 = runs_for_median[mid2]['curve']
        median_curve = (curve1 + curve2) / 2.0

    # Rozrzut całkowity (Min-Max)
    combined_df = pd.concat(model_iteration_series, axis=1)
    min_curve = combined_df.min(axis=1)
    max_curve = combined_df.max(axis=1)
    
    # --- RYSOWANIE WŁAŚCIWE ---
    dates = combined_df.index
    
    ax.fill_between(dates, min_curve, max_curve, 
                    color=current_color, alpha=0.2, zorder=2)
    
    ax.plot(dates, median_curve, color=current_color, linewidth=2.5, zorder=5)
    
    ax.set_title(f"{base_name} (N={target_n})", fontsize=24, fontweight='bold', color=current_color)
    ax.set_ylabel("Skumulowana stopa zwrotu (%)", fontsize=12)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, which='major', linestyle='--', alpha=0.5)
    ax.axhline(0, color='black', linewidth=1, alpha=0.3)

    # --- AGREGACJA METRYK ---
    metrics_df = pd.DataFrame(iteration_metrics, columns=['Ret', 'Sharpe', 'Sortino', 'MDD', 'Calmar'])
    for col in metrics_df.columns:
        all_metrics_storage.append({
            'Model': base_name,
            'Metryka': col,
            'Średnia': metrics_df[col].mean(),
            'Mediana': metrics_df[col].median(),
            'Min': metrics_df[col].min(),
            'Max': metrics_df[col].max(),
            'Std': metrics_df[col].std()
        })

# ==============================================================================
# 6. KOŃCOWE FORMATOWANIE WYKRESU I ZAPIS
# ==============================================================================

plt.tight_layout(rect=[0, 0.08, 1, 0.98])
custom_legend_elements = [
    Patch(facecolor='gray', alpha=0.3, label='Rozpiętość wyników (Min-Max)'),
    Line2D([0], [0], color='black', lw=2.5, label='Prawdziwy przebieg medianowy (wg Sortino)')
]
fig.legend(handles=custom_legend_elements, loc='lower center', 
           ncol=2, fontsize=18, bbox_to_anchor=(0.5, 0.02), 
           frameon=True, shadow=True)

# Unikalna nazwa dla wykresu i tabeli na podstawie zmiennych wejściowych
plot_filename = f"Zestawienie_Modeli_Bazowych_N{target_n}_B{NUM_BOOTSTRAP_ITERATIONS}.png"
save_path = os.path.join(AGG_DIR, plot_filename)

plt.savefig(save_path, dpi=300, bbox_inches='tight') 
print(f"\n✅ Wykres został zapisany w: {save_path}")

# ==============================================================================
# 7. TWORZENIE I WYŚWIETLANIE TABELI METRYK
# ==============================================================================

if all_metrics_storage:
    final_table = pd.DataFrame(all_metrics_storage)
    metric_name_map = {
        'Ret': 'Skumulowany Zwrot',
        'Sharpe': 'Wskaźnik Sharpe',
        'Sortino': 'Wskaźnik Sortino',
        'MDD': 'Max Drawdown',
        'Calmar': 'Wskaźnik Calmar'
    }
    final_table['Metryka'] = final_table['Metryka'].map(metric_name_map)

    pd.set_option('display.max_rows', None)
    pd.set_option('display.float_format', '{:.4f}'.format)
    print("\n" + "="*100)
    print(f"📈 TABELA METRYK PORÓWNAWCZYCH (N={target_n}, Rozkład Bootstrap z pliku nasycenia)")
    print("="*100)
    print(final_table.pivot(index=['Model', 'Metryka'], columns=[]).sort_index())

    csv_report_path = os.path.join(AGG_DIR, f"Raport_Metryk_Bazowych_N{target_n}.csv")
    final_table.to_csv(csv_report_path, sep=';', decimal=',', index=False)
    print(f"✅ Tabela metryk zapisana w: {csv_report_path}")

--- KONFIGURACJA WYSZUKIWANIA PLIKU ---


Do jakiego N był generowany plik? (max_n_to_check) [np. 100]:  50
Ile było powtórzeń Monte Carlo (B)? [np. 30]:  30
Czy użyto rzadkiej siatki N w nazwie pliku? (T/N) [N]:  T



--- KONFIGURACJA WYKRESU ---


KeyboardInterrupt: Interrupted by user

### Sprawdzanie stabilności - zmienność surowych sygnałów

In [12]:
# Flaga decydująca o rysowaniu tytułów wykresów
SHOW_TITLES = False

# ==============================================================================
# 1. KONFIGURACJA I ŁADOWANIE DANYCH
# ==============================================================================
max_n_to_check = int(input("Podaj maksymalne N użyte w symulacji [np. 100]: ") or 100)
b_runs = int(input("Podaj liczbę powtórzeń Monte Carlo (B) użytych w danych [np. 100]: ") or 100)

# Zakładamy, że masz plik z surowymi sygnałami w formacie:
is_sparse = input("Czy dane były generowane w trybie rzadkim (R)? (T/n) [n]: ").strip().upper()
file_suffix = "_RZADKIE" if is_sparse == 'T' else ""

INPUT_FILE = os.path.join(AGG_DIR, f"sygnaly_surowe_NASYCENIE_N1_{max_n_to_check}_B{b_runs}{file_suffix}.csv")

if not os.path.exists(INPUT_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku {INPUT_FILE}.")
    print("Upewnij się, że pierwszy skrypt eksportuje zmienną 'raw_act' dla każdego kroku.")
    exit()

print(f"⏳ Wczytywanie surowych sygnałów z {INPUT_FILE}...")
# Wczytujemy dane. Dla optymalizacji zmieniamy typy.
df_signals = pd.read_csv(INPUT_FILE, sep=';', decimal=',')
df_signals['Date'] = pd.to_datetime(df_signals['Date'])
df_signals['Raw_Signal'] = df_signals['Raw_Signal'].astype(np.float32)

models = df_signals['Model'].unique()
n_values = sorted(df_signals['N'].unique())

# ==============================================================================
# 2. AGREGACJA BOOTSTRAPÓW (Obliczanie Złotego Konsensusu dla N)
# ==============================================================================
print("📊 Obliczanie średnich wektorów sygnałów oraz wariancji wewnątrzkomitetowej...")

# Obliczamy średnią i odchylenie standardowe sygnałów pomiędzy iteracjami B
# Grupujemy po (Model, N, Date, Ticker)
df_grouped = df_signals.groupby(['Model', 'N', 'Date', 'Ticker'])['Raw_Signal'].agg(['mean', 'std']).reset_index()
df_grouped.rename(columns={'mean': 'Mean_Signal', 'std': 'Std_Signal'}, inplace=True)

# Dla wariancji, bierzemy średnie odchylenie standardowe dla całego systemu przy danym N
df_variance = df_grouped.groupby(['Model', 'N'])['Std_Signal'].mean().reset_index()

# ==============================================================================
# 3. BADANIE ZBIEŻNOŚCI (MAC)
# ==============================================================================
print("🔍 Analiza zbieżności i wektorów przesunięć (MAC)...")

results_list = []

for model in models:
    model_data = df_grouped[df_grouped['Model'] == model]
    
    # Przekształcamy na szeroki format: wiersze to N, kolumny to MultiIndex (Date, Ticker)
    # Pozwala to na traktowanie historii sygnałów jako jednego gigantycznego wektora
    pivot_model = model_data.pivot(index='N', columns=['Date', 'Ticker'], values='Mean_Signal')
    
    for i in range(1, len(n_values)):
        prev_n = n_values[i-1]
        curr_n = n_values[i]
        
        vec_prev = pivot_model.loc[prev_n].values
        vec_curr = pivot_model.loc[curr_n].values
        
        # 1. Średni Błąd Bezwzględny (MAC) między komitetami
        MAC = np.mean(np.abs(vec_curr - vec_prev))
        
        results_list.append({
            'Model': model,
            'N': curr_n,
            'Prev_N': prev_n,
            'MAC': MAC
        })

df_results = pd.DataFrame(results_list)

# ==============================================================================
# 4. WIZUALIZACJA 1: SPADEK BŁĘDU
# ==============================================================================
print("📈 Generowanie wykresów zbieżności sygnałów...")

# Wykres zbieżności wybranego modelu (np. najlepszego zespołu lub średniej ze wszystkich)
# Dla czytelności pokażemy średnią dla wszystkich modeli zebranych razem.
df_mean_results = df_results.groupby('N')[['MAC']].mean().reset_index()

# [NOWE] Obliczenie zakresu min/max dla MAC ze wszystkich modeli
MAC_min = df_results.groupby('N')['MAC'].min()
MAC_max = df_results.groupby('N')['MAC'].max()

fig, ax1 = plt.subplots(figsize=(12, 6))

# [NOWE] Tło: Szary obszar rozrzutu dla MAC
ax1.fill_between(df_mean_results['N'], MAC_min, MAC_max, color='gray', alpha=0.25, zorder=1, label='Zakres MAC wszystkich komitetów')

color = 'tab:red'
ax1.set_xlabel('Rozmiar Komitetu (N)', fontsize=20)
ax1.set_ylabel('MAC', color=color, fontsize=20)

# Ustawienie zorder=3 sprawia, że linia rysuje się nad szarym tłem
ax1.plot(df_mean_results['N'], df_mean_results['MAC'], marker='o', color=color, linewidth=2, zorder=3, label='Średnie MAC')

# Dostosowanie rozmiaru czcionki dla wartości na osiach
ax1.tick_params(axis='y', labelcolor=color, labelsize=15)
ax1.tick_params(axis='x', labelsize=15)

# Wymuszenie konkretnych wartości na osi X
ax1.set_xticks([1, 5, 10, 20, 30, 40, 50])

# Dostosowanie legendy
ax1.legend(loc='upper right', frameon=True, shadow=True, fontsize=20)

fig.tight_layout()

if SHOW_TITLES:
    plt.title("Zbieżność Prawdziwego Konsensusu Sieci Neuronowych (Średnia z modeli)", fontsize=14, pad=15)

plot_file_1 = os.path.join(AGG_DIR, "Konwergencja_Sygnalow_Nasycenie.png")
plt.savefig(plot_file_1, dpi=300)
plt.show()

# ==============================================================================
# 5. WIZUALIZACJA 2: ZANIKANIE WARIANCJI BOOTSTRAP (SZUMU)
# ==============================================================================
print("📈 Generowanie wykresu redukcji szumu (Prawa Wielkich Liczb)...")

# Mapowanie #E377C2ów rozszerzone o sufiksy _AVG dla bezpieczeństwa
MODEL_COLORS = {
    'A2C': '#2CA02C', 'PPO': '#DAA520', 'DDPG': '#D62728', 
    'TD3': '#1F77B4', 'SAC': '#9467BD',
    'A2C_AVG': '#2CA02C', 'PPO_AVG': '#DAA520', 'DDPG_AVG': '#D62728', 
    'TD3_AVG': '#1F77B4', 'SAC_AVG': '#9467BD'
}

base_models = ['PPO_AVG', 'A2C_AVG', 'DDPG_AVG', 'TD3_AVG', 'SAC_AVG']

fig, ax = plt.subplots(figsize=(12, 6))

# Rozdzielenie danych na modele bazowe i całą resztę (zespoły)
df_base = df_variance[df_variance['Model'].isin(base_models)]
df_other = df_variance[~df_variance['Model'].isin(base_models)]

# 1. TŁO: Rysowanie szarego obszaru dla pozostałych modeli (zespołów)
if not df_other.empty:
    other_min = df_other.groupby('N')['Std_Signal'].min()
    other_max = df_other.groupby('N')['Std_Signal'].max()
    ax.fill_between(other_min.index, other_min, other_max, color='gray', alpha=0.25, zorder=1)

# 2. LINIE: Rysowanie krzywych dla 5 głównych modeli
for model in base_models:
    model_data = df_base[df_base['Model'] == model]
    if not model_data.empty:
        ax.plot(model_data['N'], model_data['Std_Signal'], 
                marker='o', linewidth=2.5, zorder=3,
                color=MODEL_COLORS.get(model, '#000000'), 
                label=model.replace("_AVG", "")) # Usuwamy _AVG dla czytelności w legendzie

# 3. ESTETYKA I LEGENDA
if SHOW_TITLES:
    plt.title("Wygaszanie szumu w surowych sygnalach (odchylenie Standardowe w zależności od N)", fontsize=14, pad=15)

plt.xlabel("Rozmiar Komitetu (N)", fontsize=16)
plt.ylabel("Odchylenie standardowe surowych sygnałów", fontsize=16)

# Dostosowanie rozmiaru czcionki dla wartości na osiach
ax.tick_params(axis='both', which='major', labelsize=13)

# Wymuszenie konkretnych wartości na osi X
ax.set_xticks([1, 5, 10, 20, 30, 40, 50])

plt.grid(True, linestyle='--', alpha=0.6, zorder=0)

# Pobieramy dotychczasowe elementy legendy (czyli nasze 5 linii)
from matplotlib.patches import Patch
handles, labels = ax.get_legend_handles_labels()

# Jeśli istnieją inne modele, doklejamy szary kwadracik do legendy z opisem tła
if not df_other.empty:
    handles.append(Patch(facecolor='gray', alpha=0.25))
    labels.append('Zakres wariancji wszystkich komitetów')

ax.legend(handles, labels, title='Architektury', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True, fontsize=14, title_fontsize=16)
plt.tight_layout()

plot_file_2 = os.path.join(AGG_DIR, "Wariancja_Sygnalow_Nasycenie.png")
plt.savefig(plot_file_2, dpi=300)

Podaj maksymalne N użyte w symulacji [np. 100]:  50
Podaj liczbę powtórzeń Monte Carlo (B) użytych w danych [np. 100]:  30
Czy dane były generowane w trybie rzadkim (R)? (T/n) [n]:  T


⏳ Wczytywanie surowych sygnałów z /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/sygnaly_surowe_NASYCENIE_N1_50_B30_RZADKIE.csv...
📊 Obliczanie średnich wektorów sygnałów oraz wariancji wewnątrzkomitetowej...
🔍 Analiza zbieżności i wektorów przesunięć (MAC)...
📈 Generowanie wykresów zbieżności sygnałów...
📈 Generowanie wykresu redukcji szumu (Prawa Wielkich Liczb)...


### Rozkład metryk z wielu komitetów przy danej liczbości agentów algorytmów bioracych udział w komitecie

In [34]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# 1. KONFIGURACJA I ŚCIEŻKI
# ==============================================================================
SHOW_TITLES = False  # Flaga decydująca o rysowaniu głównego tytułu wykresu

BRAND_COLOR = '#007481' # #E377C2 firmowy Pantone 7474C

STYLES = {
    'BUY_AND_HOLD': ('black',   '-', 'Buy & Hold'),
    'MIN_VAR':      ('#E377C2', '-', 'Min-Variance'),      
    'MEAN_VAR':     ('#A0522D', '-', 'Mean-Semivariance')   # Różowy/Magenta
}

# [NOWE] Pobranie informacji o parametrach symulacji
max_n_to_check = int(input("Podaj maksymalne N użyte w symulacji [np. 100]: ") or 100)
num_b = int(input("Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]: ") or 30)
density_choice = input("Czy w symulacji użyto N rzadkiem? (T/n) [N]: ").strip().upper()

file_suffix = "_RZADKIE" if density_choice == 'T' else ""
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_NASYCENIE_N1_{max_n_to_check}_B{num_b}{file_suffix}.csv")

if not os.path.exists(TRACES_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku {TRACES_FILE}")
    exit()

print("\n--- ZAKRES CZASOWY (DO BENCHMARKÓW) ---")
user_start = input("Z jakiego początku pobrać benchmarki (np. 'Q1 2020') [Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Do jakiego końca pobrać benchmarki (np. 'Q4 2020') [Q4 2020]: ").strip() or "Q4 2020"

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")
period_suffix = f"_{safe_start_str}" if user_start == user_end else f"_{safe_start_str}_{safe_end_str}"

BENCH_FILE = os.path.join(BENCHMARK_DIR, f"Metryki{period_suffix}.csv")
bench_df = None

if os.path.exists(BENCH_FILE):
    try:
        try:
            bench_df = pd.read_csv(BENCH_FILE, sep=';', decimal=',')
            if 'Strategy' not in bench_df.columns:
                bench_df = pd.read_csv(BENCH_FILE)
        except:
            bench_df = pd.read_csv(BENCH_FILE)
        print(f"✅ Znaleziono plik benchmarków: Metryki{period_suffix}.csv")
    except Exception as e:
        print(f"⚠️ Błąd wczytywania benchmarków: {e}")
else:
    print(f"⚠️ Nie znaleziono pliku: {BENCH_FILE}. Benchmarki nie zostaną narysowane.")

# ==============================================================================
# 2. FUNKCJE POMOCNICZE DO METRYK
# ==============================================================================
def calculate_metrics(series):
    returns = series.pct_change().dropna()
    if len(returns) < 2:
        return {'Cum_Ret': 0.0, 'Sharpe': 0.0, 'Sortino': 0.0, 'MDD': 0.0, 'Calmar': 0.0}
    
    cum_ret = (series.iloc[-1] / series.iloc[0]) - 1
    sharpe = (returns.mean() / (returns.std() + 1e-9)) * np.sqrt(252)
    
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252) if len(downside_returns) >= 2 else returns.std() * np.sqrt(252)
    sortino = (returns.mean() * 252) / (downside_std + 1e-9)
    
    cum_max = series.cummax()
    drawdown = (series - cum_max) / cum_max
    max_dd = drawdown.min()
    
    calmar = (returns.mean() * 252) / (abs(max_dd) + 1e-9)
    
    return {
        'Skumulowany_Zwrot': cum_ret, 
        'Sharpe': sharpe, 
        'Sortino': sortino, 
        'Max_Drawdown': max_dd, 
        'Calmar': calmar
    }

# ==============================================================================
# 3. WCZYTYWANIE DANYCH I IDENTYFIKACJA N
# ==============================================================================
print(f"⏳ Wczytywanie danych z {TRACES_FILE}...")
df_all = pd.read_csv(TRACES_FILE, sep=';', decimal=',')
df_all['date'] = pd.to_datetime(df_all['date'])

available_n = sorted(df_all['N'].unique().tolist())
print("\n" + "="*60)
print(f" ZNALEZIONO WARIANTY 'N': {', '.join([str(n) for n in available_n])}")
print("="*60)
print("Rozpoczynam masowe przeliczanie metryk i generowanie rankingów...\n")

metrics_to_plot = ['Skumulowany_Zwrot', 'Sortino', 'Sharpe', 'Calmar', 'Max_Drawdown']
plot_data = {m: {'N': [], 'min': [], 'max': [], 'median': [], 'q025': [], 'q975': []} for m in metrics_to_plot}

all_detailed_metrics_list = []

# ==============================================================================
# 4 & 5. PĘTLA GŁÓWNA: OBLICZANIE METRYK I RANKINGÓW DLA KAŻDEGO N
# ==============================================================================
for current_n in available_n:
    print(f"⚙️ Przetwarzanie komitetów dla N = {current_n}...")
    df_target_n = df_all[df_all['N'] == current_n].copy()

    metrics_results = []
    grouped = df_target_n.groupby(['Model', 'Iteration'])

    for (model_name, iteration), group_df in grouped:
        group_df = group_df.sort_values(by='date')
        m_dict = calculate_metrics(group_df['account_value'])
        m_dict['Model'] = model_name
        m_dict['Iteration'] = iteration
        m_dict['N'] = current_n
        metrics_results.append(m_dict)

    df_detailed_metrics = pd.DataFrame(metrics_results)
    
    all_detailed_metrics_list.append(df_detailed_metrics)

    for metric in metrics_to_plot:
        plot_data[metric]['N'].append(current_n)
        plot_data[metric]['min'].append(df_detailed_metrics[metric].min())
        plot_data[metric]['max'].append(df_detailed_metrics[metric].max())
        plot_data[metric]['median'].append(df_detailed_metrics[metric].median())
        plot_data[metric]['q025'].append(df_detailed_metrics[metric].quantile(0.025))
        plot_data[metric]['q975'].append(df_detailed_metrics[metric].quantile(0.975))

    cols_to_agg = ['Skumulowany_Zwrot', 'Sortino', 'Sharpe', 'Calmar', 'Max_Drawdown']
    df_summary = df_detailed_metrics.groupby('Model')[cols_to_agg].median().reset_index()
    df_summary['N'] = current_n

    if bench_df is not None and not bench_df.empty:
        bench_rename_map = {
            'Strategy': 'Model',
            'Total Return': 'Skumulowany_Zwrot',
            'Sortino Ratio': 'Sortino',
            'Sharpe Ratio': 'Sharpe',
            'Calmar Ratio': 'Calmar',
            'Max Drawdown': 'Max_Drawdown'
        }
        bench_mapped = bench_df.copy()
        if 'Strategy' not in bench_mapped.columns and 'Unnamed: 0' in bench_mapped.columns:
            bench_mapped = bench_mapped.rename(columns={'Unnamed: 0': 'Strategy'})
            
        bench_mapped = bench_mapped.rename(columns=bench_rename_map)
        bench_mapped['N'] = 'Benchmark'
        bench_mapped['Model'] = "[BENCHMARK] " + bench_mapped['Model'].astype(str)
        
        cols_order = ['Model', 'N', 'Sortino', 'Skumulowany_Zwrot', 'Sharpe', 'Calmar', 'Max_Drawdown']
        bench_mapped = bench_mapped[[c for c in cols_order if c in bench_mapped.columns]]
        df_summary = pd.concat([df_summary, bench_mapped], ignore_index=True)
    else:
        cols_order = ['Model', 'N', 'Sortino', 'Skumulowany_Zwrot', 'Sharpe', 'Calmar', 'Max_Drawdown']

    df_summary = df_summary[cols_order]

    for col in ['Sortino', 'Skumulowany_Zwrot', 'Sharpe', 'Calmar', 'Max_Drawdown']:
        df_summary[col] = pd.to_numeric(df_summary[col], errors='coerce')

    df_summary = df_summary.sort_values(by='Skumulowany_Zwrot', ascending=False).reset_index(drop=True)
    df_summary.index += 1 

    out_rank = os.path.join(AGG_DIR, f"Ranking_Metryk_N{current_n}_B{num_b}.csv")
    out_det = os.path.join(AGG_DIR, f"Szczegolowe_Metryki_N{current_n}_B{num_b}.csv")
    df_summary.to_csv(out_rank, sep=';', decimal=',', index_label='Miejsce')
    df_detailed_metrics.to_csv(out_det, sep=';', decimal=',', index=False)

print(f"\n✅ Pomyślnie wygenerowano {len(available_n)} rankingów i zapisano je do katalogu {AGG_DIR}.")

# ==============================================================================
# 5.5. TABELA: % MODELI POKONUJĄCYCH BENCHMARKI (W TYM 'BRAK HANDLU') DLA KAŻDEGO N
# ==============================================================================
if bench_df is not None and not bench_df.empty:
    print("\n📊 Obliczanie % modeli pokonujących benchmarki (w tym 'Brak Handlu')...")
    
    BENCHMARKS_TO_CHECK = {
        'BUY_AND_HOLD': 'B&H',
        'MIN_VAR': 'MinVar',
        'MEAN_VAR': 'MeanVar'
    }
    
    bench_reference_values = {}

    for bench_key, bench_short in BENCHMARKS_TO_CHECK.items():
        bench_row = bench_df[bench_df['Strategy'] == bench_key]
        if not bench_row.empty:
            bench_reference_values[bench_short] = bench_row.iloc[0].to_dict()

    bench_reference_values['BrakHandlu'] = {
        'Total Return': 0.0,
        'Sortino Ratio': 0.0,
        'Sharpe Ratio': 0.0,
        'Calmar Ratio': 0.0,
        'Max Drawdown': 0.0
    }
    
    metric_map_to_bench_local = {
        'Skumulowany_Zwrot': 'Total Return',
        'Sortino': 'Sortino Ratio',
        'Sharpe': 'Sharpe Ratio',
        'Calmar': 'Calmar Ratio',
        'Max_Drawdown': 'Max Drawdown'
    }

    beat_rates = []

    for current_n in available_n:
        det_file = os.path.join(AGG_DIR, f"Szczegolowe_Metryki_N{current_n}_B{num_b}.csv")
        
        if os.path.exists(det_file):
            df_det = pd.read_csv(det_file, sep=';', decimal=',')
            n_results = {'N': current_n}

            for bench_short, bench_vals in bench_reference_values.items():
                for metric, bench_col_name in metric_map_to_bench_local.items():
                    if bench_col_name in bench_vals:
                        b_val = float(bench_vals[bench_col_name])
                        win_rate = (df_det[metric] > b_val).mean() * 100
                        col_name = f"{metric} > {bench_short} [%]"
                        n_results[col_name] = round(win_rate, 2)

            beat_rates.append(n_results)

    if beat_rates:
        df_beat_rates = pd.DataFrame(beat_rates)
        print(f"\n--- PROCENT MODELI POKONUJĄCYCH POSZCZEGÓLNE BENCHMARKI ---")
        print(df_beat_rates.to_markdown(index=False))

        OUT_BEAT = os.path.join(AGG_DIR, f"Procent_Lepszych_Od_4_Benchmarkow_N1_{max_n_to_check}_B{num_b}.csv")
        df_beat_rates.to_csv(OUT_BEAT, sep=';', decimal=',', index=False)
        print(f"\n✅ Zapisano zbiorcze zestawienie procentowe do: {OUT_BEAT}\n")
    else:
        print("⚠️ Nie udało się wygenerować tabeli (brak danych szczegółowych z iteracji).")

# ==============================================================================
# 5.7. OBLICZANIE I ZAPIS STATYSTYK OPISOWYCH (MIN, MAX, MEAN, MEDIAN, STD) PER N
# ==============================================================================
if all_detailed_metrics_list:
    print("📊 Obliczanie pełnych statystyk opisowych dla każdego N...")
    df_all_compiled_metrics = pd.concat(all_detailed_metrics_list, ignore_index=True)
    
    stat_translations = {
        'min': 'min',
        'max': 'max',
        'mean': 'srednia',
        'median': 'mediana',
        'std': 'std'
    }
    
    df_desc_stats = df_all_compiled_metrics.groupby('N')[metrics_to_plot].agg(['min', 'max', 'mean', 'median', 'std'])
    
    df_desc_stats.columns = [f"{metric}_{stat_translations[stat]}" for metric, stat in df_desc_stats.columns]
    df_desc_stats = df_desc_stats.reset_index()
    
    OUT_STATS = os.path.join(AGG_DIR, f"Statystyki_Opisowe_Metryk_N1_{max_n_to_check}_B{num_b}{file_suffix}.csv")
    df_desc_stats.to_csv(OUT_STATS, sep=';', decimal=',', index=False)
    
    print(f"✅ Pomyślnie wygenerowano i zapisano statystyki opisowe dla wszystkich N do: {OUT_STATS}")
    print(df_desc_stats.to_markdown(index=False))
else:
    print("⚠️ Brak danych do wyznaczenia statystyk opisowych.")

# ==============================================================================
# 6. WIZUALIZACJA ZBIORCZA - KONWERGENCJA METRYK (FAN CHART WZGLĘDEM N)
# ==============================================================================
print(f"📊 Generowanie zintegrowanej wizualizacji redukcji wariancji względem N...")

fig = plt.figure(figsize=(18, 11), facecolor='white')
gs = fig.add_gridspec(2, 6, hspace=0.35, wspace=0.45)

ax0 = fig.add_subplot(gs[0, 0:2])
ax1 = fig.add_subplot(gs[0, 2:4])
ax2 = fig.add_subplot(gs[0, 4:6])

ax3 = fig.add_subplot(gs[1, 1:3])
ax4 = fig.add_subplot(gs[1, 3:5])

axes = [ax0, ax1, ax2, ax3, ax4]

titles = {
    'Skumulowany_Zwrot': 'Skumulowany Zwrot',
    'Sortino': 'Wskaźnik Sortino',
    'Sharpe': 'Wskaźnik Sharpe',
    'Calmar': 'Wskaźnik Calmar',
    'Max_Drawdown': 'Maksymalne Obsunięcie (MDD)'
}

metric_map_to_bench = {
    'Skumulowany_Zwrot': 'Total Return',
    'Sortino': 'Sortino Ratio',
    'Sharpe': 'Sharpe Ratio',
    'Calmar': 'Calmar Ratio',
    'Max_Drawdown': 'Max Drawdown'
}

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    x_n = plot_data[metric]['N']
    y_min = plot_data[metric]['min']
    y_max = plot_data[metric]['max']
    y_q025 = plot_data[metric]['q025']
    y_q975 = plot_data[metric]['q975']
    y_median = plot_data[metric]['median']
    
    ax.fill_between(x_n, y_min, y_max, color=BRAND_COLOR, alpha=0.10)
    ax.fill_between(x_n, y_q025, y_q975, color=BRAND_COLOR, alpha=0.25)
    ax.plot(x_n, y_median, color=BRAND_COLOR, marker='o', lw=2.5, markersize=5, zorder=4)
    
    if bench_df is not None:
        bench_col_name = metric_map_to_bench[metric]
        if bench_col_name in bench_df.columns:
            for _, row in bench_df.iterrows():
                strategy = row['Strategy']
                if strategy in STYLES:
                    color, style, label = STYLES[strategy]
                    val = float(row[bench_col_name])
                    ax.axhline(val, color=color, linestyle=style, linewidth=2, zorder=3, alpha=0.8)
    
    ax.set_title(titles[metric], fontsize=15, fontweight='bold', pad=15)
    
    # --- ZMIANY TUTAJ: Delikatne zwiększenie czcionek osi ---
    ax.set_xlabel("Rozmiar Komitetu (N)", fontsize=13)
    ax.set_ylabel(titles[metric], fontsize=13)
    ax.tick_params(axis='both', labelsize=12)
    # --------------------------------------------------------
    
    ax.grid(True, linestyle='--', alpha=0.5, zorder=1)
    ax.set_xticks(x_n)
    
    if metric in ['Skumulowany_Zwrot', 'Max_Drawdown']:
        import matplotlib.ticker as mtick
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        
    ax.axhline(0, color='black', linewidth=1.2, alpha=0.4, zorder=2)


from matplotlib.patches import Patch
from matplotlib.lines import Line2D

legend_elements = [
    Patch(facecolor=BRAND_COLOR, alpha=0.10, label=f'Całkowity zakres wyników'),
    Patch(facecolor=BRAND_COLOR, alpha=0.25, label=f'95% przedział wyników'),
    Line2D([0], [0], color=BRAND_COLOR, lw=2.5, marker='o', label='Mediana wyników')
]

if bench_df is not None:
    for strategy_key, (color, style, label) in STYLES.items():
        if strategy_key in bench_df['Strategy'].values:
            legend_elements.append(Line2D([0], [0], color=color, lw=2.5, linestyle=style, label=f'Benchmark: {label}'))

fig.legend(handles=legend_elements, loc='lower center', ncol=3, 
           bbox_to_anchor=(0.5, 0.02), fontsize=12, frameon=True, shadow=True)

plt.subplots_adjust(bottom=0.15)

if SHOW_TITLES:
    plt.suptitle(f"Zbieżność i Redukcja Wariancji Metryk Ewaluacyjnych względem N (B = {num_b})", 
                 fontsize=18, fontweight='bold', y=0.98)

OUTPUT_PLOT = os.path.join(AGG_DIR, f"Ewolucja_Metryk_Zaleznosc_N1_{max_n_to_check}_B{num_b}.png")
plt.savefig(OUTPUT_PLOT, dpi=300, bbox_inches='tight')
print(f"✅ Zintegrowany wykres konwergencji zapisano jako: {OUTPUT_PLOT}")
plt.show()

Podaj maksymalne N użyte w symulacji [np. 100]:  50
Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]:  30
Czy w symulacji użyto N rzadkiem? (T/n) [N]:  T



--- ZAKRES CZASOWY (DO BENCHMARKÓW) ---


Z jakiego początku pobrać benchmarki (np. 'Q1 2020') [Q1 2020]:  
Do jakiego końca pobrać benchmarki (np. 'Q4 2020') [Q4 2020]:  


✅ Znaleziono plik benchmarków: Metryki_Q1_2020_Q4_2020.csv
⏳ Wczytywanie danych z /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/szczegoly_account_value_NASYCENIE_N1_50_B30_RZADKIE.csv...

 ZNALEZIONO WARIANTY 'N': 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50
Rozpoczynam masowe przeliczanie metryk i generowanie rankingów...

⚙️ Przetwarzanie komitetów dla N = 1...
⚙️ Przetwarzanie komitetów dla N = 5...
⚙️ Przetwarzanie komitetów dla N = 10...
⚙️ Przetwarzanie komitetów dla N = 15...
⚙️ Przetwarzanie komitetów dla N = 20...
⚙️ Przetwarzanie komitetów dla N = 25...
⚙️ Przetwarzanie komitetów dla N = 30...
⚙️ Przetwarzanie komitetów dla N = 35...
⚙️ Przetwarzanie komitetów dla N = 40...
⚙️ Przetwarzanie komitetów dla N = 45...
⚙️ Przetwarzanie komitetów dla N = 50...

✅ Pomyślnie wygenerowano 11 rankingów i zapisano je do katalogu /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych.

📊 Obliczanie % modeli pokonu

### Funkcje gęstości rozkładów metryk dla komitetów z wybranym N

In [6]:
import seaborn as sns
import matplotlib.ticker as mtick

# ==============================================================================
# A. FUNKCJE GĘSTOŚCI DLA KONKRETNEGO (WYBRANEGO) N
# ==============================================================================
print("\n--- GENEROWANIE ROZKŁADÓW GĘSTOŚCI DLA WYBRANEGO N ---")
chosen_n_str = input(f"Podaj N, dla którego chcesz narysować rozkłady gęstości (dostępne: {available_n}): ").strip()

try:
    chosen_n = int(chosen_n_str)
    det_file_path = os.path.join(AGG_DIR, f"Szczegolowe_Metryki_N{chosen_n}_B{num_b}.csv")
    
    if not os.path.exists(det_file_path):
        print(f"❌ BŁĄD: Nie znaleziono pliku {det_file_path}.")
    else:
        df_chosen_n = pd.read_csv(det_file_path, sep=';', decimal=',')
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor='white')
        axes = axes.flatten()
        
        for i, metric in enumerate(metrics_to_plot):
            ax = axes[i]
            
            # Rysowanie funkcji gęstości (KDE)
            sns.kdeplot(data=df_chosen_n, x=metric, ax=ax, fill=True, color=BRAND_COLOR, alpha=0.4, linewidth=2)
            
            # --- PIONOWE LINIE BENCHMARKÓW ---
            if bench_df is not None:
                bench_col_name = metric_map_to_bench[metric]
                if bench_col_name in bench_df.columns:
                    for _, row in bench_df.iterrows():
                        strategy = row['Strategy']
                        if strategy in STYLES:
                            color, style, label = STYLES[strategy]
                            val = float(row[bench_col_name])
                            ax.axvline(x=val, color=color, linestyle=style, linewidth=2.5, zorder=5, label=label)
            
            # Estetyka
            ax.set_title(titles[metric], fontsize=15, fontweight='bold', pad=15)
            ax.set_xlabel(titles[metric], fontsize=11)
            ax.set_ylabel("Gęstość Prawdopodobieństwa", fontsize=11)
            ax.grid(True, linestyle='--', alpha=0.5)
            
            if metric in ['Skumulowany_Zwrot', 'Max_Drawdown']:
                ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
                
            ax.axvline(0, color='black', linewidth=1.2, alpha=0.6, zorder=4)

        fig.delaxes(axes[5]) # Usunięcie pustego wykresu
        
        # Legenda z benchmarków
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.02), fontsize=12, frameon=True, shadow=True)
            
        plt.subplots_adjust(bottom=0.15, hspace=0.35, wspace=0.25)
        plt.suptitle(f"Rozkład Gęstości Metryk dla Komitetów o N = {chosen_n} (Iteracje B = {num_b})", fontsize=18, fontweight='bold', y=0.98)
        
        OUT_KDE_N = os.path.join(AGG_DIR, f"Rozklad_Gestosci_N{chosen_n}_B{num_b}.png")
        plt.savefig(OUT_KDE_N, dpi=300, bbox_inches='tight')
        print(f"✅ Zapisano wykres gęstości do: {OUT_KDE_N}")
        plt.show()

except ValueError:
    print("❌ BŁĄD: Podano nieprawidłową wartość N.")


--- GENEROWANIE ROZKŁADÓW GĘSTOŚCI DLA WYBRANEGO N ---


NameError: name 'available_n' is not defined

### Funkcje gęstości rozkładów dla komitetów z N=1 i N=50

In [35]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Flaga decydująca o rysowaniu tytułów wykresów
SHOW_TITLES = False

# Konfiguracja estetyki dla benchmarków (zgodnie z poprzednimi wykresami)
STYLES = {
    'BUY_AND_HOLD': ('black',   '-', 'Buy & Hold'),
    'MIN_VAR':      ('#E377C2', '-', 'Min-Variance'),       # Cyjanowy/Morski
    'MEAN_VAR':     ('#A0522D', '-', 'Mean-Semivariance')   # Różowy/Magenta
}

# ==============================================================================
# A. PORÓWNANIE FUNKCJI GĘSTOŚCI DLA N=1 ORAZ N=50
# ==============================================================================
print("\n--- GENEROWANIE PORÓWNANIA ROZKŁADÓW GĘSTOŚCI (N=1 vs N=50) ---")

n_values = [1, 50]
# Wybór #E377C2ów dla N=1 i N=50 (np. niebieski i pomarańczowy dla dobrego kontrastu)
kde_colors = ['#1f77b4', '#ff7f0e'] 

dataframes = {}
missing_files = False

# 1. Wczytywanie danych dla obu wartości N
for n in n_values:
    det_file_path = os.path.join(AGG_DIR, f"Szczegolowe_Metryki_N{n}_B{num_b}.csv")
    if not os.path.exists(det_file_path):
        print(f"❌ BŁĄD: Nie znaleziono pliku {det_file_path}.")
        missing_files = True
    else:
        dataframes[n] = pd.read_csv(det_file_path, sep=';', decimal=',')

# 2. Rysowanie wykresów (tylko jeśli oba pliki istnieją)
if not missing_files:
    fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor='white')
    axes = axes.flatten()
    
    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        
        # --- Rysowanie dwóch funkcji gęstości (KDE) ---
        for n, color in zip(n_values, kde_colors):
            sns.kdeplot(
                data=dataframes[n], 
                x=metric, 
                ax=ax, 
                fill=True, 
                color=color, 
                alpha=0.4, 
                linewidth=2,
                label=f'Komitet N={n}' # Etykieta do legendy
            )
        
        # --- PIONOWE LINIE BENCHMARKÓW ---
        if bench_df is not None:
            bench_col_name = metric_map_to_bench[metric]
            if bench_col_name in bench_df.columns:
                for _, row in bench_df.iterrows():
                    strategy = row['Strategy']
                    if strategy in STYLES:
                        color, style, label = STYLES[strategy]
                        val = float(row[bench_col_name])
                        ax.axvline(x=val, color=color, linestyle=style, linewidth=2.5, zorder=5, label=label)
        
        # --- ESTETYKA ---
        # Ustawiamy nazwę metryki jako tytuł nad wykresem (pogrubione, czcionka 16)
        ax.set_title(titles[metric], fontsize=16, fontweight='bold', y=1.02, pad=10)
        
        # Usuwamy set_xlabel dla osi X, aby nie dublować nazwy metryki
        ax.set_xlabel("") 
        
        # Zostawiamy nazwę na osi Y (Gęstość)
        ax.set_ylabel("Gęstość Prawdopodobieństwa", fontsize=15)
        
        # Zwiększone wartości na osiach
        ax.tick_params(axis='both', labelsize=12)
        
        ax.grid(True, linestyle='--', alpha=0.5)
        
        if metric in ['Skumulowany_Zwrot', 'Max_Drawdown']:
            ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
            
        ax.axvline(0, color='black', linewidth=1.2, alpha=0.6, zorder=4)

    # Usunięcie szóstego, pustego wykresu (jeśli masz tylko 5 metryk)
    if len(metrics_to_plot) < 6:
        fig.delaxes(axes[5]) 
    
    # --- LEGENDA (Zbiera KDE oraz Benchmarki z pierwszego wykresu) ---
    handles, labels = axes[0].get_legend_handles_labels()
    
    # Usuwanie duplikatów z legendy (jeśli benchmarki dodały się wielokrotnie, choć tu bierzemy tylko z axes[0])
    by_label = dict(zip(labels, handles))
    
    if by_label:
        fig.legend(
            by_label.values(), by_label.keys(), 
            loc='lower center', ncol=len(by_label), 
            bbox_to_anchor=(0.5, 0.02), fontsize=15, frameon=True, shadow=True # Zwiększona czcionka legendy
        )
        
    plt.subplots_adjust(bottom=0.15, hspace=0.35, wspace=0.25)
    
    if SHOW_TITLES:
        plt.suptitle(f"Porównanie Rozkładu Gęstości Metryk: N=1 vs N=50 (Iteracje B = {num_b})", fontsize=18, fontweight='bold', y=0.98)
    
    OUT_KDE_COMPARE = os.path.join(AGG_DIR, f"Porownanie_Rozkladow_N1_vs_N50_B{num_b}.png")
    plt.savefig(OUT_KDE_COMPARE, dpi=300, bbox_inches='tight')
    print(f"✅ Zapisano wykres gęstości do: {OUT_KDE_COMPARE}")
    plt.show()

else:
    print("❌ BŁĄD: Przerwano rysowanie z powodu brakujących plików danych.")


--- GENEROWANIE PORÓWNANIA ROZKŁADÓW GĘSTOŚCI (N=1 vs N=50) ---
✅ Zapisano wykres gęstości do: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Porownanie_Rozkladow_N1_vs_N50_B30.png


### Pełne zestawienie ile % komitetów lepszych od benchmarków

In [19]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ==============================================================================
# KONFIGURACJA
# ==============================================================================
SHOW_TITLES = False  # Flaga decydująca o rysowaniu tytułu wykresu
BRAND_COLOR = '#007481' # #E377C2 firmowy Pantone 7474C

# 1. Wczytanie wygenerowanych wcześniej danych
df_global = pd.read_csv('/Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Procent_Lepszych_Od_Benchmarkow_GLOBALNY_B30.csv', sep=';', decimal=',')
df_n = pd.read_csv('/Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Procent_Lepszych_Od_4_Benchmarkow_N1_50_B30.csv', sep=';', decimal=',')

# 2. Ujednolicenie kolumny identyfikującej
df_global = df_global.rename(columns={'Populacja': 'N'})
df_global['N'] = 'Globalnie'

# 3. Połączenie w jedną tabelę
df_combined = pd.concat([df_n, df_global], ignore_index=True)
df_combined['N'] = df_combined['N'].astype(str)

# 4. Transpozycja (odwrócenie tabeli)
df_transposed = df_combined.set_index('N').T
df_transposed.index.name = 'Metryka'
df_transposed = df_transposed.reset_index()

# 5. Podział oryginalnej nazwy na 'Benchmark' i 'Wskaźnik' (np. "Sharpe > B&H [%]")
def parse_metric(m):
    parts = m.replace(' [%]', '').split(' > ')
    if len(parts) == 2:
        return parts[1], parts[0].replace('_', ' ')
    return "Inne", m

df_transposed['Benchmark'] = df_transposed['Metryka'].apply(lambda x: parse_metric(x)[0])
df_transposed['Wskaźnik'] = df_transposed['Metryka'].apply(lambda x: parse_metric(x)[1])

# 6. Reorganizacja kolejności kolumn
cols_order = ['Benchmark', 'Wskaźnik'] + [c for c in df_transposed.columns if c not in ['Metryka', 'Benchmark', 'Wskaźnik']]
df_final = df_transposed[cols_order]

# 7. Opcjonalnie: Zmiana nazw wskaźników na Acrobat dla tekstu polskiego
wsK_map = {
    'Skumulowany Zwrot': 'Skumulowana Stopa Zwrotu',
    'Max Drawdown': 'Maksymalne Obsunięcie'
}
df_final['Wskaźnik'] = df_final['Wskaźnik'].replace(wsK_map)

# 8. Zapis wyników tabelarycznych
AGG_DIR = '/Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/'
OUT_FILE = 'Agregacja_Procentowa_Zestawienie.csv'
save_path = os.path.join(AGG_DIR, OUT_FILE)
df_final.to_csv(save_path, index=False, sep=';', decimal=',')
print(f"✅ Pomyślnie wygenerowano sformatowaną tabelę: {OUT_FILE}")


# ==============================================================================
# --- WIZUALIZACJA ZBIEŻNOŚCI POKONYWANIA BENCHMARKÓW ---
# ==============================================================================
print("\n📊 Generowanie zintegrowanego wykresu zbieżności...")

x_n = df_n['N']

fig, ax = plt.subplots(figsize=(13, 7.5), facecolor='white')

# --- GRUPA 1: BENCHMARK ABSOLUTNY ---
ax.plot(x_n, df_n['Skumulowany_Zwrot > BrakHandlu [%]'], marker='*', markersize=10, linewidth=3.5, 
        color='#222222', zorder=5, label='Dodatnia Stopa Zwrotu (Wynik > 0%)')

# --- GRUPA 2: BENCHMARKI RELATYWNE ---
ax.plot(x_n, df_n['Skumulowany_Zwrot > B&H [%]'], marker='o', linewidth=2.0, color=BRAND_COLOR, alpha=0.8, label='Skumulowana Stopa Zwrotu > B&H')
ax.plot(x_n, df_n['Sharpe > B&H [%]'], marker='s', linewidth=2.0, color='orange', alpha=0.8, label='Wskaźnik Sharpe\'a > B&H')
ax.plot(x_n, df_n['Sortino > B&H [%]'], marker='d', linewidth=2.0, color='#d9534f', alpha=0.8, label='Wskaźnik Sortino > B&H')
ax.plot(x_n, df_n['Calmar > B&H [%]'], marker='x', linewidth=2.0, color='#7952b3', alpha=0.8, label='Wskaźnik Calmar > B&H')
ax.plot(x_n, df_n['Max_Drawdown > B&H [%]'], marker='^', linewidth=2.0, color='gray', linestyle='--', alpha=0.8, label='Maksymalne Obsunięcie > B&H')

# Tytuł warunkowy
if SHOW_TITLES:
    ax.set_title('Skuteczność komitetów DRL w funkcji ich liczebności (N)\n(Porównanie rentowności absolutnej oraz efektywności względem strategii Buy & Hold)', 
                 fontsize=13, fontweight='bold', pad=15)

# Zwiększone czcionki (Etykiety 20, wartości 15)
ax.set_xlabel('Rozmiar portfela komitetu (N)', fontsize=20)
ax.set_ylabel('Udział komitetów spełniających kryterium (%)', fontsize=20)
ax.tick_params(axis='both', labelsize=15)

ax.set_xticks(x_n)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(100.0))
ax.set_ylim(-5, 105)

ax.grid(True, linestyle='--', alpha=0.5)

# Legenda z czcionką 15
ax.legend(loc='lower right', frameon=True, shadow=True, fontsize=15, facecolor='white')

plt.tight_layout()

# Zapis gotowej ryciny
PLOT_FILE = 'Zbieznosc_Skutecznosci_Komitetow.png'
plot_save_path = os.path.join(AGG_DIR, PLOT_FILE)
plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')
print(f"✅ Wykres zbieżności i rentowności zapisany jako: {plot_save_path}")
plt.show()

✅ Pomyślnie wygenerowano sformatowaną tabelę: Agregacja_Procentowa_Zestawienie.csv

📊 Generowanie zintegrowanego wykresu zbieżności...
✅ Wykres zbieżności i rentowności zapisany jako: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Zbieznosc_Skutecznosci_Komitetow.png


### Funkcję gęstości rozkładów metryk dla komitetów z WSZYSTKIMI N RAZEM + tabela % komitetów pokonujacych benchmarki w kwestii danej metryki

In [7]:
# ==============================================================================
# B. ANALIZA GLOBALNA: WSZYSTKIE KOMITETY (RÓŻNE N, MODELE, ITERACJE)
# ==============================================================================
print("\n--- GENEROWANIE GLOBALNYCH ROZKŁADÓW I TABELI (ABSOLUTNIE WSZYSTKIE KOMITETY) ---")

all_metrics_list = []
for n in available_n:
    det_file = os.path.join(AGG_DIR, f"Szczegolowe_Metryki_N{n}_B{num_b}.csv")
    if os.path.exists(det_file):
        df_temp = pd.read_csv(det_file, sep=';', decimal=',')
        df_temp['N_Źródłowe'] = n  # Przypisanie skąd pochodzą dane
        all_metrics_list.append(df_temp)

if not all_metrics_list:
    print("❌ BŁĄD: Brak danych do analizy globalnej.")
else:
    df_global = pd.concat(all_metrics_list, ignore_index=True)
    total_committees = len(df_global)
    print(f"📊 Załadowano łącznie {total_committees} wyników z pojedynczych iteracji komitetów.")

    # -------------------------------------------------------------------------
    # 1. WYKRES GĘSTOŚCI (GLOBALNY)
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor='white')
    axes = axes.flatten()
    
    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        
        # Rysowanie globalnej funkcji gęstości
        sns.kdeplot(data=df_global, x=metric, ax=ax, fill=True, color='#810074', alpha=0.4, linewidth=2) # Inny #E377C2 dla odróżnienia (np. purpurowy)
        
        # --- PIONOWE LINIE BENCHMARKÓW ---
        if bench_df is not None:
            bench_col_name = metric_map_to_bench[metric]
            if bench_col_name in bench_df.columns:
                for _, row in bench_df.iterrows():
                    strategy = row['Strategy']
                    if strategy in STYLES:
                        color, style, label = STYLES[strategy]
                        val = float(row[bench_col_name])
                        ax.axvline(x=val, color=color, linestyle=style, linewidth=2.5, zorder=5, label=label)
        
        ax.set_title(f"{titles[metric]} (Baza: {total_committees})", fontsize=15, fontweight='bold', pad=15)
        ax.set_xlabel(titles[metric], fontsize=11)
        ax.set_ylabel("Gęstość Prawdopodobieństwa", fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.5)
        
        if metric in ['Skumulowany_Zwrot', 'Max_Drawdown']:
            ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
            
        ax.axvline(0, color='black', linewidth=1.2, alpha=0.6, zorder=4)

    fig.delaxes(axes[5])
    
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.02), fontsize=12, frameon=True, shadow=True)
        
    plt.subplots_adjust(bottom=0.15, hspace=0.35, wspace=0.25)
    plt.suptitle(f"Globalny Rozkład Gęstości Metryk (Wszystkie N i Modele, Łącznie {total_committees} wyników)", fontsize=18, fontweight='bold', y=0.98)
    
    OUT_KDE_GLOBAL = os.path.join(AGG_DIR, f"Rozklad_Gestosci_GLOBALNY_B{num_b}.png")
    plt.savefig(OUT_KDE_GLOBAL, dpi=300, bbox_inches='tight')
    print(f"✅ Zapisano globalny wykres gęstości do: {OUT_KDE_GLOBAL}")
    plt.show()

    # -------------------------------------------------------------------------
    # 2. TABELA % POKONUJĄCYCH BENCHMARKI (DLA POPULACJI GLOBALNEJ)
    # -------------------------------------------------------------------------
    if bench_df is not None and not bench_df.empty:
        global_beat_rates = {'Populacja': 'Wszystkie komitety (Global)'}
        
        for bench_short, bench_vals in bench_reference_values.items():
            for metric, bench_col_name in metric_map_to_bench.items(): # używamy dict ze zmiennej wczesniej
                if bench_col_name in bench_vals:
                    b_val = float(bench_vals[bench_col_name])
                    # Obliczenie % (dla MDD wyższa wartość np. -0.05 jest lepsza niż -0.10)
                    win_rate = (df_global[metric] > b_val).mean() * 100
                    col_name = f"{metric} > {bench_short} [%]"
                    global_beat_rates[col_name] = round(win_rate, 2)
                    
        df_global_beat = pd.DataFrame([global_beat_rates])
        
        print(f"\n--- PROCENT WSZYSTKICH KOMITETÓW POKONUJĄCYCH BENCHMARKI ---")
        print(df_global_beat.to_markdown(index=False))
        
        OUT_BEAT_GLOBAL = os.path.join(AGG_DIR, f"Procent_Lepszych_Od_Benchmarkow_GLOBALNY_B{num_b}.csv")
        df_global_beat.to_csv(OUT_BEAT_GLOBAL, sep=';', decimal=',', index=False)
        print(f"\n✅ Zapisano tabelę globalną do: {OUT_BEAT_GLOBAL}\n")


--- GENEROWANIE GLOBALNYCH ROZKŁADÓW I TABELI (ABSOLUTNIE WSZYSTKIE KOMITETY) ---
📊 Załadowano łącznie 27390 wyników z pojedynczych iteracji komitetów.
✅ Zapisano globalny wykres gęstości do: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Rozklad_Gestosci_GLOBALNY_B30.png

--- PROCENT WSZYSTKICH KOMITETÓW POKONUJĄCYCH BENCHMARKI ---
| Populacja                   |   Skumulowany_Zwrot > B&H [%] |   Sortino > B&H [%] |   Sharpe > B&H [%] |   Calmar > B&H [%] |   Max_Drawdown > B&H [%] |   Skumulowany_Zwrot > MinVar [%] |   Sortino > MinVar [%] |   Sharpe > MinVar [%] |   Calmar > MinVar [%] |   Max_Drawdown > MinVar [%] |   Skumulowany_Zwrot > MeanVar [%] |   Sortino > MeanVar [%] |   Sharpe > MeanVar [%] |   Calmar > MeanVar [%] |   Max_Drawdown > MeanVar [%] |   Skumulowany_Zwrot > BrakHandlu [%] |   Sortino > BrakHandlu [%] |   Sharpe > BrakHandlu [%] |   Calmar > BrakHandlu [%] |   Max_Drawdown > BrakHandlu [%] |
|:-------------------------

### Wykres krzywej kapitału wybrany komitet oraz N (liczba agentów)

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- 1. KONFIGURACJA I WCZYTANIE DANYCH ---
SHOW_TITLE = False
BRAND_COLOR = '#007481' # Pantone 7474C

max_n_to_check = int(input("Jakie maksymalne N było w pliku nasycenia?: [np. 100] ") or 100)
num_b = int(input("Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]: ") or 30)
density_choice = input("Czy w symulacji użyto N rzadkiego? (T/n) [N]: ").strip().upper()

file_suffix = "_RZADKIE" if density_choice == 'T' else ""
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_NASYCENIE_N1_{max_n_to_check}_B{num_b}{file_suffix}.csv")

if not os.path.exists(TRACES_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku {TRACES_FILE}")
    exit()

print(f"⏳ Wczytywanie danych z {TRACES_FILE}...")
df_all = pd.read_csv(TRACES_FILE, sep=';', decimal=',')
df_all['date'] = pd.to_datetime(df_all['date'])
df_all['Config'] = df_all['Model'] + " | N=" + df_all['N'].astype(str)

# --- 2. WYBÓR KONFIGURACJI ---
available_configs = sorted(df_all['Config'].unique().tolist())
print("\n" + "="*60)
print(f"{'DOSTĘPNE KONFIGURACJE':^60}")
print("="*60)
for i in range(0, len(available_configs), 3):
    print(" | ".join([f"{m:<25}" for m in available_configs[i:i+3]]))
print("="*60)

while True:
    choice = input("\nWpisz nazwę konfiguracji (np. 'PPO_AVG | N=5'): ").strip()
    if choice in available_configs:
        chosen_config = choice
        break
    print("❌ Niepoprawna nazwa. Spróbuj ponownie.")

# --- 3. PRZYGOTOWANIE DANYCH (PIVOTOWANIE) ---
print("⚙️ Przetwarzanie krzywych kapitału...")
df_target = df_all[df_all['Config'] == chosen_config].copy()

df_pivot = df_target.pivot(index='date', columns='Iteration', values='account_value')
df_pivot = df_pivot.fillna(method='ffill')
df_returns = (df_pivot / df_pivot.iloc[0]) - 1

# --- 4. WYBÓR I WCZYTANIE BENCHMARKÓW ---
print("\n" + "="*60)
print(f"{'WYBÓR BENCHMARKÓW DO WYKRESU':^60}")
print("="*60)

user_start = input("Z jakiego początku analizować dane benchmarku (np. 'Q1 2020') [Domyślnie Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Do jakiego końca analizować dane benchmarku (np. 'Q4 2020') [Domyślnie Q4 2020]: ").strip() or "Q4 2020"

# Obejście braku QUARTERS_MAP
QUARTERS_MAP = {'Q1 2020': 0, 'Q2 2020': 1, 'Q3 2020': 2, 'Q4 2020': 3} if 'QUARTERS_MAP' not in locals() else QUARTERS_MAP
start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

if start_idx > end_idx:
    start_idx, end_idx = end_idx, start_idx
    user_start, user_end = user_end, user_start

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")
period_suffix = f"_{safe_start_str}" if start_idx == end_idx else f"_{safe_start_str}_{safe_end_str}"

excel_name = f"Benchmarki{period_suffix}.xlsx"
bench_file_path = os.path.join(BENCHMARK_DIR, excel_name)

STYLES = {
    'BUY_AND_HOLD': ('black', '-', 'Buy & Hold'),
    'MIN_VAR': ('#E377C2', '-', 'Min-Variance'),
    'MEAN_VAR': ('#A0522D', '-', 'Mean-Semivariance')
}

active_benchmarks = {}

if not os.path.exists(bench_file_path):
    print(f"⚠️ Nie znaleziono pliku benchmarków: {excel_name}. Zostaną one pominięte na wykresie.")
else:
    for sheet_name, (color, style, label) in STYLES.items():
        ans = input(f"Czy dorysować benchmark '{label}'? (T/n) [T]: ").strip().upper()
        if ans != 'N':
            active_benchmarks[sheet_name] = (color, style, label)

bench_curves = {}
if active_benchmarks:
    try:
        xl_bench = pd.ExcelFile(bench_file_path)
        for sheet_name, (color, style, label) in active_benchmarks.items():
            if sheet_name in xl_bench.sheet_names:
                df_b = pd.read_excel(xl_bench, sheet_name=sheet_name)
                df_b = df_b.dropna(subset=['date', 'account_value'])
                if df_b.empty: continue
                df_b['date'] = pd.to_datetime(df_b['date'])
                df_b = df_b.sort_values('date')
                
                initial_val = df_b['account_value'].iloc[0]
                df_b['cum_return'] = (df_b['account_value'] / initial_val) - 1
                
                bench_curves[sheet_name] = {
                    'data': df_b.set_index('date')['cum_return'],
                    'color': color,
                    'style': style,
                    'label': label
                }
            else:
                print(f"⚠️ Zakładka {sheet_name} nie istnieje w {excel_name}")
    except Exception as e:
        print(f"❌ Błąd wczytywania benchmarków: {e}")

# --- 5. RYSOWANIE WYKRESU (OBSZAR MIN-MAX + BENCHMARKI) ---
print(f"\n📊 Generowanie zintegrowanego wykresu...")
fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')

median_returns = df_returns.median(axis=1)
min_returns = df_returns.min(axis=1)
max_returns = df_returns.max(axis=1)

q025_returns = df_returns.quantile(0.025, axis=1)
q975_returns = df_returns.quantile(0.975, axis=1)

# Min-Max
ax.fill_between(
    df_returns.index,
    min_returns,
    max_returns,
    color=BRAND_COLOR,
    alpha=0.25,
    edgecolors='none'
)

# 95%
ax.fill_between(
    df_returns.index,
    q025_returns,
    q975_returns,
    color=BRAND_COLOR,
    alpha=0.45,
    edgecolors='none'
)

# Linia Mediany AI
ax.plot(median_returns.index, median_returns, color=BRAND_COLOR, alpha=0.85, linewidth=2.5, zorder=5)

# Rysowanie Benchmarków
for sheet_name, bench_info in bench_curves.items():
    b_series = bench_info['data']
    ax.plot(
    b_series.index,
    b_series.values,
    color=bench_info['color'],
    linestyle=bench_info['style'],
    linewidth=2.5,
    zorder=999
)

# Estetyka osi
title_period = f"{user_start}" if start_idx == end_idx else f"{user_start} do {user_end}"

if SHOW_TITLE:
    ax.set_title(
        f"Krzywa Kapitału z uwzględnieniem benchmarków: {chosen_config}\n"
        f"(Okres: {title_period} | B={num_b})",
        fontsize=16,
        fontweight='bold',
        color='#333333',
        pad=15
    )

ax.set_ylabel("Skumulowana stopa zwrotu (%)", fontsize=20)

ax.tick_params(axis='y', labelsize=15)
ax.tick_params(axis='x', labelsize=15)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.grid(True, which='major', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linewidth=1.5, alpha=0.3)

# Konstrukcja dedykowanej Legendy
legend_elements = [
    # Patch dla 95% (ciemniejszy - alpha=0.45)
    Patch(facecolor=BRAND_COLOR, alpha=0.45, label='Przedział 95%'),
    # Patch dla min-max (jaśniejszy - alpha=0.25)
    Patch(facecolor=BRAND_COLOR, alpha=0.25, label='Obszar min-max'),
    # Linia Mediany
    Line2D([0], [0], color=BRAND_COLOR, lw=2.5, label='Mediana wyników')
]

# Dodawanie benchmarków do legendy
for sheet_name, bench_info in bench_curves.items():
    legend_elements.append(
        Line2D([0], [0], color=bench_info['color'], lw=2.0, linestyle=bench_info['style'], label=f"{bench_info['label']}")
    )

ax.legend(
    handles=legend_elements,
    loc='upper left',
    fontsize=14, # Możesz zmniejszyć fontsize, jeśli legenda będzie zbyt duża przez dodatkowy element
    frameon=True,
    shadow=True
)
plt.tight_layout()

# Zapis
plot_filename = f"Krzywa_Kap_MinMax_Bench_{chosen_config.replace(' | ', '_').replace('=', '')}_B{num_b}{period_suffix}.png"
save_path = os.path.join(AGG_DIR, plot_filename)
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ Wykres zapisany jako: {save_path}")
plt.show()

Jakie maksymalne N było w pliku nasycenia?: [np. 100]  50
Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]:  30
Czy w symulacji użyto N rzadkiego? (T/n) [N]:  T


⏳ Wczytywanie danych z /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/szczegoly_account_value_NASYCENIE_N1_50_B30_RZADKIE.csv...

                   DOSTĘPNE KONFIGURACJE                    
A2C_AVG | N=1             | A2C_AVG | N=10            | A2C_AVG | N=15           
A2C_AVG | N=20            | A2C_AVG | N=25            | A2C_AVG | N=30           
A2C_AVG | N=35            | A2C_AVG | N=40            | A2C_AVG | N=45           
A2C_AVG | N=5             | A2C_AVG | N=50            | A_AVG_A2C-DDPG | N=1     
A_AVG_A2C-DDPG | N=10     | A_AVG_A2C-DDPG | N=15     | A_AVG_A2C-DDPG | N=20    
A_AVG_A2C-DDPG | N=25     | A_AVG_A2C-DDPG | N=30     | A_AVG_A2C-DDPG | N=35    
A_AVG_A2C-DDPG | N=40     | A_AVG_A2C-DDPG | N=45     | A_AVG_A2C-DDPG | N=5     
A_AVG_A2C-DDPG | N=50     | A_AVG_A2C-DDPG-PPO | N=1  | A_AVG_A2C-DDPG-PPO | N=10
A_AVG_A2C-DDPG-PPO | N=15 | A_AVG_A2C-DDPG-PPO | N=20 | A_AVG_A2C-DDPG-PPO | N=25
A_AVG_A2C-DDPG-PPO | N=30 | 


Wpisz nazwę konfiguracji (np. 'PPO_AVG | N=5'):  A_WIN_A2C-PPO-TD3 | N=50


⚙️ Przetwarzanie krzywych kapitału...

                WYBÓR BENCHMARKÓW DO WYKRESU                


Z jakiego początku analizować dane benchmarku (np. 'Q1 2020') [Domyślnie Q1 2020]:  
Do jakiego końca analizować dane benchmarku (np. 'Q4 2020') [Domyślnie Q4 2020]:  
Czy dorysować benchmark 'Buy & Hold'? (T/n) [T]:  
Czy dorysować benchmark 'Min-Variance'? (T/n) [T]:  
Czy dorysować benchmark 'Mean-Semivariance'? (T/n) [T]:  



📊 Generowanie zintegrowanego wykresu...
✅ Wykres zapisany jako: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Krzywa_Kap_MinMax_Bench_A_WIN_A2C-PPO-TD3_N50_B30_Q1_2020_Q4_2020.png


### Wykres przebiegu krzywej kapitału - wybrany komitet z danym N + obszar przebiegów (wszystkich komitetów przy tym N) + benchmarki

In [11]:
# --- 1. KONFIGURACJA BAZOWA ---
BRAND_COLOR = '#007481' # Pantone 7474C

# --- 2. ODCZYT PLIKÓW MODELI RL (Z UWZGLĘDNIENIEM BOOTSTRAPU) ---
max_n_to_check = int(input("Jakie maksymalne N było w pliku nasycenia?: [np. 100] ") or 100)
num_b = int(input("Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]: ") or 30)
density_choice = input("Czy w symulacji użyto N rzadkiego? (T/n) [N]: ").strip().upper()

file_suffix = "_RZADKIE" if density_choice == 'T' else ""
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_NASYCENIE_N1_{max_n_to_check}_B{num_b}{file_suffix}.csv")

if not os.path.exists(TRACES_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku modeli {TRACES_FILE}")
    exit()

print(f"⏳ Wczytywanie danych modeli z {TRACES_FILE}...")
df_all = pd.read_csv(TRACES_FILE, sep=';', decimal=',')
df_all['date'] = pd.to_datetime(df_all['date'])

# [ZMIANA] Tworzymy Config ogólny (do wyboru) oraz Config_Iter (unikalny klucz do pivota)
df_all['Config'] = df_all['Model'] + " | N=" + df_all['N'].astype(str)
df_all['Config_Iter'] = df_all['Config'] + " | Iter=" + df_all['Iteration'].astype(str)

print("⚙️ Pivotowanie danych modeli z uwzględnieniem iteracji...")
# Pivotujemy po unikalnym identyfikatorze każdej pojedynczej linii z symulacji
pivot_df = df_all.pivot(index='date', columns='Config_Iter', values='account_value')
pivot_df = pivot_df.fillna(method='ffill').fillna(method='bfill') 
pivot_returns = (pivot_df / pivot_df.iloc[0]) - 1

# --- 3. WYBÓR MODELU RL ---
# Do wyboru pokazujemy tylko główne konfiguracje (bez śmieci z napisem Iteration)
available_configs = sorted(df_all['Config'].unique().tolist())
print("\n" + "="*60)
print(f"{'DOSTĘPNE KONFIGURACJE DO WYRÓŻNIENIA':^60}")
print("="*60)
for i in range(0, len(available_configs), 3):
    print(" | ".join([f"{m:<25}" for m in available_configs[i:i+3]]))
print("="*60)

while True:
    chosen_config = input("\nWpisz nazwę konfiguracji (np. 'PPO_AVG | N=50'): ").strip()
    if chosen_config in available_configs:
        break
    print("❌ Niepoprawna nazwa. Spróbuj ponownie.")

# [ZMIANA] Obliczanie tła i lidera
chosen_n = chosen_config.split(" | N=")[1]

# 1. Filtrujemy WSZYSTKIE kolumny dla wybranego N (wszystkie modele, wszystkie iteracje)
cols_for_current_n = [col for col in pivot_returns.columns if f" | N={chosen_n} | Iter=" in col]
n_min_curve = pivot_returns[cols_for_current_n].min(axis=1)
n_max_curve = pivot_returns[cols_for_current_n].max(axis=1)

# [NOWE] Obliczamy granice dla 95% przedziału ufności
n_lower_95_curve = pivot_returns[cols_for_current_n].quantile(0.025, axis=1)
n_upper_95_curve = pivot_returns[cols_for_current_n].quantile(0.975, axis=1)

# 2. Filtrujemy WSZYSTKIE iteracje dla WYBRANEGO modelu i bierzemy z nich medianę
cols_for_chosen_config = [col for col in pivot_returns.columns if col.startswith(chosen_config + " | Iter=")]
leader_curve = pivot_returns[cols_for_chosen_config].median(axis=1)

# --- 4. WYBÓR BENCHMARKÓW ---
print("\n" + "="*60)
print(f"{'WYBÓR BENCHMARKÓW DO WYKRESU':^60}")
print("="*60)

user_start = input("Z jakiego początku analizować dane benchmarku (np. 'Q1 2020') [Domyślnie Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Do jakiego końca analizować dane benchmarku (np. 'Q4 2020') [Domyślnie Q4 2020]: ").strip() or "Q4 2020"

# Obejście braku QUARTERS_MAP jeśli kod jest uruchamiany oddzielnie
QUARTERS_MAP = {'Q1 2020': 0, 'Q2 2020': 1, 'Q3 2020': 2, 'Q4 2020': 3} if 'QUARTERS_MAP' not in locals() else QUARTERS_MAP
start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

if start_idx > end_idx:
    start_idx, end_idx = end_idx, start_idx
    user_start, user_end = user_end, user_start

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")
period_suffix = f"_{safe_start_str}" if start_idx == end_idx else f"_{safe_start_str}_{safe_end_str}"

excel_name = f"Benchmarki{period_suffix}.xlsx"
bench_file_path = os.path.join(BENCHMARK_DIR, excel_name)

STYLES = {
    'BUY_AND_HOLD': ('black', '-', 'Buy & Hold'),
    'MIN_VAR':      ('blue', '--', 'Min-Variance'),
    'MEAN_VAR':     ('orange', '-.', 'Mean-Variance')
}

active_benchmarks = {}

if not os.path.exists(bench_file_path):
    print(f"⚠️ Nie znaleziono pliku benchmarków: {excel_name}. Zostaną one pominięte.")
else:
    for sheet_name, (color, style, label) in STYLES.items():
        ans = input(f"Czy dorysować benchmark '{label}'? (T/n) [T]: ").strip().upper()
        if ans != 'N':
            active_benchmarks[sheet_name] = (color, style, label)

# Wczytanie wybranych benchmarków
bench_curves = {}
if active_benchmarks:
    try:
        xl_bench = pd.ExcelFile(bench_file_path)
        for sheet_name, (color, style, label) in active_benchmarks.items():
            if sheet_name in xl_bench.sheet_names:
                df_b = pd.read_excel(xl_bench, sheet_name=sheet_name)
                df_b = df_b.dropna(subset=['date', 'account_value'])
                if df_b.empty: continue
                df_b['date'] = pd.to_datetime(df_b['date'])
                df_b = df_b.sort_values('date')
                
                # Normalizacja
                initial_val = df_b['account_value'].iloc[0]
                df_b['cum_return'] = (df_b['account_value'] / initial_val) - 1
                
                # Ustawiamy index po dacie, aby zrównać osie x przy plotowaniu
                bench_curves[sheet_name] = {
                    'data': df_b.set_index('date')['cum_return'],
                    'color': color,
                    'style': style,
                    'label': label
                }
            else:
                print(f"⚠️ Zakładka {sheet_name} nie istnieje w {excel_name}")
    except Exception as e:
        print(f"❌ Błąd wczytywania benchmarków: {e}")

# --- 5. RYSOWANIE WYKRESU ZBIORCZEGO ---
print(f"\n📊 Generowanie zintegrowanego wykresu...")

fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
dates = pivot_returns.index

# Tło (Min-Max) ze wszystkich iteracji wszystkich modeli przy tym N
ax.fill_between(dates, n_min_curve, n_max_curve, 
                color='gray', alpha=0.15) # Zmniejszyłem lekko jasność tła dla kontrastu

# [NOWE] Wewnętrzny przedział ufności 95% (ciemniejszy)
ax.fill_between(dates, n_lower_95_curve, n_upper_95_curve, 
                color='gray', alpha=0.35, zorder=2)

# Gruba linia Lidera w #E377C2ze firmowym (mediana z B iteracji)
ax.plot(dates, leader_curve, color=BRAND_COLOR, linewidth=3, zorder=5)

# Dodawanie Benchmarków na ten sam wykres
for sheet_name, bench_info in bench_curves.items():
    b_series = bench_info['data']
    ax.plot(b_series.index, b_series.values, 
            label=bench_info['label'], color=bench_info['color'], 
            linestyle=bench_info['style'], linewidth=2.5, zorder=4)

# Estetyka
title_period = f"{user_start}" if start_idx == end_idx else f"{user_start} do {user_end}"
ax.set_title(f"Porównanie: {chosen_config} na tle zmienności sztucznej inteligencji i Benchmarków\n(Okres: {title_period} | Bootstrap B={num_b})", 
             fontsize=16, fontweight='bold', color='#333333', pad=20)
ax.set_ylabel("Skumulowana stopa zwrotu (%)", fontsize=14)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.grid(True, which='major', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linewidth=1.5, alpha=0.3)

# Legenda dopasowana
custom_legend_elements = [
    Patch(facecolor='gray', alpha=0.15, label=f'Całkowity zakres min-max (N={chosen_n})'),
    Patch(facecolor='gray', alpha=0.5, label=f'95% przedział wyników dla wszystkich modeli (N={chosen_n})'),
    Line2D([0], [0], color=BRAND_COLOR, lw=3, label=f'Krzywa (mediana B={num_b}): {chosen_config}')
]

# Dodawanie legend z benchmarków
for sheet_name, bench_info in bench_curves.items():
    custom_legend_elements.append(
        Line2D([0], [0], color=bench_info['color'], lw=2.5, linestyle=bench_info['style'], label=bench_info['label'])
    )

ax.legend(handles=custom_legend_elements, loc='upper left', fontsize=12, frameon=True, shadow=True)
plt.tight_layout()

# Zapis
chosen_model_name = chosen_config.split(" | ")[0]
plot_filename = f"Tlo_N{chosen_n}_z_Bench_{chosen_model_name}_B{num_b}{period_suffix}.png"
save_path = os.path.join(AGG_DIR, plot_filename)
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ Wykres zapisany jako: {save_path}")
plt.show()

Jakie maksymalne N było w pliku nasycenia?: [np. 100]  50
Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]:  30
Czy w symulacji użyto N rzadkiego? (T/n) [N]:  T


⏳ Wczytywanie danych modeli z /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/szczegoly_account_value_NASYCENIE_N1_50_B30_RZADKIE.csv...
⚙️ Pivotowanie danych modeli z uwzględnieniem iteracji...

            DOSTĘPNE KONFIGURACJE DO WYRÓŻNIENIA            
A2C_AVG | N=1             | A2C_AVG | N=10            | A2C_AVG | N=15           
A2C_AVG | N=20            | A2C_AVG | N=25            | A2C_AVG | N=30           
A2C_AVG | N=35            | A2C_AVG | N=40            | A2C_AVG | N=45           
A2C_AVG | N=5             | A2C_AVG | N=50            | A_AVG_A2C-DDPG | N=1     
A_AVG_A2C-DDPG | N=10     | A_AVG_A2C-DDPG | N=15     | A_AVG_A2C-DDPG | N=20    
A_AVG_A2C-DDPG | N=25     | A_AVG_A2C-DDPG | N=30     | A_AVG_A2C-DDPG | N=35    
A_AVG_A2C-DDPG | N=40     | A_AVG_A2C-DDPG | N=45     | A_AVG_A2C-DDPG | N=5     
A_AVG_A2C-DDPG | N=50     | A_AVG_A2C-DDPG-PPO | N=1  | A_AVG_A2C-DDPG-PPO | N=10
A_AVG_A2C-DDPG-PPO | N=15 | A_AVG_A2C-DDPG-PP


Wpisz nazwę konfiguracji (np. 'PPO_AVG | N=50'):  A_WIN_A2C-PPO-TD3 | N=50



                WYBÓR BENCHMARKÓW DO WYKRESU                


Z jakiego początku analizować dane benchmarku (np. 'Q1 2020') [Domyślnie Q1 2020]:  
Do jakiego końca analizować dane benchmarku (np. 'Q4 2020') [Domyślnie Q4 2020]:  
Czy dorysować benchmark 'Buy & Hold'? (T/n) [T]:  
Czy dorysować benchmark 'Min-Variance'? (T/n) [T]:  
Czy dorysować benchmark 'Mean-Variance'? (T/n) [T]:  



📊 Generowanie zintegrowanego wykresu...
✅ Wykres zapisany jako: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Tlo_N50_z_Bench_A_WIN_A2C-PPO-TD3_B30_Q1_2020_Q4_2020.png


### Obszar przebiegu krzywych kapitału dla różnych N + benchmarki

In [36]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- 1. KONFIGURACJA BAZOWA ---
SHOW_TITLES = False  # Flaga decydująca o rysowaniu tytułu wykresu
BRAND_COLOR = '#007481' # Pantone 7474C

# Zaktualizowane style benchmarków
STYLES = {
    'BUY_AND_HOLD': ('black',   '-', 'Buy & Hold'),
    'MIN_VAR':      ('#E377C2', '-', 'Min-Variance'),       # Cyjanowy/Morski
    'MEAN_VAR':     ('#A0522D', '-', 'Mean-Semivariance')   # Różowy/Magenta
}

# --- 2. ODCZYT PLIKÓW MODELI RL (Z UWZGLĘDNIENIEM BOOTSTRAPU) ---
max_n_to_check = int(input("Jakie maksymalne N było w pliku nasycenia?: [np. 100] ") or 100)
num_b = int(input("Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]: ") or 30)
density_choice = input("Czy w symulacji użyto N rzadkiego? (T/n) [N]: ").strip().upper()

file_suffix = "_RZADKIE" if density_choice == 'T' else ""
TRACES_FILE = os.path.join(AGG_DIR, f"szczegoly_account_value_NASYCENIE_N1_{max_n_to_check}_B{num_b}{file_suffix}.csv")

if not os.path.exists(TRACES_FILE):
    print(f"❌ BŁĄD: Nie znaleziono pliku modeli {TRACES_FILE}")
    exit()

print(f"⏳ Wczytywanie danych modeli z {TRACES_FILE}...")
df_all = pd.read_csv(TRACES_FILE, sep=';', decimal=',')
df_all['date'] = pd.to_datetime(df_all['date'])

df_all['Config'] = df_all['Model'] + " | N=" + df_all['N'].astype(str)
df_all['Config_Iter'] = df_all['Config'] + " | Iter=" + df_all['Iteration'].astype(str)

print("⚙️ Pivotowanie danych modeli z uwzględnieniem iteracji...")
pivot_df = df_all.pivot(index='date', columns='Config_Iter', values='account_value')
pivot_df = pivot_df.fillna(method='ffill').fillna(method='bfill') 
pivot_returns = (pivot_df / pivot_df.iloc[0]) - 1

# Pobranie i posortowanie wszystkich dostępnych wartości N
available_n = sorted(df_all['N'].unique().tolist())
print(f"✅ Znaleziono komitety dla N: {available_n}")

# --- 2.5 WYBÓR TRYBU WIZUALIZACJI N ---
print("\n" + "="*60)
print(f"{'WYBÓR TRYBU WIZUALIZACJI N':^60}")
print("="*60)
mode_n = input("Czy chcesz narysować obszary dla wszystkich N (W) czy dla wybranego (V)? [W]: ").strip().upper()

if mode_n == 'V':
    while True:
        try:
            chosen_n_str = input(f"Podaj wybrane N spośród dostępnych {available_n}: ").strip()
            chosen_n = int(chosen_n_str)
            if chosen_n in available_n:
                n_to_plot = [chosen_n]
                break
            else:
                print("❌ Wybrano N spoza dostępnej listy. Spróbuj ponownie.")
        except ValueError:
            print("❌ Podaj poprawną liczbę całkowitą.")
else:
    n_to_plot = available_n

# --- 3. WYBÓR BENCHMARKÓW ---
print("\n" + "="*60)
print(f"{'WYBÓR BENCHMARKÓW DO WYKRESU':^60}")
print("="*60)

user_start = input("Z jakiego początku analizować dane benchmarku (np. 'Q1 2020') [Domyślnie Q1 2020]: ").strip() or "Q1 2020"
user_end = input("Do jakiego końca analizować dane benchmarku (np. 'Q4 2020') [Domyślnie Q4 2020]: ").strip() or "Q4 2020"

QUARTERS_MAP = {'Q1 2020': 0, 'Q2 2020': 1, 'Q3 2020': 2, 'Q4 2020': 3} if 'QUARTERS_MAP' not in locals() else QUARTERS_MAP
start_idx = QUARTERS_MAP.get(user_start, 0)
end_idx = QUARTERS_MAP.get(user_end, 3)

if start_idx > end_idx:
    start_idx, end_idx = end_idx, start_idx
    user_start, user_end = user_end, user_start

safe_start_str = user_start.replace(" ", "_")
safe_end_str = user_end.replace(" ", "_")
period_suffix = f"_{safe_start_str}" if start_idx == end_idx else f"_{safe_start_str}_{safe_end_str}"

excel_name = f"Benchmarki{period_suffix}.xlsx"
bench_file_path = os.path.join(BENCHMARK_DIR, excel_name)

active_benchmarks = {}

if not os.path.exists(bench_file_path):
    print(f"⚠️ Nie znaleziono pliku benchmarków: {excel_name}. Zostaną one pominięte.")
else:
    for sheet_name, (color, style, label) in STYLES.items():
        ans = input(f"Czy dorysować benchmark '{label}'? (T/n) [T]: ").strip().upper()
        if ans != 'N':
            active_benchmarks[sheet_name] = (color, style, label)

bench_curves = {}
if active_benchmarks:
    try:
        xl_bench = pd.ExcelFile(bench_file_path)
        for sheet_name, (color, style, label) in active_benchmarks.items():
            if sheet_name in xl_bench.sheet_names:
                df_b = pd.read_excel(xl_bench, sheet_name=sheet_name)
                df_b = df_b.dropna(subset=['date', 'account_value'])
                if df_b.empty: continue
                df_b['date'] = pd.to_datetime(df_b['date'])
                df_b = df_b.sort_values('date')
                
                initial_val = df_b['account_value'].iloc[0]
                df_b['cum_return'] = (df_b['account_value'] / initial_val) - 1
                
                bench_curves[sheet_name] = {
                    'data': df_b.set_index('date')['cum_return'],
                    'color': color,
                    'style': style,
                    'label': label
                }
    except Exception as e:
        print(f"❌ Błąd wczytywania benchmarków: {e}")

# --- 4. RYSOWANIE WYKRESU ZBIORCZEGO (LEJEK WARIANCJI) ---
print(f"\n📊 Generowanie zintegrowanego wykresu z obszarami N...")

fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
dates = pivot_returns.index

custom_legend_elements = []

# Iterujemy po N rosnąco - dla n_to_plot
for i, current_n in enumerate(n_to_plot):
    cols_for_current_n = [col for col in pivot_returns.columns if f" | N={current_n} | Iter=" in col]
    
    if not cols_for_current_n:
        continue
        
    lower_curve = pivot_returns[cols_for_current_n].min(axis=1)
    upper_curve = pivot_returns[cols_for_current_n].max(axis=1)
    
    # [NOWE] Obliczanie 95% przedziału ufności (kwantyle 2.5% i 97.5%)
    q025_curve = pivot_returns[cols_for_current_n].quantile(0.025, axis=1)
    q975_curve = pivot_returns[cols_for_current_n].quantile(0.975, axis=1)
    
    # Obliczanie przezroczystości (alpha) w zależności od wybranego trybu
    if len(n_to_plot) > 1:
        alpha_val = 0.15 + (0.70 * (i / (len(n_to_plot) - 1)))
    else:
        # Klasyczny #E377C2 dla pojedynczego N z lekką przezroczystością
        alpha_val = 0.3
        
    # Rysowanie wypełnienia Min-Max
    ax.fill_between(dates, lower_curve, upper_curve, 
                    color=BRAND_COLOR, alpha=alpha_val, zorder=2+i, edgecolors='none')
                    
    # [NOWE] Rysowanie wypełnienia 95% (nieco ciemniejsze)
    ax.fill_between(dates, q025_curve, q975_curve, 
                    color=BRAND_COLOR, alpha=min(alpha_val + 0.15, 1.0), zorder=2+i+0.5, edgecolors='none')
    
    # Dodanie elementu legendy dla tego N (obszar całkowity min-max jako reprezentant)
    custom_legend_elements.append(
        Patch(facecolor=BRAND_COLOR, alpha=alpha_val, label=f'Obszar min-max')
    )
    
    # [NOWE Opcjonalne] Jeśli wybrano pojedyncze N, dodaj objaśnienie w legendzie dla ciemniejszego obszaru
    if len(n_to_plot) == 1:
        custom_legend_elements.append(
            Patch(facecolor=BRAND_COLOR, alpha=min(alpha_val + 0.15, 1.0), label=f'Przedział 95%')
        )

# --- Dodawanie Benchmarków na ABSOLUTNY wierzch (zorder=999) ---
for sheet_name, bench_info in bench_curves.items():
    b_series = bench_info['data']
    ax.plot(b_series.index, b_series.values, 
            label=bench_info['label'], color=bench_info['color'], 
            linestyle=bench_info['style'], linewidth=2.5, zorder=999)
    
    custom_legend_elements.append(
        Line2D([0], [0], color=bench_info['color'], lw=2.5, linestyle=bench_info['style'], label=bench_info['label'])
    )

# Estetyka
if SHOW_TITLES:
    title_period = f"{user_start}" if start_idx == end_idx else f"{user_start} do {user_end}"
    title_n_info = f"dla N={n_to_plot[0]}" if len(n_to_plot) == 1 else "w Zależności od Rozmiaru Komitetu (N)"
    ax.set_title(f"Redukcja Wariancji Przebiegów {title_n_info}\n(Okres: {title_period} | Bootstrap B={num_b})", 
                 fontsize=16, fontweight='bold', color='#333333', pad=20)

# Zwiększone czcionki osi i wartości
ax.set_ylabel("Skumulowana stopa zwrotu (%)", fontsize=20)
ax.tick_params(axis='y', labelsize=15)
ax.tick_params(axis='x', labelsize=15)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Linie siatki i poziom zero
ax.grid(True, which='major', linestyle='--', alpha=0.4, zorder=1)
ax.axhline(0, color='black', linewidth=1.5, alpha=0.6, zorder=1)

# Limit liczby elementów w legendzie (Zwiększona czcionka do 20)
ncol_legend = 2 if len(custom_legend_elements) > 6 else 1
ax.legend(handles=custom_legend_elements, loc='upper left', fontsize=20, frameon=True, shadow=True, ncol=ncol_legend)
plt.tight_layout()

# Zapis
n_suffix = f"_N{n_to_plot[0]}" if len(n_to_plot) == 1 else "_All_N"
plot_filename = f"Wizualizacja_Wariancji{n_suffix}_vs_Bench_B{num_b}{period_suffix}.png"
save_path = os.path.join(AGG_DIR, plot_filename)
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ Wykres zapisany jako: {save_path}")

Jakie maksymalne N było w pliku nasycenia?: [np. 100]  50
Podaj liczbę iteracji Bootstrapu (B) użytej w symulacji [np. 30]:  30
Czy w symulacji użyto N rzadkiego? (T/n) [N]:  t


⏳ Wczytywanie danych modeli z /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/szczegoly_account_value_NASYCENIE_N1_50_B30_RZADKIE.csv...
⚙️ Pivotowanie danych modeli z uwzględnieniem iteracji...
✅ Znaleziono komitety dla N: [1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

                 WYBÓR TRYBU WIZUALIZACJI N                 


Czy chcesz narysować obszary dla wszystkich N (W) czy dla wybranego (V)? [W]:  V
Podaj wybrane N spośród dostępnych [1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:  50



                WYBÓR BENCHMARKÓW DO WYKRESU                


Z jakiego początku analizować dane benchmarku (np. 'Q1 2020') [Domyślnie Q1 2020]:  
Do jakiego końca analizować dane benchmarku (np. 'Q4 2020') [Domyślnie Q4 2020]:  
Czy dorysować benchmark 'Buy & Hold'? (T/n) [T]:  
Czy dorysować benchmark 'Min-Variance'? (T/n) [T]:  
Czy dorysować benchmark 'Mean-Semivariance'? (T/n) [T]:  



📊 Generowanie zintegrowanego wykresu z obszarami N...
✅ Wykres zapisany jako: /Users/maciek/Desktop/magisterskie/MAGISTER TYLKO 2020 TEST/symulacja_modeli_agregowanych/Wizualizacja_Wariancji_N50_vs_Bench_B30_Q1_2020_Q4_2020.png
